# TwinPast scene-boundary GPU comparison

Upload this notebook to the same AutoDL GPU container used for refinement and
select **Restart Kernel → Run All**. It reuses the verified Python environment,
pinned LLaVA-OneVision 7B weights and clips when available. Use a GPU with at
least 22 GiB VRAM. On a fresh container, setup can require about 40 GiB of disk.

Compare coarse uniform, local uniform and the frozen boundary-constrained
ranking plus pre-cut anchor on eight development pairs. Each method supplies
six history and four matched recent frames to the original direct scorer.
Both option orders are scored. Recompute question relevance, reproduce the
CPU frame selections and check 64 archived baseline scores before reporting
paired results. Selection and replay mismatches preserve failure evidence.

A development pass requires a robust recovery with no robust regression
versus coarse uniform. The notebook reports per-stage timings, including the
additional boundary scan. The per-policy time is a sum of measured stages
with shared work allocated to each policy. It is not an independent end-to-end
latency benchmark.

After **TWINPAST_BOUNDARY_COMPLETE**, save the executed notebook and download
**HERMES_twinpast_boundary_evidence.zip**. If there is a failure, run the final
download cell manually to retrieve the diagnostic evidence.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, time, zipfile, ast
from datetime import datetime, timezone

WORK = Path('/root/autodl-tmp/hermes_twinpast_boundary')
REUSE = WORK.with_name('hermes_twinpast_refinement')
FALLBACK = WORK.with_name('hermes_causal_pilot')
BASE_REVISION = '2a1385e0abc94debb64761b0465713c799a81c1c'
MODEL_ID = 'llava-hf/llava-onevision-qwen2-7b-ov-hf'
MODEL_SHA = '0d50680527681998e456c7b78950205bedd8a068'
TF_SHA = '66bc4def9505fa7c7fe4aa7a248c34a026bb552b'
VALIDATE_ONLY = os.environ.get('HERMES_TWINPAST_BOUNDARY_VALIDATE_ONLY') == '1'
assert VALIDATE_ONLY or (WORK.is_absolute() and REUSE.is_absolute())
assert WORK != REUSE and not WORK.is_relative_to(REUSE) and not REUSE.is_relative_to(WORK)
REPO = WORK/'pinned_source'
VENV = WORK/'venv'
PYTHON = VENV/'bin/python'
RUNNER = WORK/'provenance/boundary_runner.py'
print('Plan: 8 archived pairs, 3 policies, frozen CPU rule')
print('WORK:', WORK, '\nReuse candidate:', REUSE)
print('Mode:', 'CPU notebook validation (no network/GPU)' if VALIDATE_ONLY else 'AutoDL execution')
PREVIOUS = REUSE


## 1. Verify runtime and materialize the experiment


In [ ]:
def network_environment():
    env = os.environ.copy()
    if Path('/etc/network_turbo').is_file():
        proc = subprocess.run(['bash', '-lc', 'source /etc/network_turbo >/dev/null 2>&1; env -0'], capture_output=True)
        if proc.returncode == 0:
            for item in proc.stdout.split(b'\0'):
                if b'=' in item:
                    key, value = item.split(b'=', 1)
                    env[key.decode()] = value.decode()
    model_folder = 'models--' + MODEL_ID.replace('/', '--')
    candidates = [root/'cache/huggingface' for root in (REUSE, FALLBACK, WORK)]
    snapshots = [root/'hub'/model_folder/'snapshots'/MODEL_SHA for root in candidates]
    cache = next((root for root, snap in zip(candidates, snapshots, strict=True)
                  if sum(p.is_file() for p in snap.glob('model-*-of-00004.safetensors')) == 4), None)
    if cache is None:
        cache = next((root for root, snap in zip(candidates, snapshots, strict=True) if snap.is_dir()),
                     candidates[-1])
    env.update(HF_HOME=str(cache), HF_HUB_DISABLE_XET='1',
               HF_HUB_ENABLE_HF_TRANSFER='0', TOKENIZERS_PARALLELISM='false',
               CUDA_VISIBLE_DEVICES='0', OMP_NUM_THREADS='8', GIT_LFS_SKIP_SMUDGE='1',
               UV_CACHE_DIR=str(WORK/'cache/uv'), UV_LINK_MODE='copy', TMPDIR=str(WORK),
               UV_DEFAULT_INDEX='https://pypi.tuna.tsinghua.edu.cn/simple',
               PYTORCH_ALLOC_CONF='expandable_segments:True')
    env.pop('PYTHONHOME', None)
    env['PYTHONPATH'] = ''
    return env

def logged(command, name, env=None, cwd=None):
    import queue, threading
    logfile = WORK/'logs'/name
    logfile.parent.mkdir(parents=True, exist_ok=True)
    print('SETUP_STAGE', name, flush=True)
    with logfile.open('a', encoding='utf-8') as handle:
        proc = subprocess.Popen([str(x) for x in command], env=env, cwd=cwd,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1)
        lines = queue.Queue()
        def collect():
            try:
                for line in proc.stdout:
                    lines.put(line)
            finally:
                lines.put(None)
        reader = threading.Thread(target=collect, daemon=True)
        reader.start()
        started = time.monotonic()
        while True:
            try:
                line = lines.get(timeout=30)
            except queue.Empty:
                line = f'COMMAND_ALIVE log={name} elapsed_min={(time.monotonic()-started)/60:.1f} (liveness only)\n'
            if line is None:
                break
            print(line, end='', flush=True)
            handle.write(line)
            handle.flush()
        reader.join(timeout=5)
        status = proc.wait()
        proc.stdout.close()
    if status:
        raise RuntimeError(f'Command failed ({status}); see {logfile}. Failure evidence, if created, stays in WORK.')

def probe(command, env=None, cwd=None):
    return subprocess.run([str(x) for x in command], env=env, cwd=cwd,
                          text=True, capture_output=True)

def package_environment(base, index='https://pypi.tuna.tsinghua.edu.cn/simple'):
    # AutoDL turbo is for GitHub/Hugging Face. Domestic package mirrors should
    # bypass those proxies, especially for large NVIDIA dependency wheels.
    direct = dict(base)
    for key in ('HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'NO_PROXY',
                'http_proxy', 'https_proxy', 'all_proxy', 'no_proxy'):
        direct.pop(key, None)
    direct.update(UV_HTTP_TIMEOUT='600', UV_HTTP_RETRIES='5',
                  UV_CONCURRENT_DOWNLOADS='2', UV_DEFAULT_INDEX=index,
                  PIP_DEFAULT_TIMEOUT='600', PIP_RETRIES='5')
    return direct

def install_packages(uv, python, packages, env, extra=()):
    errors = []
    for index in ('https://pypi.tuna.tsinghua.edu.cn/simple', 'https://mirrors.aliyun.com/pypi/simple'):
        try:
            logged(uv + ['pip', 'install', '--python', python, '--index-url', index]
                   + list(extra) + list(packages), 'setup.log', package_environment(env, index))
            return
        except RuntimeError as exc:
            errors.append(type(exc).__name__)
            print('Package download failed; trying alternate domestic index with cached files', flush=True)
    raise RuntimeError('Package installation failed on both indexes; cached files are preserved. See setup.log.')

if not VALIDATE_ONLY:
    assert sys.platform.startswith('linux'), 'Run this notebook in an AutoDL Linux container'
    WORK.mkdir(parents=True, exist_ok=True)
    ENV = network_environment()
    free = shutil.disk_usage(WORK).free / 1024**3
    minimum_free = 1 if any((root/'venv/bin/python').is_file() for root in (REUSE, FALLBACK)) else 18
    assert free >= minimum_free, f'Need {minimum_free} GiB free on the data disk; found {free:.1f} GiB'
    smi = probe(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], ENV)
    assert smi.returncode == 0 and smi.stdout.strip(), 'CUDA GPU is missing'
    print('GPU:', smi.stdout.strip(), '\nFree GiB:', round(free, 1))
    import re
    first_gpu = smi.stdout.splitlines()[0]
    assert float(re.search(r'(\d+(?:\.\d+)?)\s*MiB', first_gpu)[1]) >= 22528, 'Use a GPU with at least 22 GiB VRAM'
    # Never print the environment or proxy values.


In [ ]:
SOURCES = {'pinned_source/inference/llavaov_hermes.py': 'import re\nimport time\nimport torch\nimport torch.nn.functional as F\nimport types\nfrom transformers import LlavaOnevisionForConditionalGeneration, AutoProcessor, DynamicCache\nfrom logzero import logger\n\nfrom inference.abstract_hermes import Abstract_Hermes\nfrom inference.reindex_1d import (\n    _get_rotary_module,\n    get_cache_seq_len,\n    contiguous_kv,\n    compute_cos_sin_for_positions,\n    rotary_delta,\n    apply_rotary_delta_to_keys_only,\n    apply_rotary_pos_emb_1d,\n)\n\n\nclass LlavaOneVision_Hermes(LlavaOnevisionForConditionalGeneration, Abstract_Hermes):\n    def __init__(self, config, processor, init_prompt_ids, kv_size, streaming=True):\n        super().__init__(processor, init_prompt_ids, kv_size)\n        self.streaming = streaming\n        \n        num_layers = config.num_hidden_layers if hasattr(config, \'num_hidden_layers\') else 32\n        self.num_layers = num_layers\n        \n        self.short_term_ratio = 0.3\n        self.long_term_ratio = 0.3\n        self.short_term_threshold = int(self.num_layers * self.short_term_ratio)\n        self.long_term_threshold = int(self.num_layers * (1 - self.long_term_ratio))\n        \n        self._position_ids_cache = [None for _ in range(num_layers)]\n        self.token_activity_cache = [None for _ in range(num_layers)]\n        \n        self._layer_position_ids = {}\n        self._hook_handles = []\n        self._register_forward_hooks()\n        \n        self.total_processed_frames = 0\n        \n        self._patch_rotary_embeddings()\n\n    def _patch_rotary_embeddings(self):\n        """Patch rotary embedding to support dynamic resizing based on logical position IDs"""\n        def patched_forward(rotary_self, x, seq_len=None):\n            required_len = getattr(rotary_self, \'_required_seq_len\', 0)\n            if seq_len is not None and required_len > seq_len:\n                seq_len = required_len\n            return rotary_self._original_forward(x, seq_len=seq_len)\n\n        if hasattr(self.language_model.model, \'layers\'):\n            for layer in self.language_model.model.layers:\n                if hasattr(layer, \'self_attn\') and hasattr(layer.self_attn, \'rotary_emb\'):\n                    rotary_emb = layer.self_attn.rotary_emb\n                    if not hasattr(rotary_emb, \'_original_forward\'):\n                        rotary_emb._original_forward = rotary_emb.forward\n                        rotary_emb.forward = types.MethodType(patched_forward, rotary_emb)\n                        rotary_emb._required_seq_len = 0\n\n    def _set_rotary_required_len(self, required_len):\n        """Set required sequence length for rotary embeddings"""\n        if hasattr(self.language_model.model, \'layers\'):\n            for layer in self.language_model.model.layers:\n                if hasattr(layer, \'self_attn\') and hasattr(layer.self_attn, \'rotary_emb\'):\n                    layer.self_attn.rotary_emb._required_seq_len = required_len\n\n    def _register_forward_hooks(self):\n        if not hasattr(self.language_model, \'model\') or not hasattr(self.language_model.model, \'layers\'):\n            return\n        \n        def create_hook(layer_idx):\n            def hook(module, args, kwargs):\n                if layer_idx in self._layer_position_ids:\n                    kwargs[\'position_ids\'] = self._layer_position_ids[layer_idx]\n                return args, kwargs\n            return hook\n        \n        for layer_idx, layer in enumerate(self.language_model.model.layers):\n            handle = layer.register_forward_pre_hook(create_hook(layer_idx), with_kwargs=True)\n            self._hook_handles.append(handle)\n    \n    def _clear_forward_hooks(self):\n        for handle in self._hook_handles:\n            handle.remove()\n        self._hook_handles = []\n\n    def _prune_activity_cache(self, keep_indices_per_layer):\n        if all(c is None for c in self.token_activity_cache):\n            return\n\n        for layer_idx in range(self.num_layers):\n            if self.token_activity_cache[layer_idx] is not None:\n                keep_indices = keep_indices_per_layer[layer_idx]\n                if not isinstance(keep_indices, torch.Tensor):\n                    keep_indices = torch.tensor(keep_indices, device=self.device, dtype=torch.long)\n                \n                curr_len = self.token_activity_cache[layer_idx].shape[0]\n                keep_indices = keep_indices[(keep_indices >= 0) & (keep_indices < curr_len)]\n                \n                if keep_indices.numel() > 0:\n                    self.token_activity_cache[layer_idx] = self.token_activity_cache[layer_idx][keep_indices]\n                else:\n                    self.token_activity_cache[layer_idx] = torch.zeros(0, device=self.device)\n\n    def _append_position_ids_layer(self, layer_idx: int, start: int, length: int):\n        device = self.device\n        new_pos = torch.arange(start, start + length, device=device, dtype=torch.long)\n        if self._position_ids_cache[layer_idx] is None:\n            self._position_ids_cache[layer_idx] = new_pos\n        else:\n            self._position_ids_cache[layer_idx] = torch.cat(\n                [self._position_ids_cache[layer_idx], new_pos], dim=0\n            )\n    \n    def _append_position_ids(self, start_per_layer, length: int):\n        if isinstance(start_per_layer, int):\n            for layer_idx in range(self.num_layers):\n                self._append_position_ids_layer(layer_idx, start_per_layer, length)\n        else:\n            for layer_idx in range(self.num_layers):\n                self._append_position_ids_layer(layer_idx, start_per_layer[layer_idx], length)\n\n    def _get_cache_seq_len_per_layer(self) -> list:\n        if self.kv_cache is None:\n            return [0] * self.num_layers\n        \n        lengths = []\n        for layer_idx in range(len(self.kv_cache)):\n            k_layer, v_layer = self.kv_cache[layer_idx]\n            lengths.append(k_layer.shape[2])\n        return lengths\n\n    def _get_next_start_pos_per_layer(self) -> list:\n        next_pos = []\n        for layer_idx in range(self.num_layers):\n            if (layer_idx < len(self._position_ids_cache) and \n                self._position_ids_cache[layer_idx] is not None and \n                self._position_ids_cache[layer_idx].numel() > 0):\n                next_pos.append(self._position_ids_cache[layer_idx][-1].item() + 1)\n            else:\n                next_pos.append(0)\n        return next_pos\n    \n    def _sanitize_keep_indices(self, keep_indices_1d: torch.Tensor, seq_len: int) -> torch.Tensor:\n        keep_indices_1d = keep_indices_1d.to(self.device, dtype=torch.long)\n        keep_indices_1d = keep_indices_1d[(keep_indices_1d >= 0) & (keep_indices_1d < seq_len)]\n        if keep_indices_1d.numel() == 0:\n            return torch.tensor([0], device=self.device, dtype=torch.long)\n        keep_indices_1d = torch.unique(keep_indices_1d, sorted=True)\n        return keep_indices_1d\n\n    def _build_position_ids(self, past_len: int, q_len: int, batch: int) -> torch.Tensor:\n        pos_1d = torch.arange(past_len, past_len + q_len, device=self.device, dtype=torch.long)\n        position_ids = pos_1d.view(1, -1).expand(batch, -1)\n        return position_ids\n\n    @torch.inference_mode()\n    def _shrink_positions_and_rerotate_keys(self, keep_indices_per_layer):\n        device = self.device\n        \n        curr_lens = self._get_cache_seq_len_per_layer()\n        \n        for layer_idx in range(self.num_layers):\n            if (self._position_ids_cache[layer_idx] is None or \n                self._position_ids_cache[layer_idx].numel() != curr_lens[layer_idx]):\n                self._position_ids_cache[layer_idx] = torch.arange(\n                    curr_lens[layer_idx], device=device, dtype=torch.long\n                )\n        max_pos_limit = getattr(self.language_model.config, "max_position_embeddings", 32768)\n        compact_threshold = max_pos_limit - 1024\n        \n        current_max_pos = 0\n        for cache in self._position_ids_cache:\n             if cache is not None and cache.numel() > 0:\n                 current_max_pos = max(current_max_pos, cache[-1].item())\n        \n        should_compact = (current_max_pos > compact_threshold) if self.streaming else True\n        \n        if should_compact:\n            logger.info(f"[Shrink] Max position {current_max_pos} > {compact_threshold}. Compacting position IDs (Shift Left).")\n        \n        old_position_ids_cache = [cache.clone() for cache in self._position_ids_cache]\n\n        sample_k = self.kv_cache[0][0]\n        dtype = sample_k.dtype\n        \n        new_kv_cache = []\n        \n        for layer_idx, (k_layer, v_layer) in enumerate(self.kv_cache):\n            \n            keep_indices_layer = keep_indices_per_layer[layer_idx]\n            if not isinstance(keep_indices_layer, torch.Tensor):\n                keep_indices_layer = torch.as_tensor(keep_indices_layer, dtype=torch.long, device=device)\n            \n            seq_len_layer = k_layer.shape[2]\n            safe_idx = self._sanitize_keep_indices(keep_indices_layer, seq_len_layer)\n            \n            if safe_idx.numel() == 0:\n                logger.warning(f"Layer {layer_idx}: After sanitization, keep_indices is empty; keeping first token")\n                safe_idx = torch.tensor([0], device=device, dtype=torch.long)\n\n            is_long_term = (layer_idx >= self.long_term_threshold)\n            \n            # === Extract Kept Tokens ===\n            k_kept = torch.index_select(k_layer, dim=2, index=safe_idx)\n            v_kept = torch.index_select(v_layer, dim=2, index=safe_idx)\n            old_pos_kept = old_position_ids_cache[layer_idx].index_select(0, safe_idx)\n            \n            # === Determine New Positions for Kept Tokens ===\n            if should_compact:\n                new_pos_kept = torch.arange(len(safe_idx), device=device, dtype=torch.long)\n            else:\n                new_pos_kept = old_pos_kept\n\n            # === Handle RoPE for Kept Tokens ===\n            if should_compact:\n                cos_old, sin_old = compute_cos_sin_for_positions(\n                    self.language_model, len(old_pos_kept), old_pos_kept, dtype, device\n                )\n                cos_new, sin_new = compute_cos_sin_for_positions(\n                    self.language_model, len(new_pos_kept), new_pos_kept, dtype, device\n                )\n                cos_delta, sin_delta = rotary_delta(\n                    cos_old, sin_old, cos_new, sin_new\n                )\n                k_kept_final = apply_rotary_delta_to_keys_only(k_kept, cos_delta, sin_delta)\n            else:\n                k_kept_final = k_kept\n            \n            # === Handle Folding (Summary Token) for Long-term Layers ===\n            if is_long_term:\n                mask = torch.ones(seq_len_layer, dtype=torch.bool, device=device)\n                mask[safe_idx] = False\n                prune_indices = torch.nonzero(mask).squeeze(1)\n                \n                if prune_indices.numel() > 0:\n                    k_pruned = torch.index_select(k_layer, dim=2, index=prune_indices)\n                    v_pruned = torch.index_select(v_layer, dim=2, index=prune_indices)\n                    \n                    v_summary = v_pruned.mean(dim=2, keepdim=True)\n                    \n                    old_pos_pruned = old_position_ids_cache[layer_idx].index_select(0, prune_indices)\n                    \n                    if should_compact:\n                        summary_pos_id = len(safe_idx)\n                    else:\n                        summary_pos_id = new_pos_kept[-1].item() + 1\n                    \n                    summary_pos_tensor = torch.tensor([summary_pos_id], device=device, dtype=torch.long)\n                    target_pos_pruned = summary_pos_tensor.repeat(old_pos_pruned.shape[0])\n                    \n                    cos_old, sin_old = compute_cos_sin_for_positions(\n                        self.language_model, len(old_pos_pruned), old_pos_pruned, dtype, device\n                    )\n                    cos_new, sin_new = compute_cos_sin_for_positions(\n                        self.language_model, len(target_pos_pruned), target_pos_pruned, dtype, device\n                    )\n                    \n                    cos_delta, sin_delta = rotary_delta(cos_old, sin_old, cos_new, sin_new)\n                    \n                    k_pruned_aligned = apply_rotary_delta_to_keys_only(k_pruned, cos_delta, sin_delta)\n                    \n                    k_summary_final = k_pruned_aligned.mean(dim=2, keepdim=True)\n                    \n                    k_final = torch.cat([k_kept_final, k_summary_final], dim=2)\n                    v_final = torch.cat([v_kept, v_summary], dim=2)\n                    \n                    new_pos_layer = torch.cat([new_pos_kept, summary_pos_tensor])\n                    \n                else:\n                    k_final = k_kept_final\n                    v_final = v_kept\n                    new_pos_layer = new_pos_kept\n            else:\n                k_final = k_kept_final\n                v_final = v_kept\n                new_pos_layer = new_pos_kept\n\n            new_kv_cache.append((k_final.contiguous(), v_final.contiguous()))\n            \n            self._position_ids_cache[layer_idx] = new_pos_layer.clone()\n        \n        self.kv_cache = new_kv_cache\n        \n        new_lens = self._get_cache_seq_len_per_layer()\n        logger.info(f"Layer-wise shrink completed. Lengths: min={min(new_lens)}, max={max(new_lens)}, "\n                   f"first={new_lens[0]}, last={new_lens[-1]}")\n    \n    @torch.inference_mode()\n    def predict_next_question(self):\n        """\n        Generates queries to drive attention calculation.\n        Includes Semantic Anchors for better coverage.\n        """\n        if hasattr(self, \'conv_history\') and len(self.conv_history) > 0:\n            last_q, last_a, last_options = self.conv_history[-1]\n            \n            option_match = re.match(r\'^\\s*(?:\\()?([A-Z])(?:\\))?\\.?\\s*$\', last_a)\n            if option_match and last_options is not None:\n                option_char = option_match.group(1)\n                option_idx = ord(option_char) - ord(\'A\')\n                if 0 <= option_idx < len(last_options):\n                    last_a = last_options[option_idx]\n\n            last_a = re.sub(r\'[A-Z]\\) \', \'\', last_a)\n\n            last_round_history = f"Question: {last_q} Answer: {last_a}"\n            \n            global_query = f"Context summary: {last_round_history}. Summarize the video narrative, identifying main characters, key events, timeline changes, and the overall theme."\n            local_query = f"Find recent details related to: {last_round_history}. Describe the current scene in detail, focusing on specific objects, fine-grained actions, and spatial relationships."\n            \n        else:\n            global_query = (\n                "Summarize the video narrative, identifying main characters, key events, timeline changes, and the overall theme."\n            )\n            local_query = (\n                "Describe the current scene in detail, focusing on specific objects, fine-grained actions, and spatial relationships."\n            )\n            \n        print(f"Local question: {local_query}")\n        print(f"Global question: {global_query}")\n        return local_query, global_query\n    \n    @torch.inference_mode()\n    def encode_init_prompt(self):\n        super().encode_init_prompt()\n        self.total_processed_frames = 0\n        curr_lens = self._get_cache_seq_len_per_layer()\n        for layer_idx in range(self.num_layers):\n            self._position_ids_cache[layer_idx] = torch.arange(\n                curr_lens[layer_idx], device=self.device, dtype=torch.long\n            )\n    \n    def get_video_features(self, pixel_values_videos):\n            batch_size, frames, channels, height, width = pixel_values_videos.shape\n            pixel_values_videos = pixel_values_videos.view(batch_size * frames, channels, height, width)\n            video_features = self.vision_tower(pixel_values_videos, output_hidden_states=True)\n            selected_video_feature = video_features.hidden_states[self.config.vision_feature_layer]\n            if self.config.vision_feature_select_strategy == "default":\n                selected_video_feature = selected_video_feature[:, 1:]\n            elif self.config.vision_feature_select_strategy == "full":\n                selected_video_feature = selected_video_feature\n            video_features = self.multi_modal_projector(selected_video_feature)\n            video_features = self.apply_pooling(video_features)\n            frames_after_merge = video_features.shape[0]\n            video_features = video_features.reshape(batch_size, frames_after_merge * video_features.shape[1], -1)  # (B, Nv*196, D)\n            return video_features\n    \n    @torch.inference_mode()\n    def encode_video_chunk(self, video_chunk):\n        if video_chunk is None or (hasattr(video_chunk, "shape") and video_chunk.shape[0] == 0):\n            return\n\n        if len(video_chunk.shape) == 4 and video_chunk.shape[-1] == 3:\n            video_chunk = video_chunk.permute(0, 3, 1, 2)\n\n        pixel_values_videos = self.processor.video_processor(\n            video_chunk, return_tensors="pt"\n        ).pixel_values_videos.to(self.device, self.dtype)\n        video_features = self.get_video_features(pixel_values_videos)\n\n        start_pos_per_layer = self._get_next_start_pos_per_layer()\n        q_len = video_features.shape[1]\n        batch = video_features.shape[0]\n        \n        max_start_pos = max(start_pos_per_layer) if start_pos_per_layer else 0\n        self._set_rotary_required_len(max_start_pos + q_len)\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        default_position_ids = self._build_position_ids(start_pos_per_layer[0], q_len, batch)\n        \n        out = self.language_model(\n            inputs_embeds=video_features,\n            past_key_values=self.kv_cache,\n            use_cache=True,\n            return_dict=True,\n            position_ids=default_position_ids,\n            cache_position=None,\n            attention_mask=None,\n        )\n        self.kv_cache = out.past_key_values\n        self.kv_cache = contiguous_kv(self.kv_cache)\n\n        self._append_position_ids(start_pos_per_layer, q_len)\n        self.last_encoded_frames = video_chunk.shape[0]\n        self.total_processed_frames += video_chunk.shape[0]\n        \n        self._layer_position_ids.clear()\n\n    @torch.inference_mode()\n    def apply_kv_cache_pruning_strict(self, keep_indices_all_layers):\n        if self.kv_cache is None:\n            logger.warning("No KV-Cache to prune")\n            return\n        if not keep_indices_all_layers or len(keep_indices_all_layers[0]) == 0:\n            logger.warning("Empty keep_indices; skip pruning")\n            return\n\n        self._shrink_positions_and_rerotate_keys(keep_indices_all_layers)\n        self._prune_activity_cache(keep_indices_all_layers)\n        \n        logger.info(f"Strict-shrunk KV cache. New length: {get_cache_seq_len(self.kv_cache)}")\n\n    def allocate_budget_by_depth(self, total_budget, num_layers):\n        """Allocate budget uniformly across layers."""\n        budget_per_layer = [total_budget // num_layers] * num_layers\n        diff = total_budget - sum(budget_per_layer)\n        budget_per_layer[-1] += diff\n        return budget_per_layer\n\n    def _compute_attention_scores_manually(self, input_ids, past_key_values):\n        start_pos_per_layer = self._get_next_start_pos_per_layer()\n        q_len = input_ids.shape[1]\n        batch = input_ids.shape[0]\n        \n        inputs_embeds = self.get_input_embeddings()(input_ids)\n        \n        config = self.language_model.config\n        num_layers = config.num_hidden_layers\n        num_heads = config.num_attention_heads\n        num_key_value_heads = config.num_key_value_heads\n        head_dim = config.hidden_size // num_heads\n        \n        attention_weights_list = []\n        \n        hidden_states = inputs_embeds\n        \n        for layer_idx in range(num_layers):\n            layer = self.language_model.model.layers[layer_idx]\n            \n            past_k, past_v = past_key_values[layer_idx]\n            \n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len, batch)\n            \n            hidden_states_norm = layer.input_layernorm(hidden_states)\n            \n            attn = layer.self_attn\n            \n            query_states = attn.q_proj(hidden_states_norm)\n            query_states = query_states.view(batch, q_len, num_heads, head_dim).transpose(1, 2)\n            \n            key_states = attn.k_proj(hidden_states_norm)\n            key_states = key_states.view(batch, q_len, num_key_value_heads, head_dim).transpose(1, 2)\n            \n            value_states = attn.v_proj(hidden_states_norm)\n            value_states = value_states.view(batch, q_len, num_key_value_heads, head_dim).transpose(1, 2)\n            \n            kv_seq_len = position_ids.max().item() + 1\n            rotary_emb = _get_rotary_module(self.language_model)\n            cos, sin = rotary_emb(value_states, kv_seq_len)\n            cos = cos[position_ids].unsqueeze(1)\n            sin = sin[position_ids].unsqueeze(1)\n            \n            query_states = apply_rotary_pos_emb_1d(query_states, cos, sin)\n            key_states = apply_rotary_pos_emb_1d(key_states, cos, sin)\n            \n            key_states = torch.cat([past_k, key_states], dim=2)\n            value_states = torch.cat([past_v, value_states], dim=2)\n            \n            if num_key_value_heads != num_heads:\n                n_rep = num_heads // num_key_value_heads\n                key_states = torch.repeat_interleave(key_states, n_rep, dim=1)\n                value_states = torch.repeat_interleave(value_states, n_rep, dim=1)\n            \n            attn_weights = torch.matmul(query_states.float(), key_states.float().transpose(-2, -1)) / (head_dim ** 0.5)\n            attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)\n            attention_weights_list.append(attn_weights)\n        \n        return attention_weights_list\n\n    def prune_kv_cache_by_attention(self, attn_weights_local, attn_weights_global, attn_weights_mixed, num_keep=3000):\n        device = self.device\n        visual_start_idx = self.visual_start_idx\n        num_layers = len(attn_weights_local)\n        \n        curr_cache_lens = self._get_cache_seq_len_per_layer()\n        \n        question_len_local = attn_weights_local[0].shape[2]\n        question_len_global = attn_weights_global[0].shape[2]\n        question_len_mixed = attn_weights_mixed[0].shape[2]\n\n        total_budget = num_keep * num_layers\n        budget_per_layer = self.allocate_budget_by_depth(total_budget, num_layers)\n        \n        keep_indices_all_layers = []\n\n        layer_raw_scores = []\n        layer_configs = []\n\n        for layer_idx in range(len(attn_weights_local)):\n            curr_len = curr_cache_lens[layer_idx]\n            if self.token_activity_cache[layer_idx] is None:\n                self.token_activity_cache[layer_idx] = torch.zeros(curr_len, device=device, dtype=torch.float32)\n            elif self.token_activity_cache[layer_idx].shape[0] < curr_len:\n                diff = curr_len - self.token_activity_cache[layer_idx].shape[0]\n                zeros = torch.zeros(diff, device=device, dtype=torch.float32)\n                self.token_activity_cache[layer_idx] = torch.cat([self.token_activity_cache[layer_idx], zeros])\n            \n            if self.token_activity_cache[layer_idx].shape[0] > curr_len:\n                 self.token_activity_cache[layer_idx] = self.token_activity_cache[layer_idx][:curr_len]\n\n            # Memory hierarchy mode\n            if layer_idx < self.short_term_threshold:\n                layer_type = "short-term"\n                layer_attn_weights = attn_weights_local[layer_idx]\n                question_len = question_len_local\n                layer_recency_alpha = 1\n                k = 20\n                \n            elif layer_idx >= self.long_term_threshold:\n                layer_type = "long-term"\n                layer_attn_weights = attn_weights_global[layer_idx]\n                question_len = question_len_global\n                layer_recency_alpha = 0\n                k = 0.0\n                \n            else:\n                layer_type = "mid-term"\n                layer_attn_weights = attn_weights_mixed[layer_idx]\n                question_len = question_len_mixed\n                progress = (layer_idx - self.short_term_threshold) / (self.long_term_threshold - self.short_term_threshold)\n                layer_recency_alpha = 1.0 - 0.9 * progress\n                k = 20 - 19.5 * progress\n            \n            visual_attn_weights = layer_attn_weights[0].mean(dim=0)[:,visual_start_idx:-1*question_len].mean(dim=0)\n            num_visual_tokens = visual_attn_weights.shape[0]\n            \n            end_idx = visual_start_idx + num_visual_tokens\n            if end_idx <= self.token_activity_cache[layer_idx].shape[0]:\n                self.token_activity_cache[layer_idx][visual_start_idx:end_idx] += visual_attn_weights\n            \n            layer_budget = budget_per_layer[layer_idx]\n            \n            if layer_type == \'long-term\':\n                layer_budget = max(0, layer_budget - 1)\n\n            positions = torch.arange(num_visual_tokens, device=device, dtype=torch.float32)\n            time_distances = (num_visual_tokens - 1 - positions) / max(num_visual_tokens - 1, 1)\n            \n            # Exponential decay\n            recency_weights = torch.exp(-k * time_distances)\n            \n            attn_norm = (visual_attn_weights - visual_attn_weights.min()) / \\\n                        (visual_attn_weights.max() - visual_attn_weights.min() + 1e-6)\n            recency_norm = (recency_weights - recency_weights.min()) / \\\n                        (recency_weights.max() - recency_weights.min() + 1e-6)\n            \n            raw_score = attn_norm * (1 - layer_recency_alpha) + recency_norm * layer_recency_alpha\n            \n            layer_raw_scores.append(raw_score)\n            layer_configs.append({\n                \'budget\': min(layer_budget, num_visual_tokens),\n                \'layer_type\': layer_type,\n                \'visual_start_idx\': visual_start_idx\n            })\n        \n        refined_scores = [s.clone() for s in layer_raw_scores]\n        \n        for i in range(len(refined_scores) - 2, -1, -1):\n            current_type = layer_configs[i][\'layer_type\']\n            \n            # TODO: change hard code here\n            if current_type == \'long-term\':\n                gamma = 0.4  \n            elif current_type == \'mid-term\':\n                gamma = 0.3  \n            else:\n                gamma = 0.1\n            \n            score_current = refined_scores[i]\n            score_next = refined_scores[i+1]\n            \n            if score_current.shape[0] != score_next.shape[0]:\n                score_next_reshaped = score_next.view(1, 1, -1)\n                score_next_interp = F.interpolate(\n                    score_next_reshaped, \n                    size=score_current.shape[0], \n                    mode=\'linear\', \n                    align_corners=False\n                ).view(-1)\n                refined_scores[i] = (1 - gamma) * score_current + gamma * score_next_interp\n            else:\n                refined_scores[i] = (1 - gamma) * score_current + gamma * score_next\n\n        for layer_idx, score in enumerate(refined_scores):\n            config = layer_configs[layer_idx]\n            actual_num_keep = config[\'budget\']\n            start_idx = config[\'visual_start_idx\']\n            \n            topk_indices_relative = torch.topk(score, actual_num_keep, sorted=False)[1]\n            topk_indices_absolute = topk_indices_relative + start_idx\n            topk_indices_absolute_sorted = torch.sort(topk_indices_absolute)[0]\n            \n            keep_indices = torch.cat([\n                torch.arange(start_idx, device=device), \n                topk_indices_absolute_sorted\n            ]).tolist()\n            \n            keep_indices_all_layers.append(keep_indices)\n        \n        return keep_indices_all_layers\n    \n    @torch.inference_mode()\n    def pseudo_forward(self, local_question=None, global_question=None):\n        device = self.device\n        \n        if local_question is None:\n            local_question = "What is happening in the video?"\n        if global_question is None:\n            global_question = "What is the main topic of the video?"\n        \n        local_input_ids = self.processor.tokenizer(local_question).input_ids\n        local_input_ids = torch.as_tensor([local_input_ids], device=device, dtype=torch.int)\n\n        start_pos_per_layer = self._get_next_start_pos_per_layer()\n        q_len_local = local_input_ids.shape[1]\n        batch = local_input_ids.shape[0]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len_local, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids_local = self._build_position_ids(start_pos_per_layer[0], q_len_local, batch)\n\n        max_start = max(start_pos_per_layer) if start_pos_per_layer else 0\n        self._set_rotary_required_len(max_start + q_len_local)\n\n        use_flash_attn = (hasattr(self.language_model.config, \'_attn_implementation\') and \n                        self.language_model.config._attn_implementation in ["flash_attention_2", "sdpa"])\n\n        if use_flash_attn:\n            attn_weights_local = self._compute_attention_scores_manually(local_input_ids, self.kv_cache)\n        else:\n            out_local = self.language_model(\n                input_ids=local_input_ids,\n                use_cache=False,\n                past_key_values=self.kv_cache,\n                output_attentions=True,\n                position_ids=position_ids_local,\n                cache_position=None,\n                attention_mask=None,\n            )\n            attn_weights_local = out_local.attentions\n        \n        global_input_ids = self.processor.tokenizer(global_question).input_ids\n        global_input_ids = torch.as_tensor([global_input_ids], device=device, dtype=torch.int)\n        \n        q_len_global = global_input_ids.shape[1]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len_global, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids_global = self._build_position_ids(start_pos_per_layer[0], q_len_global, batch)\n        \n        self._set_rotary_required_len(max_start + q_len_global)\n        \n        if use_flash_attn:\n            attn_weights_global = self._compute_attention_scores_manually(global_input_ids, self.kv_cache)\n        else:\n            out_global = self.language_model(\n                input_ids=global_input_ids,\n                use_cache=False,\n                past_key_values=self.kv_cache,\n                output_attentions=True,\n                position_ids=position_ids_global,\n                cache_position=None,\n                attention_mask=None,\n            )\n            attn_weights_global = out_global.attentions\n        \n        mixed_question = local_question + "; " + global_question\n        mixed_input_ids = self.processor.tokenizer(mixed_question).input_ids\n        mixed_input_ids = torch.as_tensor([mixed_input_ids], device=device, dtype=torch.int)\n        \n        q_len_mixed = mixed_input_ids.shape[1]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len_mixed, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids_mixed = self._build_position_ids(start_pos_per_layer[0], q_len_mixed, batch)\n        \n        self._set_rotary_required_len(max_start + q_len_mixed)\n        \n        if use_flash_attn:\n            attn_weights_mixed = self._compute_attention_scores_manually(mixed_input_ids, self.kv_cache)\n        else:\n            out_mixed = self.language_model(\n                input_ids=mixed_input_ids,\n                use_cache=False,\n                past_key_values=self.kv_cache,\n                output_attentions=True,\n                position_ids=position_ids_mixed,\n                cache_position=None,\n                attention_mask=None,\n            )\n            attn_weights_mixed = out_mixed.attentions\n        \n        self._layer_position_ids.clear()\n        \n        print(f"GPU memory usage: {self.get_gpu_memory_usage_gb()} GB")\n        current_k_states_len = self.kv_cache[0][0].shape[2]\n        \n        keep_indices_all_layers = self.prune_kv_cache_by_attention(\n            attn_weights_local, attn_weights_global, attn_weights_mixed, \n            num_keep=self.kv_size\n        )\n        \n        if current_k_states_len > self.kv_size:\n            print(f"Applying KV-Cache compression due to k_states > {self.kv_size}")\n            self.apply_kv_cache_pruning_strict(keep_indices_all_layers)\n    \n    @torch.inference_mode()\n    def predict_and_compress(self):\n        local_question, global_question = self.predict_next_question()\n        self.pseudo_forward(local_question, global_question)\n\n    @torch.inference_mode()\n    def question_answering(self, input_text, max_new_tokens=128, temperature=0, repetition_penalty=1.1, pseudo_forward=False):\n        device = self.device\n        stop_token_ids = [self.processor.tokenizer.eos_token_id]\n        output_ids = []\n\n        start_time = time.perf_counter()\n        \n        prompt = input_text[\'prompt\']\n        input_ids = self.processor.tokenizer(prompt).input_ids\n        input_ids = torch.as_tensor([input_ids], device=device)\n\n        past_lens_prefill = self._get_cache_seq_len_per_layer()\n        start_pos_prefill = self._get_next_start_pos_per_layer()\n\n        inputs_embeds = self.get_input_embeddings()(input_ids)\n        q_len_prefill = inputs_embeds.shape[1]\n        batch = inputs_embeds.shape[0]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_prefill[layer_idx], q_len_prefill, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids = self._build_position_ids(start_pos_prefill[0], q_len_prefill, batch)\n\n        max_start_pos = max(start_pos_prefill) if start_pos_prefill else 0\n        self._set_rotary_required_len(max_start_pos + q_len_prefill)\n\n        out = self.language_model(\n            inputs_embeds=inputs_embeds,\n            use_cache=True,\n            past_key_values=self.kv_cache,\n            position_ids=position_ids,\n            cache_position=None,\n            attention_mask=None,\n        )\n        past_key_values = out.past_key_values\n        logits = out.logits\n        \n        self._append_position_ids(start_pos_prefill, q_len_prefill)\n        self._layer_position_ids.clear()\n\n        for step in range(max_new_tokens):\n            last_token_logits = logits[0, -1, :]\n\n            if repetition_penalty != 1.0 and len(output_ids) > 0:\n                for token_id in set(output_ids):\n                    if last_token_logits[token_id] < 0:\n                        last_token_logits[token_id] *= repetition_penalty\n                    else:\n                        last_token_logits[token_id] /= repetition_penalty\n\n            if temperature == 0.0:\n                _, indices = torch.topk(last_token_logits, 1)\n                token = int(indices[0])\n            else:\n                scaled_logits = last_token_logits / temperature\n                scaled_logits = torch.nan_to_num(\n                    scaled_logits, nan=-float(\'inf\'), posinf=float(\'inf\'), neginf=-float(\'inf\')\n                )\n                probs = F.softmax(scaled_logits, dim=-1)\n                probs = torch.nan_to_num(probs, nan=0.0, posinf=0.0, neginf=0.0)\n                probs_sum = probs.sum()\n                if probs_sum > 0:\n                    probs = probs / probs_sum\n                    token = torch.multinomial(probs, num_samples=1).item()\n                else:\n                    _, indices = torch.topk(last_token_logits, 1)\n                    token = int(indices[0])\n\n            output_ids.append(token)\n            if token in stop_token_ids:\n                break\n\n            curr_start_pos = self._get_next_start_pos_per_layer()\n            \n            max_curr_start = max(curr_start_pos) if curr_start_pos else 0\n            self._set_rotary_required_len(max_curr_start + 1)\n            \n            self._layer_position_ids.clear()\n            for layer_idx in range(self.num_layers):\n                pos_step = torch.tensor([curr_start_pos[layer_idx]], device=device, dtype=torch.long)\n                position_ids_layer = pos_step.view(1, 1)\n                self._layer_position_ids[layer_idx] = position_ids_layer\n            \n            pos_step = torch.tensor([curr_start_pos[0]], device=device, dtype=torch.long)\n            position_ids = pos_step.view(1, 1)\n\n            out = self.language_model(\n                input_ids=torch.as_tensor([[token]], device=device),\n                use_cache=True,\n                past_key_values=past_key_values,\n                position_ids=position_ids,\n                cache_position=None,\n                attention_mask=None,\n            )\n            \n            if (not pseudo_forward) and (step == 0):\n                end_time = time.perf_counter()\n                print(f"TTFT: {end_time - start_time} seconds")\n                \n            logits = out.logits\n            past_key_values = out.past_key_values\n\n            self._append_position_ids(curr_start_pos, 1)\n            self._layer_position_ids.clear()\n\n        output = self.processor.tokenizer.decode(\n            output_ids,\n            skip_special_tokens=True,\n            spaces_between_special_tokens=False,\n            clean_up_tokenization_spaces=True,\n        )\n\n        if not pseudo_forward:\n            current_question = input_text[\'question\']\n            current_options = None\n            formatted_question = input_text.get(\'formatted_question\', None)\n            if formatted_question:\n                option_matches = re.findall(r\'\\([A-Z]\\)\\s*(.+?)(?=\\n\\([A-Z]\\)|\\nThe best answer|\\n*$)\', formatted_question, re.DOTALL)\n                if option_matches:\n                    current_options = [opt.strip() for opt in option_matches]\n            self.conv_history.append((current_question, output, current_options))\n            logger.info(f"Saved conversation to history. Total conversations: {len(self.conv_history)}")\n\n        self._truncate_kv_cache(past_lens_prefill)\n        for layer_idx in range(self.num_layers):\n            if (self._position_ids_cache[layer_idx] is not None and \n                self._position_ids_cache[layer_idx].numel() > past_lens_prefill[layer_idx]):\n                self._position_ids_cache[layer_idx] = self._position_ids_cache[layer_idx][\n                    :past_lens_prefill[layer_idx]\n                ].contiguous()\n\n        new_lens = self._get_cache_seq_len_per_layer()\n        print(f"Answering Cache lengths: min={min(new_lens)}, max={max(new_lens)}")\n        return output\n    \n    def _truncate_kv_cache(self, target_lengths):\n        if self.kv_cache is None:\n            return\n        \n        truncated_cache = []\n        for layer_idx, (k_cache, v_cache) in enumerate(self.kv_cache):\n            if isinstance(target_lengths, int):\n                target_len = target_lengths\n            else:\n                target_len = target_lengths[layer_idx]\n            \n            truncated_k = k_cache[:, :, :target_len, :]\n            truncated_v = v_cache[:, :, :target_len, :]\n            truncated_cache.append((truncated_k, truncated_v))\n        \n        if isinstance(self.kv_cache, DynamicCache):\n            self.kv_cache = DynamicCache.from_legacy_cache(truncated_cache)\n        else:\n            self.kv_cache = truncated_cache\n\n\ndef load_model(model_path=\'llava-onevision-qwen2-7b-ov-hf\',\n               n_init=None, kv_size=None, streaming=True, device="cuda", sample_fps=0.5):\n    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)\n    processor.tokenizer.padding_side = \'left\'\n    \n    system_prompt = \'<|im_start|>system \\nYou are a helpful assistant.<|im_end|><|im_start|>user \'\n    init_prompt_ids = processor.tokenizer(system_prompt, return_tensors="pt").input_ids.to(device)\n    \n    base_model = LlavaOnevisionForConditionalGeneration.from_pretrained(\n        model_path,\n        device_map="auto",\n        torch_dtype=torch.float16,\n        trust_remote_code=True,\n    )\n\n    model = LlavaOneVision_Hermes.__new__(LlavaOneVision_Hermes)\n    model.__dict__ = base_model.__dict__.copy()\n\n    Abstract_Hermes.__init__(\n        model, \n        processor, \n        init_prompt_ids.tolist(), \n        kv_size,\n    )\n    model.streaming = streaming\n    \n    num_layers = base_model.language_model.config.num_hidden_layers\n    model.num_layers = num_layers\n    model._position_ids_cache = [None for _ in range(num_layers)]\n    \n    model.short_term_ratio = 0.1\n    model.long_term_ratio = 0.3\n    model.short_term_threshold = int(model.num_layers * model.short_term_ratio)\n    model.long_term_threshold = int(model.num_layers * (1 - model.long_term_ratio))\n    model.token_activity_cache = [None for _ in range(num_layers)]\n    \n    model.total_processed_frames = 0\n    \n    model._layer_position_ids = {}\n    model._hook_handles = []\n    model._register_forward_hooks()\n    model._patch_rotary_embeddings()\n    \n    logger.info(f\'n_init: {init_prompt_ids.shape[1] if n_init is None else n_init}\')\n    logger.info(f\'kv_size: {kv_size}\')\n\n    model.eval()\n\n    return model, processor\n', 'pinned_source/inference/abstract_hermes.py': 'import torch\n\nclass Abstract_Hermes:\n    kv_cache = None\n\n    def __init__(self, processor, init_prompt_ids, kv_size):\n        self.processor = processor\n        self.init_prompt_ids = init_prompt_ids\n        self.kv_size = kv_size\n        self.last_encoded_frames = 0\n        self.visual_start_idx = 14\n        self.conv_history = []\n        \n    def clear_cache(self):\n        self.kv_cache = None\n        self.conv_history = []\n        torch.cuda.empty_cache()\n        torch.cuda.ipc_collect()\n\n    def encode_init_prompt(self):\n        if not isinstance(self.init_prompt_ids, torch.Tensor):\n            self.init_prompt_ids = torch.as_tensor(self.init_prompt_ids, device=self.device)\n        output = self.language_model(input_ids=self.init_prompt_ids, use_cache=True, return_dict=True)\n        self.kv_cache = output.past_key_values\n        self.visual_start_idx = self.kv_cache[0][0].shape[2]\n\n    def get_prompt(self, query, mc=False):\n        prompt = f"\\n{query}<|im_end|><|im_start|>assistant\\n"\n        \n        if mc:\n            prompt += \'Best option: (\'\n        return prompt\n\n    def get_gpu_memory_usage_gb(self):\n        return torch.cuda.max_memory_allocated() / (1024**3)', 'pinned_source/inference/reindex_1d.py': 'import torch\nfrom typing import Tuple\nfrom transformers import DynamicCache\n\n\ndef get_cache_seq_len(past_key_values) -> int:\n    if past_key_values is None:\n        return 0\n    if isinstance(past_key_values, DynamicCache):\n        return past_key_values.get_seq_length()\n    return past_key_values[0][0].shape[2]\n\n\ndef contiguous_kv(past_key_values):\n    new_legacy = []\n    for (k_layer, v_layer) in past_key_values:\n        new_legacy.append((k_layer.contiguous(), v_layer.contiguous()))\n    return new_legacy\n\n\ndef _get_rotary_module(llm) -> torch.nn.Module:\n    if hasattr(llm, "rotary_emb"):\n        return llm.rotary_emb\n    if hasattr(llm, "model") and hasattr(llm.model, "rotary_emb"):\n        return llm.model.rotary_emb\n\n    if hasattr(llm, "model") and hasattr(llm.model, "layers"):\n        if len(llm.model.layers) > 0 and hasattr(llm.model.layers[0], "self_attn"):\n            if hasattr(llm.model.layers[0].self_attn, "rotary_emb"):\n                return llm.model.layers[0].self_attn.rotary_emb\n    raise AttributeError("Cannot find rotary_emb module on language_model")\n\n\ndef rotate_half(x):\n    x1 = x[..., : x.shape[-1] // 2]\n    x2 = x[..., x.shape[-1] // 2 :]\n    return torch.cat((-x2, x1), dim=-1)\n\n\ndef apply_rotary_pos_emb_1d(x, cos, sin):\n    if cos.dim() == 3 and x.dim() == 4:\n        cos = cos.unsqueeze(1)  # [batch, 1, seq_len, head_dim]\n        sin = sin.unsqueeze(1)  # [batch, 1, seq_len, head_dim]\n    \n    x_embed = (x * cos) + (rotate_half(x) * sin)\n    return x_embed\n\n\ndef rotary_delta(cos_old, sin_old, cos_new, sin_new):\n    cos_delta = cos_new * cos_old + sin_new * sin_old\n    sin_delta = sin_new * cos_old - cos_new * sin_old\n    return cos_delta, sin_delta\n\n\n@torch.inference_mode()\ndef compute_cos_sin_for_positions(\n    llm, \n    seq_len: int, \n    position_ids_1d: torch.Tensor, \n    dtype: torch.dtype, \n    device: torch.device\n) -> Tuple[torch.Tensor, torch.Tensor]:\n    \n    rotary_emb = _get_rotary_module(llm)\n    \n    config = llm.config if hasattr(llm, "config") else llm.model.config\n    hidden_size = getattr(config, "hidden_size", 4096)\n    \n    position_ids = position_ids_1d.view(1, -1).to(device)\n    \n    dummy_h = torch.zeros((1, seq_len, hidden_size), device=device, dtype=dtype)\n    \n    \n    seq_len_rope = position_ids.max().item() + 1\n    cos, sin = rotary_emb(dummy_h, seq_len=seq_len_rope)\n    cos = cos[position_ids]\n    sin = sin[position_ids]\n    \n    cos = cos.to(dtype)\n    sin = sin.to(dtype)\n    \n    return cos, sin\n\n@torch.inference_mode()\ndef compute_cos_sin_with_custom_base(\n    llm,\n    seq_len: int,\n    position_ids_1d: torch.Tensor,\n    dtype: torch.dtype,\n    device: torch.device,\n    rope_base: float = 10000.0\n) -> Tuple[torch.Tensor, torch.Tensor]:\n    \n    config = llm.config if hasattr(llm, "config") else llm.model.config\n    hidden_size = getattr(config, "hidden_size", 4096)\n    num_attention_heads = getattr(config, "num_attention_heads", 32)\n    head_dim = hidden_size // num_attention_heads\n    \n    \n    inv_freq = 1.0 / (rope_base ** (torch.arange(0, head_dim, 2, device=device, dtype=torch.float32) / head_dim))\n    \n    freqs = torch.outer(position_ids_1d.float(), inv_freq)\n    \n    \n    emb = torch.cat([freqs, freqs], dim=-1)\n    \n    cos = emb.cos().unsqueeze(0).to(dtype)  # [1, seq_len, head_dim]\n    sin = emb.sin().unsqueeze(0).to(dtype)  # [1, seq_len, head_dim]\n    \n    return cos, sin\n\n\ndef get_layer_rope_base(\n    layer_idx: int,\n    num_layers: int,\n    short_term_ratio: float = 0.3,\n    long_term_ratio: float = 0.3,\n    short_term_base: float = 10000.0,\n    mid_term_base_range: Tuple[float, float] = (10000.0, 50000.0),\n    long_term_base: float = 50000.0,\n) -> float:\n    \n    short_term_threshold = int(num_layers * short_term_ratio)\n    long_term_threshold = int(num_layers * (1 - long_term_ratio))\n    \n    if layer_idx < short_term_threshold:\n        \n        return short_term_base\n    elif layer_idx >= long_term_threshold:\n        \n        return long_term_base\n    else:\n        \n        mid_start = short_term_threshold\n        mid_end = long_term_threshold\n        progress = (layer_idx - mid_start) / max(mid_end - mid_start, 1)\n        base_start, base_end = mid_term_base_range\n        return base_start + (base_end - base_start) * progress\n\n\n@torch.inference_mode()\ndef apply_rotary_delta_to_keys_only(\n    key_states: torch.Tensor, \n    cos_delta: torch.Tensor, \n    sin_delta: torch.Tensor\n) -> torch.Tensor:\n    \n    k_rot = apply_rotary_pos_emb_1d(key_states, cos_delta, sin_delta)\n    return k_rot\n\n', 'provenance/twinpast_base.py': '"""Run the frozen, annotation-guided TwinPast development diagnostic."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport shutil\nimport statistics\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nfrom zipfile import ZIP_DEFLATED, ZipFile\n\nARTIFACT = "HERMES_twinpast_diagnostic_evidence.zip"\nARMS = ("question_only", "recent_four", "history_a", "history_b")\nORDERS = ("canonical", "reversed")\nLETTERS = "ABCDEFGH"\nHOSTS = ("huggingface.co", "hf-mirror.com")\n\n\ndef sha(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef digest(path):\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef atomic(path, data):\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef fetch(bundle, relative, dest, expected_sha, size=None, reuse=None):\n    import requests\n\n    if dest.is_file() and digest(dest) == expected_sha:\n        return\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    temporary = dest.with_suffix(".part")\n    if reuse and reuse.is_file() and digest(reuse) == expected_sha:\n        shutil.copyfile(reuse, temporary)\n        temporary.replace(dest)\n        print(f"MEDIA_REUSED {relative}", flush=True)\n        return\n    cap = size if size is not None else 2 * 1024**2\n    remaining_attempts = [(host, direct) for direct in (False, True) for host in HOSTS]\n    for host, direct in remaining_attempts:\n        url = f"https://{host}/datasets/{bundle[\'manifest\'][\'dataset\']}/resolve/{bundle[\'manifest\'][\'revision\']}/{relative}"\n        try:\n            with requests.Session() as session:\n                session.trust_env = not direct\n                with session.get(url, stream=True, timeout=(8, 15)) as response, temporary.open("wb") as stream:\n                    response.raise_for_status()\n                    start, last_report, count = time.monotonic(), time.monotonic(), 0\n                    for block in response.iter_content(128 * 1024):\n                        count += len(block)\n                        now = time.monotonic()\n                        if count > cap or now - start > 120:\n                            raise ValueError("Download exceeded size or elapsed-time bound")\n                        stream.write(block)\n                        if now - last_report >= 8:\n                            print(f"MEDIA_PROGRESS {relative} {count / 1024**2:.1f} MiB", flush=True)\n                            last_report = now\n            if (size is not None and count != size) or digest(temporary) != expected_sha:\n                raise ValueError("Downloaded size or SHA-256 does not match")\n            temporary.replace(dest)\n            print(f"MEDIA_READY {relative} {count / 1024**2:.1f} MiB", flush=True)\n            return\n        except (requests.RequestException, OSError, ValueError) as exc:\n            print(f"MEDIA_RETRY {relative} host={host} direct={direct} reason={type(exc).__name__}", flush=True)\n    raise RuntimeError(f"Could not acquire {relative}. Verified completed files remain available for rerun.")\n\n\ndef cases(bundle):\n    plans = {row["pair_id"]: row for row in bundle["manifest"]["selected_pairs"]}\n    visual = {row["pair_id"]: row for row in bundle["visual"]}\n    media = {row["pair_id"]: row for row in bundle["media"]}\n    assert len(plans) == len(visual) == len(media) == 12 and plans.keys() == visual.keys() == media.keys()\n    selected = []\n    for pair_id, plan in plans.items():\n        if visual[pair_id]["status"] == "clear_contrast":\n            checked = media[pair_id]\n            assert checked["status"] == "checked" and checked["task"] == plan["task"]\n            assert checked["grid_s"] == sorted(set(checked["grid_s"])) and len(checked["grid_s"]) == 6\n            assert checked["grid_s"][-4:] == plan["latest_four_grid_s"]\n            assert checked["grid_s"][-1] < plan["query_time_s"]\n            for branch in "ab":\n                assert all(checked["decoded"][branch]["early_samples_inside"])\n            assert checked["decoded"]["a"]["decoded_rgb_sha256"][-4:] == checked["decoded"]["b"]["decoded_rgb_sha256"][-4:]\n            selected.append((plan, checked, visual[pair_id]))\n    assert len(selected) == 9\n    return selected\n\n\ndef annotation_rows(bundle, raw):\n    assert sha(raw) == bundle["manifest"]["annotation_sha256"], "Pinned annotations changed"\n    rows = [json.loads(line) for line in raw.splitlines() if line.strip()]\n    result = {row["pair_id"]: row for row in rows}\n    assert len(rows) == len(result) == bundle["manifest"]["dataset_pairs"]\n    for plan, media, visual in cases(bundle):\n        row = result[plan["pair_id"]]\n        assert all(row[field] == plan[field] for field in ("video_a", "video_b", "task"))\n        assert row["t_query"] == plan["query_time_s"] and row["question"].strip()\n        choices = row["options"]\n        assert 2 <= len(choices) <= len(LETTERS) and all(c.strip() for c in choices)\n        assert len({c.strip().casefold() for c in choices}) == len(choices)\n        assert row["answer_a"] != row["answer_b"]\n        for branch in "ab":\n            assert row[f"answer_{branch}"] in LETTERS[:len(choices)]\n            assert row[f"answer_{branch}"] == visual[f"observed_option_{branch}"]\n            seconds = media["decoded"][branch]["frame_start_s"]\n            assert all(t < row["t_query"] for t in seconds)\n            assert all(any(lo - 1e-6 <= t <= hi + 1e-6\n                           for lo, hi in row["evidence_intervals"][branch.upper()]) for t in seconds[:2])\n    return result\n\n\ndef decode(path, recorded):\n    from decord import VideoReader\n\n    reader = VideoReader(str(path), num_threads=1)\n    frames = reader.get_batch(recorded["source_frames"]).asnumpy()\n    assert [sha(frame.tobytes()) for frame in frames] == recorded["decoded_rgb_sha256"], (\n        f"Decoded frames differ from the frozen audit for {path.name}")\n    return frames\n\n\ndef features(model, frames):\n    import torch\n\n    chw = torch.from_numpy(frames).permute(0, 3, 1, 2)\n    pixels = model.processor.video_processor(chw, return_tensors="pt").pixel_values_videos\n    output = model.get_video_features(pixels.to(model.device, model.dtype))\n    assert output.shape[0] == 1 and output.shape[1] % len(frames) == 0\n    return output\n\n\ndef score(model, system, prefix, option_ids, visual, cached=False):\n    import torch\n\n    lm = model.language_model\n    embed = lm.get_input_embeddings()\n    question = embed(prefix)\n    left = system if visual is None else torch.cat((system, visual), dim=1)\n    if cached:\n        initial = lm(inputs_embeds=left, use_cache=True, return_dict=True)\n        output = lm(inputs_embeds=question, past_key_values=initial.past_key_values, use_cache=True, return_dict=True)\n    else:\n        output = lm(inputs_embeds=torch.cat((left, question), dim=1), use_cache=False, return_dict=True)\n    result = output.logits[0, -1, option_ids].float().detach().cpu()\n    assert torch.isfinite(result).all()\n    return result, left.shape[1] + question.shape[1]\n\n\ndef make_prompt(model, question, choices, order, with_video):\n    import torch\n\n    perm = list(range(len(choices)))\n    if order == "reversed":\n        perm.reverse()\n    letters = LETTERS[:len(choices)]\n    context = "Answer the question about the video." if with_video else "Answer the question."\n    options = "\\n".join(f"({letter}) {choices[i]}" for letter, i in zip(letters, perm, strict=True))\n    text = model.get_prompt(f"{context}\\nQuestion: {question}\\nOptions:\\n{options}\\n"\n                            "Reply with only the letter of the best option.") + "Answer letter:\\n"\n    tokenizer = model.processor.tokenizer\n    ids = tokenizer.encode(text, add_special_tokens=False)\n    tokens = []\n    for letter in letters:\n        suffix = tokenizer.encode(letter, add_special_tokens=False)\n        assert len(suffix) == 1 and tokenizer.encode(text + letter, add_special_tokens=False) == ids + suffix\n        tokens.extend(suffix)\n    assert len(set(tokens)) == len(tokens)\n    return torch.tensor([ids], device=model.device), tokens, perm, sha(text.encode("utf-8"))\n\n\ndef evaluate_pair(model, plan, media, row, paths, preflight=False):\n    import numpy as np\n    import torch\n\n    frames = {branch: decode(paths[branch], media["decoded"][branch]) for branch in "ab"}\n    assert np.array_equal(frames["a"][2:], frames["b"][2:])\n    recent = features(model, frames["a"][2:])\n    a = features(model, frames["a"][:2])\n    b = features(model, frames["b"][:2])\n    assert a.shape == b.shape and a.shape[1] * 2 == recent.shape[1]\n    visuals = {"question_only": None, "recent_four": recent,\n               "history_a": torch.cat((a, recent), dim=1), "history_b": torch.cat((b, recent), dim=1)}\n    lm = model.language_model\n    system = lm.get_input_embeddings()(torch.as_tensor(model.init_prompt_ids, device=model.device))\n    results, checks = [], []\n    for order in ORDERS:\n        prompt = make_prompt(model, row["question"], row["options"], order, True)\n        for arm in ARMS:\n            ids, option_ids, perm, prompt_sha = prompt if arm != "question_only" else make_prompt(\n                model, row["question"], row["options"], order, False)\n            started = time.perf_counter()\n            output, count = score(model, system, ids, option_ids, visuals[arm])\n            elapsed = time.perf_counter() - started\n            if preflight:\n                replay, replay_count = score(model, system, ids, option_ids, visuals[arm])\n                torch.testing.assert_close(output, replay, atol=0, rtol=0)\n                assert replay_count == count\n                if arm == "history_a":\n                    cached, _ = score(model, system, ids, option_ids, visuals[arm], cached=True)\n                    torch.testing.assert_close(output, cached, atol=0.025, rtol=0.005)\n                    assert output.argmax().item() == cached.argmax().item(), "Native-cache top option differs"\n                    checks.append(dict(order=order, native_cache_max_abs=float((output - cached).abs().max())))\n            values = [0.0] * len(perm)\n            for idx, value in zip(perm, output.tolist(), strict=True):\n                values[idx] = value\n            # Canonical tie-breaking prevents label order from deciding exact ties.\n            pred = LETTERS[max(range(len(values)), key=values.__getitem__)]\n            recent_sha = media["decoded"]["a"]["decoded_rgb_sha256"][-4:] if arm != "question_only" else []\n            earlier_sha = media["decoded"][arm[-1]]["decoded_rgb_sha256"][:2] if arm.startswith("history") else []\n            results.append(dict(pair_id=plan["pair_id"], task=plan["task"], order=order, arm=arm, prediction=pred,\n                                gold_a=row["answer_a"], gold_b=row["answer_b"], scores=values, permutation=perm,\n                                label_token_ids=option_ids, prompt_sha256=prompt_sha, lm_tokens=count,\n                                earlier_frame_sha256=earlier_sha, recent_frame_sha256=recent_sha,\n                                forward_s=elapsed))\n    return results, checks\n\n\ndef summarize(blocks):\n    pairs = []\n    for block in blocks:\n        records = {(r["order"], r["arm"]): r for r in block}\n        pair = dict(pair_id=block[0]["pair_id"], task=block[0]["task"])\n        for order in ORDERS:\n            a, b = (records[order, arm] for arm in ("history_a", "history_b"))\n            assert a["lm_tokens"] == b["lm_tokens"] and a["prompt_sha256"] == b["prompt_sha256"]\n            assert a["recent_frame_sha256"] == b["recent_frame_sha256"]\n            ai, bi = (LETTERS.index(a[f"gold_{branch}"]) for branch in "ab")\n            ma = a["scores"][ai] - a["scores"][bi]\n            mb = b["scores"][ai] - b["scores"][bi]\n            wins = int(a["prediction"] == a["gold_a"]) + int(b["prediction"] == b["gold_b"])\n            reversal = a["prediction"] != b["prediction"]\n            pair[order] = dict(correct_branches=wins, correct_reversal=wins == 2, unchanged=not reversal,\n                               wrong_reversal=reversal and wins != 2, margin_a=ma, margin_b=mb,\n                               directional_change=ma - mb)\n            for arm in ("question_only", "recent_four"):\n                prediction = records[order, arm]["prediction"]\n                pair[order][f"{arm}_correct_branches"] = int(prediction in (a["gold_a"], a["gold_b"]))\n        pair["both_orders_correct"] = all(pair[order]["correct_reversal"] for order in ORDERS)\n        pair["predictions_order_stable"] = {\n            arm: records["canonical", arm]["prediction"] == records["reversed", arm]["prediction"] for arm in ARMS\n        }\n        pairs.append(pair)\n    orders = {order: dict(\n        both_correct=sum(row[order]["correct_reversal"] for row in pairs),\n        correct_branches=sum(row[order]["correct_branches"] for row in pairs),\n        question_only_correct_branches=sum(row[order]["question_only_correct_branches"] for row in pairs),\n        recent_four_correct_branches=sum(row[order]["recent_four_correct_branches"] for row in pairs),\n        wrong_reversals=sum(row[order]["wrong_reversal"] for row in pairs),\n        unchanged=sum(row[order]["unchanged"] for row in pairs),\n        median_directional_change=statistics.median(row[order]["directional_change"] for row in pairs))\n        for order in ORDERS} if pairs else {}\n    robust = sum(row["both_orders_correct"] for row in pairs)\n    numeric_gate = len(pairs) == 9 and robust >= 5 and all(x["median_directional_change"] > 0 for x in orders.values())\n    return dict(pairs=len(pairs), robust_correct_pairs=robust, order_summary=orders, per_pair=pairs,\n                numeric_development_gate=numeric_gate,\n                advancement="pending source-group verification" if numeric_gate else\n                ("stop on this backbone if diagnostic validity holds" if len(pairs) == 9 and robust == 0\n                 else "review diagnostic outcomes before advancing"),\n                source_groups="The pinned annotations do not identify original source groups. Verify independently.")\n\n\ndef package(work):\n    artifact = work / ARTIFACT\n    temporary = work / (ARTIFACT + ".tmp")\n    files = [path for name in ("exports", "checkpoints", "provenance", "pinned_source", "logs")\n             for path in (work / name).rglob("*") if path.is_file() and path.suffix != ".tmp"]\n    files += [path for path in (work / "launch.json", work / "completion.json") if path.exists()]\n    with ZipFile(temporary, "w", ZIP_DEFLATED) as archive:\n        entries = []\n        for path in sorted(files):\n            raw = path.read_bytes()\n            relative = path.relative_to(work).as_posix()\n            archive.writestr(relative, raw)\n            entries.append(dict(path=relative, bytes=len(raw), sha256=sha(raw)))\n        archive.writestr("evidence_manifest.json", json.dumps(entries, indent=2))\n    temporary.replace(artifact)\n    print(f"EVIDENCE_ZIP={artifact}", flush=True)\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    bundle_path = work / "provenance" / "bundle.json"\n    bundle = json.loads(bundle_path.read_text(encoding="utf-8"))\n    selection = cases(bundle)\n    dataset = work / "media" / "data.jsonl"\n    fetch(bundle, "data.jsonl", dataset, bundle["manifest"]["annotation_sha256"])\n    rows = annotation_rows(bundle, dataset.read_bytes())\n    atomic(work / "provenance" / "selected_annotations.json", [rows[plan["pair_id"]] for plan, _, _ in selection])\n    needed = sum(meta["bytes"] for plan, media, _ in selection\n                 for branch, meta in media["decoded"].items()\n                 if not (work / "media" / plan[f"video_{branch}"]).is_file()\n                 or (work / "media" / plan[f"video_{branch}"]).stat().st_size != meta["bytes"])\n    assert shutil.disk_usage(work).free >= needed + 256 * 1024**2, "Free data-disk space before running"\n    locations = {}\n    print(f"MEDIA_PLAN 18 pinned clips, up to {needed / 1024**2:.1f} MiB not present", flush=True)\n    with ThreadPoolExecutor(max_workers=3) as pool:\n        pending = []\n        for plan, media, _ in selection:\n            locations[plan["pair_id"]] = {}\n            for branch, meta in media["decoded"].items():\n                relative = plan[f"video_{branch}"]\n                dest = work / "media" / relative\n                reuse = args.media_reuse / Path(relative).relative_to("videos")\n                pending.append(pool.submit(fetch, bundle, relative, dest, meta["sha256"], meta["bytes"], reuse))\n                locations[plan["pair_id"]][branch] = dest\n        for download in pending:\n            download.result()\n    settings = dict(bundle_sha256=digest(bundle_path), runner_sha256=digest(Path(__file__)),\n                    model_name=args.model.name, torch_version=torch.__version__,\n                    gpu=torch.cuda.get_device_name(0), cuda=torch.version.cuda,\n                    driver=torch.cuda.driver_version() if hasattr(torch.cuda, "driver_version") else None)\n    identity = sha(json.dumps(settings, sort_keys=True).encode())\n    checkpoint_dir = work / "checkpoints"\n    checkpoint_dir.mkdir(exist_ok=True)\n    atomic(work / "exports" / "run_config.json", settings)\n    print("LOADING_PINNED_MODEL", args.model.name, flush=True)\n    model, _ = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.device.type == "cuda"\n    blocks = []\n    with torch.inference_mode():\n        for index, (plan, media, _) in enumerate(selection):\n            pair_id = plan["pair_id"]\n            checkpoint = checkpoint_dir / f"{pair_id}.json"\n            if index and checkpoint.is_file():\n                saved = json.loads(checkpoint.read_text(encoding="utf-8"))\n                assert saved["identity"] == identity and saved["pair_id"] == pair_id, "Use a new WORK for changed settings"\n                records = saved["records"]\n                assert len(records) == 8 and {(r["order"], r["arm"]) for r in records} == {\n                    (order, arm) for order in ORDERS for arm in ARMS}\n                assert all(r["pair_id"] == pair_id for r in records)\n                print(f"PAIR_REUSED {index + 1}/9 {pair_id}", flush=True)\n            else:\n                print(f"PAIR_RUNNING {index + 1}/9 {pair_id}", flush=True)\n                records, checks = evaluate_pair(model, plan, media, rows[pair_id], locations[pair_id], preflight=index == 0)\n                if index == 0:\n                    if checkpoint.exists():\n                        assert json.loads(checkpoint.read_text(encoding="utf-8"))["identity"] == identity, (\n                            "Use a new WORK for changed settings")\n                    atomic(work / "exports" / "preflight.json", dict(status="pass", pair_id=pair_id, checks=checks))\n                    print("TWINPAST_PREFLIGHT_PASS", flush=True)\n                atomic(checkpoint, dict(identity=identity, pair_id=pair_id, records=records))\n                print(f"PAIR_DONE {index + 1}/9 {pair_id}", flush=True)\n            blocks.append(records)\n    summary = summarize(blocks)\n    atomic(work / "exports" / "summary.json", summary)\n    fields = ("pair_id", "task", "order", "arm", "prediction", "gold_a", "gold_b", "scores", "permutation",\n              "label_token_ids", "prompt_sha256", "lm_tokens", "earlier_frame_sha256", "recent_frame_sha256", "forward_s")\n    with (work / "exports" / "scores.csv").open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows({key: json.dumps(value) if isinstance(value, list) else value for key, value in row.items()}\n                         for block in blocks for row in block)\n    atomic(work / "completion.json", dict(status="complete", pairs=len(blocks), scores=sum(map(len, blocks))))\n    print(f"TWINPAST_DIAGNOSTIC_COMPLETE pairs={len(blocks)} scores={sum(map(len, blocks))}", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--work", required=True, type=Path)\n    parser.add_argument("--model", required=True, type=Path)\n    parser.add_argument("--media-reuse", type=Path, default=Path("/root/autodl-tmp/twinpast-development"))\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            package(args.work)\n\n\nif __name__ == "__main__":\n    main()\n', 'provenance/history_only_runner.py': '"""Test whether earlier evidence alone supports the frozen TwinPast answers."""\nimport argparse\nfrom concurrent.futures import ThreadPoolExecutor\nimport json\nfrom pathlib import Path\nimport time\n\nimport twinpast_base as base\n\nARTIFACT = "HERMES_twinpast_history_only_evidence.zip"\nDENSE_COUNT = 6\n\n\ndef dense_indices(seconds, intervals, earlier, cutoff):\n    """Keep the two original frames and add four spread across valid native timestamps."""\n    import numpy as np\n\n    mask = np.zeros(len(seconds), dtype=bool)\n    for start, stop in intervals:\n        mask |= (seconds >= start - 1e-6) & (seconds <= stop + 1e-6)\n    eligible = np.flatnonzero(mask & (seconds < cutoff)).tolist()\n    assert sorted(set(earlier)) == earlier and len(earlier) == 2\n    assert set(earlier) <= set(eligible), "Frozen history frame outside eligible annotated evidence"\n    if len(eligible) < DENSE_COUNT:\n        raise ValueError(f"Only {len(eligible)} distinct annotated frames before recent context")\n    chosen = set(earlier)\n    for k in range(DENSE_COUNT - len(earlier)):\n        target = k * (len(eligible) - 1) / (DENSE_COUNT - len(earlier) - 1)\n        pos = min((i for i, ix in enumerate(eligible) if ix not in chosen), key=lambda i: abs(i - target))\n        chosen.add(eligible[pos])\n    return sorted(chosen)\n\n\ndef prepare_dense(paths, annotated, media):\n    import numpy as np\n    from decord import VideoReader\n\n    result = {}\n    cutoff = min(annotated["t_query"], *(media["decoded"][b]["frame_start_s"][2] for b in "ab"))\n    for branch in "ab":\n        reader = VideoReader(str(paths[branch]), num_threads=1)\n        seconds = np.asarray(reader.get_frame_timestamp(list(range(len(reader)))), dtype=np.float64)\n        seconds -= seconds[0, 0]\n        seconds = seconds[:, 0]\n        indices = dense_indices(seconds, annotated["evidence_intervals"][branch.upper()],\n                                media["decoded"][branch]["source_frames"][:2], cutoff)\n        frames = reader.get_batch(indices).asnumpy()\n        result[branch] = dict(frames=frames, indices=indices, seconds=seconds[indices].tolist(),\n                              cutoff_s=cutoff, hashes=[base.sha(frame.tobytes()) for frame in frames])\n    return result\n\n\ndef contact_sheet(path, pair_id, condition, windows):\n    from PIL import Image, ImageDraw\n\n    cols = len(windows["a"]["frames"])\n    canvas = Image.new("RGB", (320 * cols, 450), "white")\n    draw = ImageDraw.Draw(canvas)\n    draw.text((8, 5), f"{pair_id} | {condition} | Native frames resized to fit", fill="black")\n    for row, branch in enumerate("ab"):\n        for col, (frame, seconds) in enumerate(zip(windows[branch]["frames"], windows[branch]["seconds"], strict=True)):\n            x, y = col * 320, 23 + row * 210\n            picture = Image.fromarray(frame)\n            picture.thumbnail((320, 180))\n            canvas.paste(picture, (x, y))\n            draw.text((x + 4, y + 182), f"{branch.upper()} | {seconds:.3f} s", fill="black")\n    path.parent.mkdir(exist_ok=True)\n    canvas.save(path, quality=90)\n\n\ndef evaluate(model, annotated, windows, reference, preflight=False):\n    import torch\n\n    system = model.language_model.get_input_embeddings()(\n        torch.as_tensor(model.init_prompt_ids, device=model.device))\n    visual = {branch: base.features(model, window["frames"]) for branch, window in windows.items()}\n    assert visual["a"].shape == visual["b"].shape\n    rows, checks = [], []\n    for order in base.ORDERS:\n        ids, option_ids, perm, prompt_sha = base.make_prompt(\n            model, annotated["question"], annotated["options"], order, True)\n        for branch in "ab":\n            old = reference[order, branch]\n            assert old["prompt_sha256"] == prompt_sha and old["label_token_ids"] == option_ids\n            start = time.perf_counter()\n            output, length = base.score(model, system, ids, option_ids, visual[branch])\n            elapsed = time.perf_counter() - start\n            if preflight:\n                repeat, _ = base.score(model, system, ids, option_ids, visual[branch])\n                torch.testing.assert_close(output, repeat, rtol=0, atol=0)\n                if branch == "a":\n                    cached, _ = base.score(model, system, ids, option_ids, visual[branch], cached=True)\n                    torch.testing.assert_close(output, cached, rtol=0.005, atol=0.025)\n                    assert output.argmax().item() == cached.argmax().item()\n                    checks.append(dict(order=order, max_native_cache_diff=float((output - cached).abs().max())))\n            values = [0.0] * len(perm)\n            for i, value in zip(perm, output.tolist(), strict=True):\n                values[i] = value\n            prediction = base.LETTERS[max(range(len(values)), key=values.__getitem__)]\n            gold = annotated[f"answer_{branch}"]\n            window = windows[branch]\n            rows.append(dict(order=order, branch=branch, prediction=prediction, gold=gold, scores=values,\n                             prompt_sha256=prompt_sha, lm_tokens=length, frame_indices=window["indices"],\n                             frame_seconds=window["seconds"], frame_hashes=window["hashes"],\n                             prior_prediction=old["prediction"], prior_scores=old["scores"], forward_s=elapsed))\n        assert rows[-1]["lm_tokens"] == rows[-2]["lm_tokens"]\n    return rows, checks\n\n\ndef outcome(rows, annotated):\n    wins = {}\n    ai, bi = (base.LETTERS.index(annotated[f"answer_{b}"]) for b in "ab")\n    for order in base.ORDERS:\n        a, b = (next(r for r in rows if r["order"] == order and r["branch"] == branch) for branch in "ab")\n        margin = lambda x: x[ai] - x[bi]\n        wins[order] = dict(branches_correct=int(a["prediction"] == a["gold"]) + int(b["prediction"] == b["gold"]),\n                           directional_change=margin(a["scores"]) - margin(b["scores"]))\n    return dict(orders=wins, robust=all(row["branches_correct"] == 2 for row in wins.values()))\n\n\ndef acquire(args, bundle, selection):\n    paths = {}\n    with ThreadPoolExecutor(max_workers=3) as pool:\n        downloads = []\n        for plan, media, _ in selection:\n            pair_paths = paths[plan["pair_id"]] = {}\n            for branch, check in media["decoded"].items():\n                relative = Path(plan[f"video_{branch}"])\n                reused = args.previous / "media" / relative\n                if reused.is_file() and base.digest(reused) == check["sha256"]:\n                    pair_paths[branch] = reused\n                    print(f"MEDIA_REUSED {relative.as_posix()}", flush=True)\n                else:\n                    dest = args.work / "media" / relative\n                    downloads.append(pool.submit(base.fetch, bundle, relative.as_posix(), dest, check["sha256"],\n                                                 check["bytes"], args.media_reuse / relative.relative_to("videos")))\n                    pair_paths[branch] = dest\n        for download in downloads:\n            download.result()\n    return paths\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work / "provenance"\n    bundle = json.loads((provenance / "bundle.json").read_bytes())\n    prior = json.loads((provenance / "prior.json").read_bytes())\n    selection = base.cases(bundle)\n    annotations = {r["pair_id"]: r for r in prior["annotations"]}\n    assert annotations.keys() == {plan["pair_id"] for plan, _, _ in selection}\n    paths = acquire(args, bundle, selection)\n    print("LOADING_PINNED_MODEL", args.model.name, flush=True)\n    model, _ = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.device.type == "cuda"\n    config = dict(model_revision=args.model.name, gpu=torch.cuda.get_device_name(0), torch=torch.__version__,\n                  cuda=torch.version.cuda, provenance_hashes={p.name: base.digest(p) for p in sorted(provenance.iterdir())})\n    ident = base.sha(json.dumps(config, sort_keys=True).encode())\n    exports = work / "exports"\n    checkpoints = work / "checkpoints"\n    checkpoints.mkdir(exist_ok=True)\n    base.atomic(exports / "run_config.json", config)\n    reports = []\n    with torch.inference_mode():\n        for index, (plan, media, screened) in enumerate(selection):\n            pair_id = plan["pair_id"]\n            annotated = annotations[pair_id]\n            assert all(annotated[f"answer_{b}"] == screened[f"observed_option_{b}"] for b in "ab")\n            old = {(r["order"], r["arm"][-1]): r for r in prior["pairs"][pair_id]}\n            checkpoint = checkpoints / f"{pair_id}.json"\n            expected = json.loads(checkpoint.read_bytes()) if checkpoint.is_file() else None\n            if expected is not None:\n                assert expected["identity"] == ident and expected["pair_id"] == pair_id\n            if index and expected is not None:\n                reports.append(expected["summary"])\n                print(f"PAIR_REUSED {index + 1}/9 {pair_id}", flush=True)\n                continue\n            two = {}\n            for branch in "ab":\n                record = media["decoded"][branch]\n                assert all(r["earlier_frame_sha256"] == record["decoded_rgb_sha256"][:2] for (_, b), r in old.items() if b == branch)\n                frames = base.decode(paths[pair_id][branch], record)\n                two[branch] = dict(frames=frames[:2], indices=record["source_frames"][:2],\n                                   seconds=record["frame_start_s"][:2], hashes=record["decoded_rgb_sha256"][:2])\n            rows, checks = evaluate(model, annotated, two, old, preflight=index == 0)\n            if index == 0:\n                base.atomic(exports / "preflight.json", dict(status="pass", checks=checks))\n                print("TWINPAST_HISTORY_ONLY_PREFLIGHT_PASS", flush=True)\n            comparison = outcome(rows, annotated)\n            summary = dict(pair_id=pair_id, task=plan["task"], two_history_only=comparison,\n                           prior_history_with_recent_robust=all(r["prediction"] == r[f"gold_{b}"] for (_, b), r in old.items()))\n            dense_rows = []\n            if plan["task"] == "HSR":\n                contact_sheet(exports / "review" / f"{pair_id}_two.jpg", pair_id, "Two earlier frames", two)\n                if not comparison["robust"]:\n                    try:\n                        dense = prepare_dense(paths[pair_id], annotated, media)\n                    except ValueError as exc:\n                        summary["dense_skip"] = str(exc)\n                    else:\n                        dense_rows, _ = evaluate(model, annotated, dense, old)\n                        summary["dense_history_only"] = outcome(dense_rows, annotated)\n                        contact_sheet(exports / "review" / f"{pair_id}_dense.jpg", pair_id, "Six earlier frames", dense)\n            base.atomic(checkpoint, dict(pair_id=pair_id, identity=ident, summary=summary,\n                                        history_two=rows, history_dense=dense_rows))\n            reports.append(summary)\n            print(f"PAIR_DONE {index + 1}/9 {pair_id} two_order_robust={comparison[\'robust\']}", flush=True)\n    dense = [item["dense_history_only"] for item in reports if "dense_history_only" in item]\n    summary = dict(pairs=len(reports), two_history_only_robust=sum(r["two_history_only"]["robust"] for r in reports),\n                   restored_from_original=sum(r["two_history_only"]["robust"] and not r["prior_history_with_recent_robust"]\n                                              for r in reports),\n                   lost_from_original=sum(not r["two_history_only"]["robust"] and r["prior_history_with_recent_robust"]\n                                          for r in reports),\n                   dense_pairs=len(dense), dense_robust=sum(r["robust"] for r in dense), per_pair=reports)\n    base.atomic(exports / "summary.json", summary)\n    base.atomic(work / "completion.json", dict(status="complete", pairs=len(reports), dense_pairs=len(dense)))\n    print("TWINPAST_HISTORY_ONLY_COMPLETE", json.dumps({k: v for k, v in summary.items() if k != "per_pair"}), flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "media-reuse", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(args.work)\n\n\nif __name__ == "__main__":\n    main()\n', 'provenance/backbone_runner.py': '"""Replay the frozen history-only windows on the larger same-family backbone."""\nimport argparse\nimport json\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport history_only_runner as history\n\nbase = history.base\nARTIFACT = "HERMES_twinpast_backbone_evidence.zip"\nMODEL_REVISION = "0d50680527681998e456c7b78950205bedd8a068"\n\n\ndef reproduce_windows(paths, records):\n    windows = {}\n    for branch in "ab":\n        left, right = (next(r for r in records if r["branch"] == branch and r["order"] == order)\n                       for order in base.ORDERS)\n        assert all(left[k] == right[k] for k in ("frame_indices", "frame_seconds", "frame_hashes"))\n        frames = base.decode(paths[branch], dict(source_frames=left["frame_indices"],\n                                                 decoded_rgb_sha256=left["frame_hashes"]))\n        windows[branch] = dict(frames=frames, indices=left["frame_indices"],\n                               seconds=left["frame_seconds"], hashes=left["frame_hashes"])\n    return windows\n\n\ndef run_pair(model, paths, case, smoke=False):\n    windows = reproduce_windows(paths, case["reference_rows"])\n    original = {(r["order"], r["arm"][-1]): r for r in case["original_prompt_rows"]}\n    reference = {}\n    for row in case["reference_rows"]:\n        key = row["order"], row["branch"]\n        assert row["prompt_sha256"] == original[key]["prompt_sha256"]\n        reference[key] = {**row, "label_token_ids": original[key]["label_token_ids"]}\n    rows, checks = history.evaluate(model, case["annotation"], windows, reference, preflight=smoke)\n    prior = history.outcome(case["reference_rows"], case["annotation"])\n    current = history.outcome(rows, case["annotation"])\n    return dict(pair_id=case["annotation"]["pair_id"], cohort=case["cohort"], prior=prior,\n                current=current, preflight=checks, rows=rows)\n\n\ndef summarize(blocks):\n    states = [x for x in blocks if x["cohort"] == "historical_state"]\n    controls = [x for x in blocks if x["cohort"] == "positive_control"]\n    robust_states = sum(x["current"]["robust"] for x in states)\n    robust_controls = sum(x["current"]["robust"] for x in controls)\n    return dict(pairs=len(blocks), scores=sum(len(x["rows"]) for x in blocks),\n                historical_state_robust=robust_states, historical_state_total=len(states),\n                positive_control_robust=robust_controls, positive_control_total=len(controls),\n                development_criterion_met=len(states) == 4 and len(controls) == 3\n                and robust_states >= 2 and robust_controls >= 2,\n                per_pair=[{k: v for k, v in x.items() if k not in {"preflight", "rows"}} for x in blocks])\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work / "provenance"\n    bundle = json.loads((provenance / "bundle.json").read_bytes())\n    frozen = json.loads((provenance / "backbone_cases.json").read_bytes())\n    cases = {c["annotation"]["pair_id"]: c for c in frozen["cases"]}\n    assert len(cases) == 7 and frozen["model_revision"] == MODEL_REVISION == args.model.name\n    selection = [pair for pair in base.cases(bundle) if pair[0]["pair_id"] in cases]\n    assert {p["pair_id"] for p, _, _ in selection} == cases.keys()\n    # Start with the six-frame historical-state case, which exercises the longest input.\n    assert cases[selection[0][0]["pair_id"]]["cohort"] == "historical_state"\n    paths = history.acquire(SimpleNamespace(work=work, previous=args.previous, media_reuse=args.media_reuse),\n                            bundle, selection)\n    print("LOADING_PINNED_7B", args.model.name, flush=True)\n    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.device.type == "cuda" and model.dtype == torch.float16\n    assert {str(p.device) for p in model.parameters()} == {"cuda:0"}, "7B must fit fully on GPU 0"\n    config = dict(model_revision=MODEL_REVISION, gpu=torch.cuda.get_device_name(0),\n                  torch=torch.__version__, cuda=torch.version.cuda, dtype=str(model.dtype),\n                  video_processor=processor.video_processor.to_dict(),\n                  runtime_source_hashes={p.name: base.digest(p) for p in sorted(provenance.iterdir()) if p.is_file()},\n                  source_history_only_archive=frozen["history_only_archive_sha256"])\n    ident = base.sha(json.dumps(config, sort_keys=True).encode())\n    base.atomic(work / "exports" / "run_config.json", config)\n    checkpoints = work / "checkpoints"\n    checkpoints.mkdir(exist_ok=True)\n    blocks = []\n    with torch.inference_mode():\n        for i, (plan, _, _) in enumerate(selection):\n            pair = plan["pair_id"]\n            checkpoint = checkpoints / f"{pair}.json"\n            saved = json.loads(checkpoint.read_bytes()) if checkpoint.is_file() else None\n            if saved is not None:\n                assert saved["identity"] == ident and saved["block"]["pair_id"] == pair\n            if i and saved is not None:\n                blocks.append(saved["block"])\n                print(f"PAIR_REUSED {i + 1}/7 {pair}", flush=True)\n                continue\n            print(f"PAIR_RUNNING {i + 1}/7 {pair}", flush=True)\n            torch.cuda.reset_peak_memory_stats()\n            block = run_pair(model, paths[pair], cases[pair], smoke=i == 0)\n            block["peak_gpu_allocated_bytes"] = torch.cuda.max_memory_allocated()\n            if i == 0:\n                base.atomic(work / "exports" / "preflight.json",\n                            dict(status="pass", pair_id=pair, checks=block["preflight"],\n                                 peak_gpu_allocated_bytes=block["peak_gpu_allocated_bytes"]))\n                print("TWINPAST_BACKBONE_PREFLIGHT_PASS", flush=True)\n            base.atomic(checkpoint, dict(identity=ident, block=block))\n            blocks.append(block)\n            print(f"PAIR_DONE {i + 1}/7 {pair} robust={block[\'current\'][\'robust\']}", flush=True)\n    report = summarize(blocks)\n    assert report["pairs"] == 7 and report["scores"] == 28\n    base.atomic(work / "exports" / "summary.json", report)\n    base.atomic(work / "completion.json", dict(status="complete", pairs=7, scores=28))\n    print("TWINPAST_BACKBONE_COMPLETE", json.dumps({k: v for k, v in report.items() if k != "per_pair"}), flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "media-reuse", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(args.work)\n\n\nif __name__ == "__main__":\n    main()\n', 'provenance/automatic_runner.py': '"""Run a frozen, annotation-blind retrieval and answer-correction screen."""\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\nimport time\n\nimport backbone_runner as backbone\n\nbase = backbone.base\nARTIFACT = "HERMES_twinpast_automatic_evidence.zip"\nMAX_CANDIDATES = 13\nMAX_HISTORY = 6\nMIN_LIFT = 0.5\nMETHODS = ("recent", "latest", "uniform", "relevance", "margin_gate", "history_control_gate")\n\n\ndef spread(count, budget):\n    if count <= budget:\n        return list(range(count))\n    return [i * (count - 1) // (budget - 1) for i in range(budget)]\n\n\ndef sample_prefix(path, query):\n    import numpy as np\n    from decord import VideoReader\n\n    reader = VideoReader(str(path), num_threads=1)\n    times = np.asarray(reader.get_frame_timestamp(list(range(len(reader)))), dtype=np.float64)\n    times -= times[0, 0]\n    grid = list(range(0, math.ceil(query), 2))\n    assert len(grid) >= 8 and grid[-1] < query and grid[-1] <= times[-1, 1] + 1e-6\n    older_grid, recent_grid = grid[:-4], grid[-4:]\n    picks = [older_grid[i] for i in spread(len(older_grid), MAX_CANDIDATES)] + recent_grid\n    indices = np.searchsorted(times[:, 0], picks, side="right") - 1\n    sampled_s = times[indices, 0]\n    assert indices[0] >= 0 and np.all(np.diff(indices) > 0)\n    assert np.all(sampled_s <= picks) and sampled_s[-1] < query\n    frames = reader.get_batch(indices.tolist()).asnumpy()\n    return frames, dict(grid_s=picks, indices=indices.tolist(), timestamps_s=sampled_s.tolist(),\n                        hashes=[base.sha(f.tobytes()) for f in frames])\n\n\ndef encode_frames(model, frames):\n    """Vision encoding is frame-local. Limit the encoder batch to four frames."""\n    bank = []\n    for start in range(0, len(frames), 4):\n        batch = frames[start:start + 4]\n        encoded = base.features(model, batch)\n        bank.extend(encoded.split(encoded.shape[1] // len(batch), dim=1))\n    assert len(bank) == len(frames)\n    return bank\n\n\ndef best(scores):\n    return max(range(len(scores)), key=scores.__getitem__)\n\n\ndef answer_policies(recent, direct, uniform, same_history):\n    initial, alternative = best(recent), best(direct)\n    margin = direct[alternative] - direct[initial]\n    lift = margin - (recent[alternative] - recent[initial])\n    specificity = margin - (uniform[alternative] - uniform[initial])\n    accept = alternative != initial and lift >= MIN_LIFT\n    controlled = accept and not same_history and specificity >= MIN_LIFT\n    return dict(relevance=alternative, margin_gate=alternative if accept else initial,\n                history_control_gate=alternative if controlled else initial,\n                scores=dict(history_margin=margin, lift_from_recent=lift, lift_from_uniform=specificity))\n\n\ndef branch_policy(model, path, query, question, options):\n    """Infer for one branch with no evidence labels, gold, or counterpart data."""\n    import torch\n\n    started = time.perf_counter()\n    frames, sample = sample_prefix(path, query)\n    bank = encode_frames(model, frames)\n    history_count = len(bank) - 4\n    size = min(MAX_HISTORY, history_count)\n    recent = list(range(history_count, len(bank)))\n    uniform = spread(history_count, size)\n    latest = list(range(history_count - size, history_count))\n    embedding = model.language_model.get_input_embeddings()\n    system = embedding(torch.as_tensor(model.init_prompt_ids, device=model.device))\n    calls = []\n    logits_cache = {}\n\n    def score(prompt, window, phase, order):\n        ids, option_ids, perm, digest = prompt\n        key = digest, tuple(window)\n        if key not in logits_cache:\n            visual = torch.cat([bank[i] for i in window], dim=1)\n            tick = time.perf_counter()\n            logits, length = base.score(model, system, ids, option_ids, visual)\n            values = [0.0] * len(perm)\n            for i, value in zip(perm, logits.tolist(), strict=True):\n                values[i] = value\n            logits_cache[key] = values\n            calls.append(dict(phase=phase, order=order, slots=window, forward_s=time.perf_counter()-tick,\n                              visual_tokens=visual.shape[1], lm_tokens=length, prompt_sha256=digest))\n        return logits_cache[key]\n\n    probe = ("Does this earlier excerpt contain visual evidence needed to answer the following "\n             "question about the specified event, state, or entity? " + question)\n    prompts = {o: base.make_prompt(model, probe, ["Yes", "No"], o, True) for o in base.ORDERS}\n    candidates = []\n    relevance = [-math.inf] * history_count\n    for slot in range(history_count - 1):\n        window = [slot, slot + 1]\n        probe_scores = {o: score(prompts[o], window, "retrieval", o) for o in base.ORDERS}\n        average = sum(v[0] - v[1] for v in probe_scores.values()) / len(probe_scores)\n        candidates.append(dict(slots=window, yes_no=probe_scores, relevance=average))\n        for i in window:\n            relevance[i] = max(relevance[i], average)\n    ranked = sorted(sorted(range(history_count), key=lambda i: (-relevance[i], i))[:size])\n    assert len(set(ranked)) == len(uniform) == len(latest) == size\n    selection = {"recent": recent, "latest": latest + recent,\n                 "uniform": uniform + recent, "relevance": ranked + recent}\n    answers = {}\n    for order in base.ORDERS:\n        prompt = base.make_prompt(model, question, options, order, True)\n        vectors = {arm: score(prompt, slots, arm, order) for arm, slots in selection.items()}\n        decision = answer_policies(vectors["recent"], vectors["relevance"], vectors["uniform"], ranked == uniform)\n        choices = {arm: best(values) for arm, values in vectors.items()}\n        choices.update({k: decision[k] for k in ("margin_gate", "history_control_gate")})\n        answers[order] = dict(logits=vectors, predictions={arm: base.LETTERS[i] for arm, i in choices.items()},\n                              margins=decision["scores"], prompt_sha256=prompt[3])\n    return dict(sample=sample, history_budget=size, candidates=candidates, selection=selection, answers=answers,\n                calls=calls, wall_s=time.perf_counter()-started)\n\n\ndef summarize(blocks, targets):\n    report = {}\n    opened = {arm: 0 for arm in ("margin_gate", "history_control_gate")}\n    gate_wins, gate_losses = 0, 0\n    for arm in METHODS:\n        robust, wins, losses = [], 0, 0\n        for block in blocks:\n            pair_id = block["pair_id"]\n            success = True\n            for order in base.ORDERS:\n                for branch in "ab":\n                    values = block["branches"][branch]["answers"][order]["predictions"]\n                    gold = targets[pair_id][f"answer_{branch}"]\n                    correct = values[arm] == gold\n                    reference = values["recent"] == gold\n                    wins += correct and not reference\n                    losses += reference and not correct\n                    success = success and correct\n                    if arm in opened:\n                        opened[arm] += values[arm] != values["recent"]\n                    if arm == "history_control_gate":\n                        direct_correct = values["relevance"] == gold\n                        gate_wins += correct and not direct_correct\n                        gate_losses += direct_correct and not correct\n            if success:\n                robust.append(pair_id)\n        report[arm] = dict(robust_pairs=robust, recoveries_vs_recent=wins, regressions_vs_recent=losses)\n    return dict(pairs=len(blocks), branch_order_answers=len(blocks)*4, methods=report, overrides=opened,\n                controlled_gate_vs_same_frames_direct=dict(recoveries=gate_wins, regressions=gate_losses),\n                total_forward_calls=sum(len(v["calls"]) for b in blocks for v in b["branches"].values()))\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work / "provenance"\n    spec = json.loads((provenance / "automatic_cases.json").read_bytes())\n    assert args.model.name == backbone.MODEL_REVISION == spec["model_revision"]\n    cases = {c["pair_id"]: c for c in spec["cases"]}\n    assert len(cases) == 7\n    paths = backbone.history.acquire(args, spec["media_source"], spec["media_plan"])\n    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.dtype == torch.float16 and {str(p.device) for p in model.parameters()} == {"cuda:0"}\n    assert json.loads(json.dumps(processor.video_processor.to_dict())) == spec["video_processor"]\n    config = dict(model_revision=args.model.name, torch=torch.__version__, cuda=torch.version.cuda,\n                  gpu=torch.cuda.get_device_name(0), min_lift=MIN_LIFT, max_candidates=MAX_CANDIDATES,\n                  max_history=MAX_HISTORY, source_archive_sha256=spec["fixed_recent_archive_sha256"],\n                  hashes={p.name: base.digest(p) for p in sorted(provenance.iterdir()) if p.is_file()})\n    identity = base.sha(json.dumps(config, sort_keys=True).encode())\n    exports = work / "exports"\n    base.atomic(exports / "run_config.json", config)\n    checkpoints = work / "checkpoints"\n    checkpoints.mkdir(exist_ok=True)\n    blocks = []\n    with torch.inference_mode():\n        for i, case in enumerate(spec["cases"]):\n            pair_id = case["pair_id"]\n            checkpoint = checkpoints / f"{pair_id}.json"\n            if checkpoint.is_file():\n                saved = json.loads(checkpoint.read_bytes())\n                assert saved["identity"] == identity and saved["block"]["pair_id"] == pair_id\n                blocks.append(saved["block"])\n                print(f"PAIR_REUSED {i+1}/7 {pair_id}", flush=True)\n                continue\n            torch.cuda.reset_peak_memory_stats()\n            branches = {b: branch_policy(model, paths[pair_id][b], case["query_s"], case["question"], case["options"])\n                        for b in "ab"}\n            left, right = branches["a"], branches["b"]\n            assert left["sample"]["grid_s"] == right["sample"]["grid_s"]\n            assert left["sample"]["hashes"][-4:] == right["sample"]["hashes"][-4:] == case["recent_hashes"]\n            assert left["history_budget"] == right["history_budget"]\n            for order in base.ORDERS:\n                assert left["answers"][order]["predictions"]["recent"] == right["answers"][order]["predictions"]["recent"]\n            block = dict(pair_id=pair_id, branches=branches,\n                         peak_gpu_allocated_bytes=torch.cuda.max_memory_allocated())\n            base.atomic(checkpoint, dict(identity=identity, block=block))\n            blocks.append(block)\n            print(f"PAIR_DONE {i+1}/7 {pair_id}", flush=True)\n    # Ground truth is loaded only after all branch predictions have been written.\n    targets = json.loads((provenance / "evaluation_targets.json").read_bytes())\n    assert targets.keys() == cases.keys()\n    report = summarize(blocks, targets)\n    coverage = []\n    for block in blocks:\n        pair_id = block["pair_id"]\n        for branch, trace in block["branches"].items():\n            intervals = targets[pair_id]["evidence_intervals"][branch.upper()]\n            timestamps = trace["sample"]["timestamps_s"]\n            inside = [i for i, t in enumerate(timestamps[:-4]) if any(lo <= t <= hi for lo, hi in intervals)]\n            coverage.append(dict(pair_id=pair_id, branch=branch, candidate_slots_in_evidence=inside,\n                                 selected_slots_in_evidence={name: sorted(set(inside) & set(trace["selection"][name]))\n                                                             for name in ("latest", "uniform", "relevance")}))\n    base.atomic(exports / "postrun_evidence_overlap.json", coverage)\n    base.atomic(exports / "summary.json", report)\n    base.atomic(work / "completion.json", dict(status="complete", pairs=len(blocks)))\n    print("TWINPAST_AUTOMATIC_COMPLETE", json.dumps(report), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "media-reuse", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(args.work)\n', 'provenance/temporal_runner.py': '"""Evaluate endpoint-preserving retrieval on a new TwinPast development screen."""\nimport argparse\nfrom collections import Counter\nfrom itertools import combinations\nimport json\nimport math\nfrom pathlib import Path\nimport time\n\nimport automatic_runner as auto\n\nbase = auto.base\nbackbone = auto.backbone\nARTIFACT = "HERMES_twinpast_temporal_windows_evidence.zip"\nMETHODS = ("recent", "uniform", "individual", "endpoints")\nWINDOW_SPAN = 3\nWINDOW_CAP = 12\n\n\ndef decode_grid(path, grid):\n    import numpy as np\n    from decord import VideoReader\n\n    reader = VideoReader(str(path), num_threads=1)\n    times = np.asarray(reader.get_frame_timestamp(list(range(len(reader)))), dtype=np.float64)\n    times -= times[0, 0]\n    if grid is None:\n        # Source-screening frames never enter the policy.\n        grid = range(0, math.ceil(times[-1, 1]), 2)\n    grid = np.asarray(list(grid))\n    if grid[-1] > times[-1, 1] + 1e-6:\n        raise ValueError("Clip ends before the frozen grid")\n    indices = np.searchsorted(times[:, 0], grid, side="right") - 1\n    if indices[0] < 0 or not np.all(np.diff(indices) > 0):\n        raise ValueError("Clip does not contain distinct frames on the frozen grid")\n    frames = reader.get_batch(indices.tolist()).asnumpy()\n    return frames, dict(grid_s=grid.tolist(), indices=indices.tolist(), timestamps_s=times[indices, 0].tolist(),\n                        hashes=[base.sha(frame.tobytes()) for frame in frames])\n\n\ndef fingerprints(path):\n    import numpy as np\n    from PIL import Image\n\n    frames, trace = decode_grid(path, None)\n    signatures = []\n    for frame, sha in zip(frames, trace["hashes"], strict=True):\n        image = Image.fromarray(frame)\n        gray = np.asarray(image.convert("L").resize((9, 8)))\n        bits = int.from_bytes(np.packbits((gray[:, 1:] > gray[:, :-1]).reshape(-1)).tobytes(), "big")\n        rgb = np.asarray(image.resize((4, 4)), dtype=int).reshape(-1)\n        if gray.std() >= 8:\n            signatures.append((sha, bits, rgb))\n    return signatures\n\n\ndef shared_frames(a, b):\n    """Flag repeat footage on at least three distinct sampled frame matches."""\n    import numpy as np\n\n    matches = set()\n    for sha_a, hash_a, rgb_a in a:\n        for sha_b, hash_b, rgb_b in b:\n            if sha_a == sha_b or ((hash_a ^ hash_b).bit_count() <= 4 and np.abs(rgb_a - rgb_b).mean() <= 6):\n                matches.add(sha_b)\n                break\n    return len(matches)\n\n\ndef screen_pairs(cases, excluded, paths, max_pairs):\n    signature_bank = {pair: {b: fingerprints(paths[pair][b]) for b in "ab"} for pair in excluded}\n    decisions, accepted = [], []\n    for case in cases:\n        pair = case["pair_id"]\n        if len(accepted) == max_pairs:\n            break\n        grid = list(range(math.ceil(case["query_s"])))\n        try:\n            recent = {b: decode_grid(paths[pair][b], grid[-4:])[1]["hashes"] for b in "ab"}\n        except ValueError as exc:\n            decisions.append(dict(pair_id=pair, disposition="invalid_grid", detail=str(exc)))\n            continue\n        if recent["a"] != recent["b"]:\n            decisions.append(dict(pair_id=pair, disposition="recent_context_mismatch"))\n            continue\n        candidate = {b: fingerprints(paths[pair][b]) for b in "ab"}\n        overlaps = []\n        for reference, signature in signature_bank.items():\n            matches = max(shared_frames(candidate[b], signature[c]) for b in "ab" for c in "ab")\n            if matches >= 3:\n                overlaps.append(dict(reference=reference, sampled_matches=matches))\n        if overlaps:\n            decisions.append(dict(pair_id=pair, disposition="possible_media_overlap", matches=overlaps))\n            continue\n        signature_bank[pair] = candidate\n        accepted.append(case)\n        decisions.append(dict(pair_id=pair, disposition="accepted", recent_hashes=recent["a"]))\n    return accepted, decisions\n\n\ndef select_windows(windows):\n    order = sorted(windows, key=lambda row: row["slots"])\n    compatible = (group for group in combinations(order, 3)\n                  if all(left["slots"][-1] < right["slots"][0] for left, right in zip(group, group[1:])))\n    best = max(compatible, key=lambda group: sum(r["relevance"] for r in group))\n    return [i for r in best for i in r["slots"]]\n\n\ndef branch_policy(model, path, case):\n    import torch\n\n    tick = time.perf_counter()\n    frames, sample = decode_grid(path, range(math.ceil(case["query_s"])))\n    assert sample["timestamps_s"][-1] < case["query_s"]\n    bank = auto.encode_frames(model, frames)\n    num_history = len(frames) - 4\n    recent = list(range(num_history, len(frames)))\n    system = model.language_model.get_input_embeddings()(torch.as_tensor(model.init_prompt_ids, device=model.device))\n    log, cache = [], {}\n\n    def score(prompt, slots, phase, order):\n        ids, option_ids, permutation, prompt_hash = prompt\n        key = (prompt_hash, tuple(slots))\n        if key not in cache:\n            visual = torch.cat([bank[i] for i in slots], dim=1)\n            started = time.perf_counter()\n            values, tokens = base.score(model, system, ids, option_ids, visual)\n            canonical = [0.0] * len(permutation)\n            for i, value in zip(permutation, values.tolist(), strict=True):\n                canonical[i] = value\n            cache[key] = canonical\n            log.append(dict(phase=phase, order=order, slots=slots, forward_s=time.perf_counter()-started,\n                            lm_tokens=tokens, visual_tokens=visual.shape[1]))\n        return cache[key]\n\n    question = case["question"]\n    probe = ("Does this earlier excerpt contain visual evidence needed to answer the following "\n             "question about the specified event, state, or entity? " + question)\n    prompts = {order: base.make_prompt(model, probe, ["Yes", "No"], order, True) for order in base.ORDERS}\n    windows, individual_scores = [], {}\n    for start in auto.spread(num_history - WINDOW_SPAN, WINDOW_CAP):\n        endpoints = [start, start + WINDOW_SPAN]\n        vectors = {o: score(prompts[o], endpoints, "retrieval", o) for o in base.ORDERS}\n        value = sum(v[0] - v[1] for v in vectors.values()) / len(vectors)\n        windows.append(dict(slots=endpoints, relevance=value, yes_no=vectors))\n        for i in endpoints:\n            individual_scores[i] = max(value, individual_scores.get(i, -math.inf))\n    selection = dict(recent=recent, uniform=auto.spread(num_history, 6) + recent,\n                     individual=sorted(sorted(individual_scores, key=lambda i: (-individual_scores[i], i))[:6]) + recent,\n                     endpoints=select_windows(windows) + recent)\n    assert len(selection["recent"]) == 4 and all(len(selection[m]) == 10 for m in METHODS[1:])\n    assert all(slots == sorted(set(slots)) for slots in selection.values())\n    answers = {}\n    for o in base.ORDERS:\n        prompt = base.make_prompt(model, question, case["options"], o, True)\n        vectors = {m: score(prompt, selection[m], m, o) for m in METHODS}\n        answers[o] = dict(logits=vectors, predictions={m: base.LETTERS[auto.best(v)] for m, v in vectors.items()})\n    return dict(sample=sample, windows=windows, selection=selection, answers=answers,\n                calls=log, wall_s=time.perf_counter()-tick)\n\n\ndef evaluate(blocks, targets):\n    methods = {}\n    for method in METHODS:\n        pairs, correct = [], 0\n        for block in blocks:\n            p = block["pair_id"]\n            wins = [block["branches"][b]["answers"][o]["predictions"][method] == targets[p][f"answer_{b}"]\n                    for b in "ab" for o in base.ORDERS]\n            correct += sum(wins)\n            if all(wins):\n                pairs.append(p)\n        methods[method] = dict(robust_pairs=pairs, correct_branch_orders=correct)\n    endpoints = set(methods["endpoints"]["robust_pairs"])\n    matched = {m: dict(recoveries=sorted(endpoints - set(methods[m]["robust_pairs"])),\n                       regressions=sorted(set(methods[m]["robust_pairs"]) - endpoints))\n               for m in ("uniform", "individual")}\n    calls = Counter(call["phase"] for p in blocks for b in p["branches"].values() for call in b["calls"])\n    return dict(pairs=len(blocks), methods=methods, matched_pair_changes=matched,\n                endpoints_more_robust_than_both_dense_baselines=len(methods["endpoints"]["robust_pairs"]) >\n                max(len(methods[m]["robust_pairs"]) for m in ("uniform", "individual")),\n                forward_calls=sum(calls.values()), forward_calls_by_phase=dict(calls))\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work / "provenance"\n    spec = json.loads((provenance / "temporal_cases.json").read_bytes())\n    assert args.model.name == backbone.MODEL_REVISION == spec["model_revision"]\n    paths = backbone.history.acquire(args, spec["media_source"], spec["media_plan"])\n    cases, screening = screen_pairs(spec["cases"], spec["excluded"], paths, spec["expected_max_pairs"])\n    exports = work / "exports"\n    base.atomic(exports / "source_screen.json", dict(decisions=screening, source_status=spec["source_status"]))\n    if not cases:\n        raise RuntimeError("No pairs passed the shared-input and media-overlap screen")\n    print(f"SOURCE_SCREEN_PASS {len(cases)}/{spec[\'expected_max_pairs\']} pairs", flush=True)\n    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.dtype == torch.float16 and {str(p.device) for p in model.parameters()} == {"cuda:0"}\n    assert json.loads(json.dumps(processor.video_processor.to_dict())) == spec["video_processor"]\n    config = dict(model_revision=args.model.name, cuda=torch.version.cuda, torch=torch.__version__,\n                  gpu=torch.cuda.get_device_name(0), accepted=[c["pair_id"] for c in cases],\n                  window_span=WINDOW_SPAN, window_cap=WINDOW_CAP,\n                  hashes={p.name: base.digest(p) for p in sorted(provenance.iterdir()) if p.is_file()})\n    base.atomic(exports / "run_config.json", config)\n    identity = base.sha(json.dumps(config, sort_keys=True).encode())\n    checkpoints = work / "checkpoints"\n    checkpoints.mkdir(exist_ok=True)\n    blocks = []\n    with torch.inference_mode():\n        for i, case in enumerate(cases):\n            pair = case["pair_id"]\n            path = checkpoints / f"{pair}.json"\n            if path.is_file():\n                saved = json.loads(path.read_bytes())\n                assert saved["identity"] == identity and saved["block"]["pair_id"] == pair\n                block = saved["block"]\n                print(f"PAIR_REUSED {i+1}/{len(cases)} {pair}", flush=True)\n            else:\n                torch.cuda.reset_peak_memory_stats()\n                branches = {b: branch_policy(model, paths[pair][b], case) for b in "ab"}\n                assert branches["a"]["sample"]["hashes"][-4:] == branches["b"]["sample"]["hashes"][-4:]\n                assert all(branches["a"]["answers"][o]["predictions"]["recent"] ==\n                           branches["b"]["answers"][o]["predictions"]["recent"] for o in base.ORDERS)\n                block = dict(pair_id=pair, branches=branches, peak_gpu_allocated_bytes=torch.cuda.max_memory_allocated())\n                base.atomic(path, dict(identity=identity, block=block))\n                print(f"PAIR_DONE {i+1}/{len(cases)} {pair}", flush=True)\n            blocks.append(block)\n    targets = json.loads((provenance / "evaluation_targets.json").read_bytes())\n    report = evaluate(blocks, targets)\n    base.atomic(exports / "summary.json", report)\n    base.atomic(work / "completion.json", dict(status="complete", pairs=len(blocks)))\n    print("TWINPAST_TEMPORAL_WINDOWS_COMPLETE", json.dumps(report), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "media-reuse", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(args.work)\n', 'provenance/evaluation_targets.json': '{\n  "twinpast_0255": {\n    "answer_a": "A",\n    "answer_b": "B"\n  },\n  "twinpast_0118": {\n    "answer_a": "A",\n    "answer_b": "B"\n  },\n  "twinpast_0338": {\n    "answer_a": "A",\n    "answer_b": "B"\n  },\n  "twinpast_0164": {\n    "answer_a": "B",\n    "answer_b": "A"\n  },\n  "twinpast_0120": {\n    "answer_a": "A",\n    "answer_b": "B"\n  },\n  "twinpast_0085": {\n    "answer_a": "B",\n    "answer_b": "A"\n  },\n  "twinpast_0206": {\n    "answer_a": "A",\n    "answer_b": "B"\n  },\n  "twinpast_0021": {\n    "answer_a": "B",\n    "answer_b": "A"\n  }\n}\n', 'provenance/readout_runner.py': '"""Compare option-blind observation and direct readouts on archived TwinPast frames."""\nimport argparse\nfrom concurrent.futures import ThreadPoolExecutor\nimport json\nfrom pathlib import Path\nimport time\n\nimport automatic_runner as auto\n\nbase = auto.base\nARTIFACT = "HERMES_twinpast_readout_evidence.zip"\nFRAME_SETS = ("coarse_uniform", "state_change")\nREPLAY_ATOL = 0.05\nNOTE_TOKENS = 24\n\n\ndef note_prompt(question):\n    return ("Describe only the visible state of the object or event named in the question. "\n            "Use the requested moment in the video. If the state is unclear, write \'unclear\'. "\n            "Write one short observation.\\nQuestion: " + question)\n\n\ndef observe(model, system, visual, question):\n    import torch\n\n    tokenizer = model.processor.tokenizer\n    text = model.get_prompt(note_prompt(question))\n    tokens = tokenizer.encode(text, add_special_tokens=False)\n    ids = torch.tensor([tokens], device=model.device)\n    lm = model.language_model\n    prompt = torch.cat((system, visual, lm.get_input_embeddings()(ids)), dim=1)\n    stops = lm.generation_config.eos_token_id\n    stops = set(stops if isinstance(stops, (list, tuple)) else [stops]) | {tokenizer.eos_token_id}\n    generated, consumed = [], prompt.shape[1]\n    start = time.perf_counter()\n    output = lm(inputs_embeds=prompt, use_cache=True, return_dict=True)\n    forwards = 1\n    for step in range(NOTE_TOKENS):\n        last = output.logits[:, -1, :]\n        if not torch.isfinite(last).all():\n            raise ValueError("Non-finite observation logits")\n        token = last.argmax(dim=-1, keepdim=True)\n        if token.item() in stops:\n            break\n        generated.append(token.item())\n        if step + 1 < NOTE_TOKENS:\n            output = lm(input_ids=token, past_key_values=output.past_key_values, use_cache=True, return_dict=True)\n            consumed += 1\n            forwards += 1\n    note = " ".join(tokenizer.decode(generated, skip_special_tokens=True).split()) or "unclear"\n    return dict(text=note, generated_token_ids=generated, forward_calls=forwards, input_tokens=consumed,\n                prompt_sha256=base.sha(text.encode("utf-8")), forward_s=time.perf_counter()-start)\n\n\ndef choose(model, system, visual, question, options, order):\n    prompt, option_ids, permutation, prompt_hash = base.make_prompt(model, question, options, order, True)\n    start = time.perf_counter()\n    scores, tokens = base.score(model, system, prompt, option_ids, visual)\n    values = [0.0] * len(permutation)\n    for i, score in zip(permutation, scores.tolist(), strict=True):\n        values[i] = score\n    return dict(logits=values, prediction=base.LETTERS[auto.best(values)], lm_tokens=tokens,\n                visual_tokens=visual.shape[1], prompt_sha256=prompt_hash, forward_s=time.perf_counter()-start)\n\n\ndef fixed_inputs(model, path, branch):\n    import torch\n\n    coarse, fine = branch["coarse"], branch["rgb"]\n    frames = base.decode(path, dict(source_frames=coarse["indices"], decoded_rgb_sha256=coarse["hashes"]))\n    rgb = base.decode(path, dict(source_frames=fine["indices"], decoded_rgb_sha256=fine["hashes"]))\n    # Preserve the original vision encoder batch boundaries for numerical replay.\n    bank = auto.encode_frames(model, frames)\n    recent = bank[-4:]\n    history = [bank[i] for i in coarse["slots"]]\n    return dict(coarse_uniform=torch.cat(history+recent, dim=1),\n                state_change=torch.cat(auto.encode_frames(model, rgb)+recent, dim=1))\n\n\ndef acquire(args, spec):\n    def clip(case, side):\n        receipt = case["media"][side]\n        relative = receipt["file"]\n        cached = args.previous / "media" / relative\n        if cached.is_file() and cached.stat().st_size == receipt["bytes"] and base.digest(cached) == receipt["sha256"]:\n            print("MEDIA_REUSED", relative, flush=True)\n            return cached\n        target = args.work / "media" / relative\n        base.fetch(spec["media_source"], relative, target, receipt["sha256"], receipt["bytes"])\n        return target\n\n    with ThreadPoolExecutor(max_workers=3) as pool:\n        jobs = {(case["pair_id"], side): pool.submit(clip, case, side)\n                for case in spec["cases"] for side in "ab"}\n        return {key: job.result() for key, job in jobs.items()}\n\n\ndef replay(case, system, model, paths):\n    pair = case["pair_id"]\n    records, visuals, failures = {}, {}, []\n    for side in "ab":\n        archived = case["branches"][side]\n        for frame_set, visual in fixed_inputs(model, paths[pair, side], archived).items():\n            visuals[side, frame_set] = visual.cpu()\n            for order in base.ORDERS:\n                result = choose(model, system, visual, case["question"], case["options"], order)\n                expected = archived["direct"][frame_set][order]\n                delta = max(abs(a-b) for a, b in zip(result["logits"], expected["logits"], strict=True))\n                result["replay_max_abs"] = delta\n                if delta > REPLAY_ATOL or any(result[k] != expected[k] for k in\n                                               ("prediction", "prompt_sha256", "lm_tokens", "visual_tokens")):\n                    failures.append(dict(pair_id=pair, branch=side, frame_set=frame_set, order=order,\n                                         archived=expected, replay=result))\n                records[side, frame_set, order] = result\n    return records, visuals, failures\n\n\ndef compare(case, model, system, packed):\n    direct, visuals, _ = packed\n    rows = []\n    for (side, frame_set), cpu_visual in visuals.items():\n        visual = cpu_visual.to(model.device)\n        note = observe(model, system, visual, case["question"])\n        with_note = case["question"] + "\\nVisual observation from the first pass (may be incomplete): " + note["text"]\n        results = {order: dict(direct=direct[side, frame_set, order],\n                               observation=choose(model, system, visual, with_note, case["options"], order))\n                   for order in base.ORDERS}\n        rows.append(dict(branch=side, frame_set=frame_set, observation=note, orders=results))\n    return dict(pair_id=case["pair_id"], rows=rows)\n\n\ndef summarize(blocks, targets):\n    summary = {}\n    for frame_set in FRAME_SETS:\n        methods = {m: dict(robust_pairs=[], correct_branch_orders=0, forward_calls=0, input_tokens=0, forward_s=0.0)\n                   for m in ("direct", "observation")}\n        max_replay = 0.0\n        for block in blocks:\n            count = dict.fromkeys(methods, 0)\n            pair = block["pair_id"]\n            for branch in (r for r in block["rows"] if r["frame_set"] == frame_set):\n                note = branch["observation"]\n                for field in ("forward_calls", "input_tokens", "forward_s"):\n                    methods["observation"][field] += note[field]\n                for result in branch["orders"].values():\n                    max_replay = max(max_replay, result["direct"]["replay_max_abs"])\n                    for method, row in result.items():\n                        count[method] += row["prediction"] == targets[pair][f"answer_{branch[\'branch\']}"]\n                        methods[method]["forward_calls"] += 1\n                        methods[method]["input_tokens"] += row["lm_tokens"]\n                        methods[method]["forward_s"] += row["forward_s"]\n            for method, correct in count.items():\n                methods[method]["correct_branch_orders"] += correct\n                if correct == 4:\n                    methods[method]["robust_pairs"].append(pair)\n        direct, observed = (set(methods[m]["robust_pairs"]) for m in methods)\n        summary[frame_set] = dict(methods=methods, recoveries=sorted(observed-direct),\n                                  regressions=sorted(direct-observed), max_replay_abs=max_replay)\n    primary = summary["coarse_uniform"]\n    score_count = sum(len(result) for block in blocks for row in block["rows"] for result in row["orders"].values())\n    return dict(pairs=len(blocks), choice_scores=score_count, frame_sets=summary,\n                development_criterion_met=len(blocks) == 8 and bool(primary["recoveries"]) and not primary["regressions"])\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work / "provenance"\n    spec = json.loads((provenance / "readout_cases.json").read_bytes())\n    if args.model.name != spec["model_revision"]:\n        raise ValueError("Model revision differs from the archived run")\n    paths = acquire(args, spec)\n    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.dtype == torch.float16 and {str(p.device) for p in model.parameters()} == {"cuda:0"}\n    assert json.loads(json.dumps(processor.video_processor.to_dict())) == spec["video_processor"]\n    exports = work / "exports"\n    base.atomic(exports / "run_config.json", dict(model_revision=args.model.name, torch=torch.__version__,\n                cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(), replay_atol=REPLAY_ATOL,\n                observation_max_new_tokens=NOTE_TOKENS, source_archive_sha256=spec["source_archive_sha256"]))\n    with torch.inference_mode():\n        system = model.language_model.get_input_embeddings()(torch.as_tensor(model.init_prompt_ids, device=model.device))\n        replays = []\n        print("REPLAY_STARTED", flush=True)\n        for i, case in enumerate(spec["cases"], 1):\n            replays.append(replay(case, system, model, paths))\n            print(f"REPLAY_PAIR {i}/8 {case[\'pair_id\']}", flush=True)\n        failures = [failure for _, _, checks in replays for failure in checks]\n        base.atomic(exports / "replay_gate.json", dict(scores=sum(len(r) for r, _, _ in replays),\n                    max_abs=max(x["replay_max_abs"] for r, _, _ in replays for x in r.values()), failures=failures))\n        if failures:\n            raise RuntimeError("Archived direct replay failed. See exports/replay_gate.json.")\n        print("REPLAY_PASS", flush=True)\n        blocks = []\n        for i, case in enumerate(spec["cases"], 1):\n            block = compare(case, model, system, replays[i-1])\n            base.atomic(work / "checkpoints" / f"{case[\'pair_id\']}.json", block)\n            blocks.append(block)\n            print(f"READOUT_PAIR {i}/8 {case[\'pair_id\']}", flush=True)\n    targets = json.loads((provenance / "evaluation_targets.json").read_bytes())\n    assert set(targets) == {b["pair_id"] for b in blocks}\n    report = summarize(blocks, targets)\n    base.atomic(exports / "summary.json", report)\n    base.atomic(work / "completion.json", dict(status="complete", pairs=len(blocks)))\n    print("TWINPAST_READOUT_COMPLETE", json.dumps(report), flush=True)\n\n\ndef main():\n    from filelock import FileLock\n\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    work = args.work\n    with FileLock(str(work / "run.lock"), timeout=0):\n        for name in ("exports", "checkpoints"):\n            (work / name).mkdir(exist_ok=True)\n        # A rerun must not package scores from an earlier incomplete attempt.\n        for case in json.loads((work / "provenance/readout_cases.json").read_bytes())["cases"]:\n            (work / "checkpoints" / f"{case[\'pair_id\']}.json").unlink(missing_ok=True)\n        for name in ("summary.json", "replay_gate.json", "run_config.json"):\n            (work / "exports" / name).unlink(missing_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(work)\n\n\nif __name__ == "__main__":\n    main()\n', 'provenance/boundary_cases.json': '{\n  "cases": [\n    {\n      "pair_id": "twinpast_0255",\n      "query_s": 35.5,\n      "question": "In the earlier clip, what was the order of the letters at the beginning?",\n      "options": [\n        "sel",\n        "home"\n      ],\n      "branches": {\n        "a": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480,\n              504,\n              528,\n              552,\n              576,\n              600,\n              624,\n              648,\n              672,\n              696,\n              720,\n              744,\n              768,\n              792,\n              816,\n              840\n            ],\n            "hashes": [\n              "86c5724ce4beed8fffd3eb99d8bdedca6973253765f26e7446679a5f459fcc52",\n              "bfe648ad16a20d0658555d00668dae575baa8e1a10e33c71130c7724e0000448",\n              "dd701c870681ad87f1402c00750c5ce95f518846cb0ac3fbd040b4fa1a041365",\n              "e536f6d84da4a0f78357efdd7a80da6490c1f499a92dd2201a90d70a47bdfb2d",\n              "ff4c2031803dad98a2627a0cbb05a64f551bec8be5c3ec1554ca5f6428fe6aac",\n              "dc6d5db9b3ad729b5e1b5c77040f92e1d4b99185254c3b716a4ba2cc82d5783c",\n              "c2c621e305c0a49ea79fe41fd7158c0f3ee26fc6fa40c80ff728c17d2872ed48",\n              "7de3278f95c6d5b4aa5f5a68d2b78023d2395b3d404ffd7aeb9de69f32af2b8a",\n              "691818fff3e959eca007c822ab481ae1b99f4c47f3a4f8822ec7f734ae26239e",\n              "192e7af0176637586889bf856aa6a1399606bf692c367965c234532e63f3713d",\n              "89fbbdd7bbfd3a1a4fbbe55b46df878429b5cb843acaad2f47efc702ae390263",\n              "cffe1453875b87fa9aec466381bf6e517c4e874fc28be06e8bec65251d676eb3",\n              "0786f85234223c62de783312eb8501ef87edec38129552b560935988e07ffc82",\n              "794ad16f831378da74e4cbae2ec294daff138d86cffb772421828ffcaa4d712a",\n              "4401aa21249bc8f4ace7a7501f866a0c50d0d6e328e38273a80c4120bca2425c",\n              "79a347b6f7e1010eac77b687b649c8f1d3cbfdb38f90ecc66ddc83de290be6e2",\n              "4d6e4a948dc74c1dcdc29d52c54f80bd9cc61410c2a91e79466a7376d1113b36",\n              "5fe036e3dadf476aed87faff3d7694031c574bab72ee9c3b9ae98afd959acb8f",\n              "bf41f650704a9dd1c9c2d08a39bbdfacd8d4630225937471a04fa3c277c3a0ef",\n              "95f00f0a54e3e02013180d0d92cd1a64f42265fc9e0ae1f8353f2dacc4b62642",\n              "125aa0ce1ad8af4c67674c4e54910fc2c5913346ecb3ab189fbab08a57db1f9e",\n              "b599875d4b2c6baab32786f5376626346dff79372d4377e0f67cfdab49349227",\n              "3ecc75e5316c4821f38521ca3158ae39053c819c2c28f550c85a249e97b28c43",\n              "c722723bbfe74200ffb168bc461a804c345debdc513997f128ea98ab888eafda",\n              "4c09c58806f433e419002e32c65465e567e557148b9bb232bbb9193c6a2f5ce8",\n              "1d7511411cc2a1611f11c06e7d341293e1f0cb578dd1d354c2af13b7eab55f59",\n              "ca9d0c062962e9578c1ccee1aceedaaca296c3e304a417cebc918f39b873ba86",\n              "57d1bea24b5e9b6d51ee008515bd5bb604cef10e13b9ec8058027cb434bb8683",\n              "f4fd4c96e7b185ced21993159ab41651cf8327d5c9a90111b7ca589f0c93f8af",\n              "e95d40ceb9af3969c7a0578356b7efe814fd487e8e169474693c36c566170018",\n              "1f3b911afab3ea9ef619782dd8da1d543d88c0a7303d02a5080a8be7bca6e6f5",\n              "b5ea73b3f4973f8aaccc9a644736ba967a1e351bddaec413e14c4659893a32b2",\n              "f5cd133ff1425b3be2c81d98d167b20861294673bfbdba51da0590a76230b3a9",\n              "f6802a6ac9a28f6085c108fb5f4d9683c48735ff5dee4b7cd352f65f44afa03a",\n              "7e5cb16c5de7709ce2105728c8aa52a675f4b7d3528bd363d8cb6cf2dc2cdd07",\n              "7ab4a4d4e8fa6f41ae2d4b41456560934d858968f883c0b0b62d25d5e79dcd41"\n            ],\n            "slots": [\n              0,\n              6,\n              12,\n              18,\n              24,\n              31\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  21.09375,\n                  16.21875\n                ],\n                "prediction": "A",\n                "lm_tokens": 2026,\n                "visual_tokens": 1960,\n                "prompt_sha256": "9aa960924967309f3e89c8b54bb1149aad17c8f2b837486aec860db6dce0b037"\n              },\n              "reversed": {\n                "logits": [\n                  18.875,\n                  13.4609375\n                ],\n                "prediction": "A",\n                "lm_tokens": 2026,\n                "visual_tokens": 1960,\n                "prompt_sha256": "1195eaf943344939c9233b1a0f31320daedb7d9104f372f858c96c53d35f14c0"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  20.671875,\n                  15.34375\n                ],\n                "prediction": "A",\n                "prompt_sha256": "9aa960924967309f3e89c8b54bb1149aad17c8f2b837486aec860db6dce0b037",\n                "lm_tokens": 2026,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  18.890625,\n                  13.2109375\n                ],\n                "prediction": "A",\n                "prompt_sha256": "1195eaf943344939c9233b1a0f31320daedb7d9104f372f858c96c53d35f14c0",\n                "lm_tokens": 2026,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              168,\n              174,\n              180,\n              186,\n              192,\n              198,\n              204,\n              210,\n              216,\n              222,\n              228,\n              234,\n              240,\n              288,\n              294,\n              300,\n              306,\n              312,\n              318,\n              324,\n              330,\n              336,\n              342,\n              348,\n              354,\n              360,\n              528,\n              534,\n              540,\n              546,\n              552,\n              558,\n              564,\n              570,\n              576,\n              582,\n              588,\n              594,\n              600\n            ],\n            "hashes": [\n              "7de3278f95c6d5b4aa5f5a68d2b78023d2395b3d404ffd7aeb9de69f32af2b8a",\n              "7ed8b8ab85e018f6853c11a12a643008479290226acbfaee62bed9be4b525d6c",\n              "a601fb4f4d0cc3ad289b5455bbe00b946ac93cfaf1cdf7c388217194fbaee5b2",\n              "87ca93fb5ced68a90d96cba7fc6b30412fd16aa289f2461ad17d36a89797adea",\n              "691818fff3e959eca007c822ab481ae1b99f4c47f3a4f8822ec7f734ae26239e",\n              "11dfdc46e7f021c4e274c9f322e68dce777988e1160846cec1bf38c4f3e9d281",\n              "bef618fc1f20cf3d270ffa9edd4669890f89eccec6035b80b7f534c03e7f9f10",\n              "7dfe93350e9e7cc219d39638b5d18d83965ee2fe728dcbe4eb6d791007cda145",\n              "192e7af0176637586889bf856aa6a1399606bf692c367965c234532e63f3713d",\n              "28b5e2663bacd25d6419a0163cf4e34d7090cc893a7b6c826d1fd555a314cb10",\n              "e6927715270b3f822200e69e2c08f66b3399f8f04e49ceab09b01457e276d865",\n              "9a46e07c8385e11d0d56fc105bf4017ca004c197a9b6ac04859752e7aa2a3ded",\n              "89fbbdd7bbfd3a1a4fbbe55b46df878429b5cb843acaad2f47efc702ae390263",\n              "0786f85234223c62de783312eb8501ef87edec38129552b560935988e07ffc82",\n              "527bca0f83375d9cb1da942ce657bc25e6b8e7f3668909249dd6b6fce3dfa1da",\n              "da6954c349bb30b0a3bc01aad04e9471f795319338753781045969a658c7d344",\n              "1540ec264952818295eeaf74e5974675ad31d30d6a49dcefe5f892e7c17d249b",\n              "794ad16f831378da74e4cbae2ec294daff138d86cffb772421828ffcaa4d712a",\n              "98a00dfd312389a2e2f37c2464a1e80409d799dacb250b62e390c7b69120d901",\n              "579cc1cdf8e8736398862329a0a072c920d846691a5d6eedc07f71efcb196620",\n              "543ac0b78f16f9c0a3e8a7f463b24634e6b134d1d9027ac4d0320b0b84c5631a",\n              "4401aa21249bc8f4ace7a7501f866a0c50d0d6e328e38273a80c4120bca2425c",\n              "720f2c411d9c6e381618464f24261b8ce074fdc2d01a769bbd93e0add2556286",\n              "89cd35e1b3925ab10902c761593f5dff63e1fdf9c1a2ad26c63df489ccb93869",\n              "b9a1e841d58d688c5566cb1b6b9911a7a6ff34a98703cf9e6a2fbb064cfa465a",\n              "79a347b6f7e1010eac77b687b649c8f1d3cbfdb38f90ecc66ddc83de290be6e2",\n              "3ecc75e5316c4821f38521ca3158ae39053c819c2c28f550c85a249e97b28c43",\n              "acbc3ee68de9d20a0c64d68358d5a414520f99cbf9d1bfc220d78f180548e2ae",\n              "000638949e7627d82c8df1fbc44353b1bf57aa29b4a9388ae85d60c9d853c06c",\n              "2fb0a1f38a7a488b541a42adadfcb0898a0fbb437310a3ca85d2f3c2191727d0",\n              "c722723bbfe74200ffb168bc461a804c345debdc513997f128ea98ab888eafda",\n              "913ec64220260cba4ca326e7995b3e5669bde5aa3fe95b06663c292d6cf42be0",\n              "5d74934adae9c4dd5479fd070b030d7e113d0f6f1e47eaee97e96fe2d81378e8",\n              "810d21ecb60c2fd9f454baef9806dff439059755b695750d6a81ca66384e5a13",\n              "4c09c58806f433e419002e32c65465e567e557148b9bb232bbb9193c6a2f5ce8",\n              "7e35ef9f85b77e259777b6e6960711e2061a144bcae6722a80588e485244c82b",\n              "69fb424bfa01b9ee379cd252332937bb7dc889025d517488dd88eb57fcd80386",\n              "af62cee5e9bbb6fdcc174c5fb6e458d44750dfc3d99a53baf08ea47a774e6dc3",\n              "1d7511411cc2a1611f11c06e7d341293e1f0cb578dd1d354c2af13b7eab55f59"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": 0.0546875,\n              "yes_no": {\n                "canonical": [\n                  20.171875,\n                  20.25\n                ],\n                "reversed": [\n                  20.046875,\n                  19.859375\n                ]\n              }\n            },\n            {\n              "slots": [\n                2,\n                5\n              ],\n              "relevance": 0.0078125,\n              "yes_no": {\n                "canonical": [\n                  20.3125,\n                  20.53125\n                ],\n                "reversed": [\n                  20.375,\n                  20.140625\n                ]\n              }\n            },\n            {\n              "slots": [\n                5,\n                8\n              ],\n              "relevance": 0.171875,\n              "yes_no": {\n                "canonical": [\n                  20.46875,\n                  20.421875\n                ],\n                "reversed": [\n                  20.34375,\n                  20.046875\n                ]\n              }\n            },\n            {\n              "slots": [\n                7,\n                10\n              ],\n              "relevance": 0.25,\n              "yes_no": {\n                "canonical": [\n                  21.015625,\n                  20.875\n                ],\n                "reversed": [\n                  20.859375,\n                  20.5\n                ]\n              }\n            },\n            {\n              "slots": [\n                10,\n                13\n              ],\n              "relevance": 0.265625,\n              "yes_no": {\n                "canonical": [\n                  21.0,\n                  20.796875\n                ],\n                "reversed": [\n                  20.796875,\n                  20.46875\n                ]\n              }\n            },\n            {\n              "slots": [\n                12,\n                15\n              ],\n              "relevance": 0.3046875,\n              "yes_no": {\n                "canonical": [\n                  20.828125,\n                  20.59375\n                ],\n                "reversed": [\n                  20.640625,\n                  20.265625\n                ]\n              }\n            },\n            {\n              "slots": [\n                15,\n                18\n              ],\n              "relevance": 0.2265625,\n              "yes_no": {\n                "canonical": [\n                  20.921875,\n                  20.78125\n                ],\n                "reversed": [\n                  20.796875,\n                  20.484375\n                ]\n              }\n            },\n            {\n              "slots": [\n                17,\n                20\n              ],\n              "relevance": 0.1953125,\n              "yes_no": {\n                "canonical": [\n                  20.828125,\n                  20.75\n                ],\n                "reversed": [\n                  20.6875,\n                  20.375\n                ]\n              }\n            },\n            {\n              "slots": [\n                20,\n                23\n              ],\n              "relevance": 0.234375,\n              "yes_no": {\n                "canonical": [\n                  20.84375,\n                  20.71875\n                ],\n                "reversed": [\n                  20.71875,\n                  20.375\n                ]\n              }\n            },\n            {\n              "slots": [\n                22,\n                25\n              ],\n              "relevance": 0.2734375,\n              "yes_no": {\n                "canonical": [\n                  20.4375,\n                  20.234375\n                ],\n                "reversed": [\n                  20.390625,\n                  20.046875\n                ]\n              }\n            },\n            {\n              "slots": [\n                25,\n                28\n              ],\n              "relevance": -0.125,\n              "yes_no": {\n                "canonical": [\n                  20.875,\n                  21.4375\n                ],\n                "reversed": [\n                  21.296875,\n                  20.984375\n                ]\n              }\n            },\n            {\n              "slots": [\n                28,\n                31\n              ],\n              "relevance": -0.21875,\n              "yes_no": {\n                "canonical": [\n                  20.796875,\n                  21.515625\n                ],\n                "reversed": [\n                  21.296875,\n                  21.015625\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                7,\n                10\n              ],\n              [\n                12,\n                15\n              ],\n              [\n                22,\n                25\n              ]\n            ],\n            "constrained_windows": [\n              [\n                7,\n                10\n              ],\n              [\n                12,\n                15\n              ],\n              [\n                22,\n                25\n              ]\n            ],\n            "indices": [\n              186,\n              222,\n              306,\n              342,\n              546,\n              582\n            ],\n            "scanned_indices": 125\n          }\n        },\n        "b": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480,\n              504,\n              528,\n              552,\n              576,\n              600,\n              624,\n              648,\n              672,\n              696,\n              720,\n              744,\n              768,\n              792,\n              816,\n              840\n            ],\n            "hashes": [\n              "db6103179f7ae6f76afbc040633cfbc71952edba20b3fecb7fe9135a5fe0a4e0",\n              "fb409dd26b5dd520b6daed6d5264d6e5ebe6caba80ef622b9fecafb5c5adc3b8",\n              "d93b1b4922038fe0502ff2cab474bab20a9ce7df966de15258e436455143770f",\n              "953dbaaaec76992e88db1cd0970e84d78990d1250df3bb0f9f8942502bc6ce8f",\n              "41d3abe1e1934f75f6dac30536af452033494b0ec455507f67a8a56e8afa986e",\n              "033822ab402370671b02a65e2d676019a15a14f531ceabe1ba219a8e808076cb",\n              "3fa80ac9b1969a1521806eefd054f0d0a898e31434da01a91ce76f6ef613ccef",\n              "7e5e717b1b935dcb2f37b743d89ebd0bd36983d8f70ca7143223961642d1fe9d",\n              "3ca07c892c806700bdee4cd6584b3fb8fa7c9d69b8f5ef34262d50f4fae12a54",\n              "980a618a06d4b6ba636a4baa1035e2223fb71b9c228b111f56bc4a84c9bfca52",\n              "8c42ac6c165b694abd273b41d1c2b5f4bc1ef640e78ba6eb5a96ec2d788f8dcd",\n              "0acc0358db5b7db8ec8753dc0e48e1c8e7a92b7835924510f329de3ffb943586",\n              "f4d3e25a9c36208ba3773af434d22465585e64c92389865e5e45683b4e586c6f",\n              "d5f1490677da6dc0c28bc0ce50564223ab404c5d8892f192a6b616e33517c1af",\n              "2487bd29dde20a48e0292612d8e38e29ac649fce70b8ac0596e7be827d767268",\n              "c10ead19e4c2aebdcb403d4f3cb75075c90b89b70c3d05427f9329b683be94ff",\n              "1fa7edd1a53a674aa2082f49534a88ebcb592dd6bd12da0602b40905c090cb4b",\n              "087e10eeadd2c3390ebed6ba81690fe7684614867c62af3dbff1098fef218aec",\n              "56febc41d7ff6c45fe5ddb6e66ff8951b03926a14a814dcbaabedb720c1da877",\n              "7a2e751285c0b84295277a099bfd4268651b6842ca9feb799ddf51cbb88d4f90",\n              "82bee4694f1d25ed42cc779a07eba80276c4adda92b92631c22db34d462b6094",\n              "89dd963a2d9b24b94ce3486c22d5631aeff735c591f3dce6715eb9c808ea350f",\n              "3c71c21de8db88714a3dbb4ba639d296f97189f7d114c4defecb26379e173e3a",\n              "1506c66b12ce00cc9ac52abc94ab212992e58fded24eed10cc33ad1cfb7b8ddb",\n              "9a809b98fad31b309309509a8d3b716c33976f29eaa14f9d0bdb5eb2154db90e",\n              "1d7511411cc2a1611f11c06e7d341293e1f0cb578dd1d354c2af13b7eab55f59",\n              "ca9d0c062962e9578c1ccee1aceedaaca296c3e304a417cebc918f39b873ba86",\n              "57d1bea24b5e9b6d51ee008515bd5bb604cef10e13b9ec8058027cb434bb8683",\n              "f4fd4c96e7b185ced21993159ab41651cf8327d5c9a90111b7ca589f0c93f8af",\n              "e95d40ceb9af3969c7a0578356b7efe814fd487e8e169474693c36c566170018",\n              "1f3b911afab3ea9ef619782dd8da1d543d88c0a7303d02a5080a8be7bca6e6f5",\n              "b5ea73b3f4973f8aaccc9a644736ba967a1e351bddaec413e14c4659893a32b2",\n              "f5cd133ff1425b3be2c81d98d167b20861294673bfbdba51da0590a76230b3a9",\n              "f6802a6ac9a28f6085c108fb5f4d9683c48735ff5dee4b7cd352f65f44afa03a",\n              "7e5cb16c5de7709ce2105728c8aa52a675f4b7d3528bd363d8cb6cf2dc2cdd07",\n              "7ab4a4d4e8fa6f41ae2d4b41456560934d858968f883c0b0b62d25d5e79dcd41"\n            ],\n            "slots": [\n              0,\n              6,\n              12,\n              18,\n              24,\n              31\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  17.96875,\n                  19.546875\n                ],\n                "prediction": "B",\n                "lm_tokens": 2026,\n                "visual_tokens": 1960,\n                "prompt_sha256": "9aa960924967309f3e89c8b54bb1149aad17c8f2b837486aec860db6dce0b037"\n              },\n              "reversed": {\n                "logits": [\n                  20.09375,\n                  20.453125\n                ],\n                "prediction": "B",\n                "lm_tokens": 2026,\n                "visual_tokens": 1960,\n                "prompt_sha256": "1195eaf943344939c9233b1a0f31320daedb7d9104f372f858c96c53d35f14c0"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  19.53125,\n                  19.640625\n                ],\n                "prediction": "B",\n                "prompt_sha256": "9aa960924967309f3e89c8b54bb1149aad17c8f2b837486aec860db6dce0b037",\n                "lm_tokens": 2026,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  20.21875,\n                  18.875\n                ],\n                "prediction": "A",\n                "prompt_sha256": "1195eaf943344939c9233b1a0f31320daedb7d9104f372f858c96c53d35f14c0",\n                "lm_tokens": 2026,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              120,\n              126,\n              132,\n              138,\n              144,\n              150,\n              156,\n              162,\n              168,\n              174,\n              180,\n              186,\n              192,\n              288,\n              294,\n              300,\n              306,\n              312,\n              318,\n              324,\n              330,\n              336,\n              342,\n              348,\n              354,\n              360,\n              480,\n              486,\n              492,\n              498,\n              504,\n              510,\n              516,\n              522,\n              528,\n              534,\n              540,\n              546,\n              552\n            ],\n            "hashes": [\n              "033822ab402370671b02a65e2d676019a15a14f531ceabe1ba219a8e808076cb",\n              "d11c87549b559ddcbcd1a72465f036b0ee0914c65a2f5a00ec94c64df3216aca",\n              "e3e91097c0dd3bf3db5c29359691b861b897c0b3b4d2757e9abd0afd1e39ae4d",\n              "dcfc184cc0e773e466cd06213ae2e53420cd54df23ea39f019ba4b6c85412385",\n              "3fa80ac9b1969a1521806eefd054f0d0a898e31434da01a91ce76f6ef613ccef",\n              "a3fe760526e153cb5393637e298ec0b5854447fe78c790af2298ac4a49527591",\n              "d937e27a6cacaf842fff13f7ddc4810615ff90a10ccb48cb7f23a87c82e619a1",\n              "fbcad8aa9f477c00ac2dab07d48e85024f2a864423ad9e1abc0b82b8733d32cf",\n              "7e5e717b1b935dcb2f37b743d89ebd0bd36983d8f70ca7143223961642d1fe9d",\n              "af5d672c05e859689ccc90479a5aed9df9e8bd0c6502d60d0cc63254f0215005",\n              "ae8b437440adf43632c7673a576750df9be87df325cdec51415979ecf9b18ebb",\n              "083189246bd36a5c975bd1eb53cfb062255d131abcdf924246f1e7278f27182c",\n              "3ca07c892c806700bdee4cd6584b3fb8fa7c9d69b8f5ef34262d50f4fae12a54",\n              "f4d3e25a9c36208ba3773af434d22465585e64c92389865e5e45683b4e586c6f",\n              "db6acf91740104199909263c0d112f8a4f36635dd6c4ab5279dae51355bebd8c",\n              "0a93d3291d602bb6eb898db17981b0c4b6ea8a8e118bd66182945f248ab4e866",\n              "8c6fc0803b29c37c1a4906efe0579b6676e94cdb2661da496093a22420d3c3b5",\n              "d5f1490677da6dc0c28bc0ce50564223ab404c5d8892f192a6b616e33517c1af",\n              "10f91cb1d1172024704b45200e5c80690fd2969223395710cf4742b55fb1e1be",\n              "2648ddf03ab1fe94c1b5fdebea6c692e29be47f874f9e5d2d1b70604b9169c22",\n              "5dad0ec797aabdf30efc96b40ad21c503122f50912074f3ff58dd2b7a22f53db",\n              "2487bd29dde20a48e0292612d8e38e29ac649fce70b8ac0596e7be827d767268",\n              "b54587cdc7b6c1f4351a0f84bee9b9e7995246ad1472f270b9bd3321aad30706",\n              "2083216ba031182f2fa1d53e21e1f18b700d963df09ea494c10efd5b2a78b634",\n              "e67e3432bf5c6f4362d950704291327ea2b14c208cb012d79a6f990026cb55ee",\n              "c10ead19e4c2aebdcb403d4f3cb75075c90b89b70c3d05427f9329b683be94ff",\n              "82bee4694f1d25ed42cc779a07eba80276c4adda92b92631c22db34d462b6094",\n              "86c990d21960326e93f02edfb9662fe10dcfb0af0e6bf770ee97c474d6e2d56b",\n              "a32b99804ec72cca3711ec49456f3e14f75de4c018fa52b0c7b5994e679a3ead",\n              "63268da59797b917bcdde340211b6b80173f446bd47f644186ff57734fa4103b",\n              "89dd963a2d9b24b94ce3486c22d5631aeff735c591f3dce6715eb9c808ea350f",\n              "4d8b2ccb3c33ffd5a92962024567c275c559641aa79c7de3bc3606c936c5032e",\n              "5ed93f651ba46021afd422ab90baf5d54e363a53425db91de23b8e51d02681b9",\n              "fdf7c0b3fd453740f3e3a99b504fc0b10edaaf689d58efc9592a653373ced807",\n              "3c71c21de8db88714a3dbb4ba639d296f97189f7d114c4defecb26379e173e3a",\n              "015a86bd3a6f7d510e3737c4c7e0620bf64c01b564c166643d5f44aca28bb6f3",\n              "20af20386072d205870349ce24500aa530653ccecf3525f71027c9a03e88e635",\n              "85c5d302440c72f2f63eb09c91f7e7d33551d550cd163ec721413788ac9f90c1",\n              "1506c66b12ce00cc9ac52abc94ab212992e58fded24eed10cc33ad1cfb7b8ddb"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": 0.0546875,\n              "yes_no": {\n                "canonical": [\n                  20.578125,\n                  20.65625\n                ],\n                "reversed": [\n                  20.609375,\n                  20.421875\n                ]\n              }\n            },\n            {\n              "slots": [\n                2,\n                5\n              ],\n              "relevance": 0.109375,\n              "yes_no": {\n                "canonical": [\n                  20.71875,\n                  20.484375\n                ],\n                "reversed": [\n                  20.546875,\n                  20.5625\n                ]\n              }\n            },\n            {\n              "slots": [\n                5,\n                8\n              ],\n              "relevance": 0.140625,\n              "yes_no": {\n                "canonical": [\n                  20.671875,\n                  20.46875\n                ],\n                "reversed": [\n                  20.578125,\n                  20.5\n                ]\n              }\n            },\n            {\n              "slots": [\n                7,\n                10\n              ],\n              "relevance": 0.1171875,\n              "yes_no": {\n                "canonical": [\n                  20.9375,\n                  20.78125\n                ],\n                "reversed": [\n                  20.796875,\n                  20.71875\n                ]\n              }\n            },\n            {\n              "slots": [\n                10,\n                13\n              ],\n              "relevance": 0.0390625,\n              "yes_no": {\n                "canonical": [\n                  20.8125,\n                  21.0625\n                ],\n                "reversed": [\n                  21.0625,\n                  20.734375\n                ]\n              }\n            },\n            {\n              "slots": [\n                12,\n                15\n              ],\n              "relevance": 0.234375,\n              "yes_no": {\n                "canonical": [\n                  20.90625,\n                  20.640625\n                ],\n                "reversed": [\n                  20.71875,\n                  20.515625\n                ]\n              }\n            },\n            {\n              "slots": [\n                15,\n                18\n              ],\n              "relevance": 0.140625,\n              "yes_no": {\n                "canonical": [\n                  20.859375,\n                  20.90625\n                ],\n                "reversed": [\n                  21.09375,\n                  20.765625\n                ]\n              }\n            },\n            {\n              "slots": [\n                17,\n                20\n              ],\n              "relevance": 0.015625,\n              "yes_no": {\n                "canonical": [\n                  20.828125,\n                  21.078125\n                ],\n                "reversed": [\n                  21.109375,\n                  20.828125\n                ]\n              }\n            },\n            {\n              "slots": [\n                20,\n                23\n              ],\n              "relevance": 0.1015625,\n              "yes_no": {\n                "canonical": [\n                  21.078125,\n                  21.21875\n                ],\n                "reversed": [\n                  21.34375,\n                  21.0\n                ]\n              }\n            },\n            {\n              "slots": [\n                22,\n                25\n              ],\n              "relevance": -0.203125,\n              "yes_no": {\n                "canonical": [\n                  20.921875,\n                  21.578125\n                ],\n                "reversed": [\n                  21.375,\n                  21.125\n                ]\n              }\n            },\n            {\n              "slots": [\n                25,\n                28\n              ],\n              "relevance": -0.125,\n              "yes_no": {\n                "canonical": [\n                  20.875,\n                  21.4375\n                ],\n                "reversed": [\n                  21.296875,\n                  20.984375\n                ]\n              }\n            },\n            {\n              "slots": [\n                28,\n                31\n              ],\n              "relevance": -0.21875,\n              "yes_no": {\n                "canonical": [\n                  20.796875,\n                  21.515625\n                ],\n                "reversed": [\n                  21.296875,\n                  21.015625\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                5,\n                8\n              ],\n              [\n                12,\n                15\n              ],\n              [\n                20,\n                23\n              ]\n            ],\n            "constrained_windows": [\n              [\n                5,\n                8\n              ],\n              [\n                12,\n                15\n              ],\n              [\n                22,\n                25\n              ]\n            ],\n            "indices": [\n              138,\n              174,\n              306,\n              342,\n              546,\n              582\n            ],\n            "scanned_indices": 125\n          }\n        }\n      },\n      "media": {\n        "a": {\n          "file": "videos/twinpast_0255/A.mp4",\n          "bytes": 16624513,\n          "sha256": "a5abe9605f5b8748ef6532db89ab6cbb55612f7db562d9609168e3cf135d64eb"\n        },\n        "b": {\n          "file": "videos/twinpast_0255/B.mp4",\n          "bytes": 16846600,\n          "sha256": "050cd5028cef8274781cddbaa38ed58b3de975613daf570795074febeb58c6da"\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0118",\n      "query_s": 45.625,\n      "question": "At the end of the guitar-case scene, before the player in a green shirt appears, was the case cover lowered over the guitar body or lifted away from it?",\n      "options": [\n        "Lowered over the guitar body.",\n        "Lifted away from the guitar body."\n      ],\n      "branches": {\n        "a": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480,\n              504,\n              528,\n              552,\n              576,\n              600,\n              624,\n              648,\n              672,\n              696,\n              720,\n              744,\n              768,\n              792,\n              816,\n              840,\n              864,\n              888,\n              912,\n              936,\n              960,\n              984,\n              1008,\n              1032,\n              1056,\n              1080\n            ],\n            "hashes": [\n              "6217617ade9e407f40c6e660d18d6aacb8df22b96b4a83baab3aa89e689f6b87",\n              "0a0363b2298c2e51502642ee8822599c5928705e6108206e784edcf3d5343a5b",\n              "f6457a1647d7429ac991f8826f56d3b5476f18786fddec6e22ed3b7ced0ba837",\n              "b8f57ec7fbc96ec7d5a4f3038ce329d705b625f08ddf1507d797505637581e4a",\n              "a6627e78468231771b29a57bf8092723c642b2143190c7a62e09082f2135eba9",\n              "d2fe72938d0b8fff5488aeb3d0f1abcbbcb149b019cc935d349e956b5eacb1ab",\n              "94a70e604372242c5cd285ddcedfd8cd5c787bcbd026cbffa82ccce4499dd6b5",\n              "f7e3420afc80f74abd82bee8c9f7dd242b36079db6ab0e956e3175c8157dfbc9",\n              "26172b51ead5d075ebf86c96c503145170ec8ec493b401b9c7267a08a4acaf28",\n              "d340b77831787dd80c8402ac14ae8e0d60b5f03d1139958e60b6c0c8e173c38e",\n              "dcb494c232a7c57dcb1fc51963fb128a6e60a355b0de936d7d2b1ce422a05f4f",\n              "da535aab842e2ca693bf0815634b161b0ad0d3bcba68d0949b57d56bb1125dd3",\n              "f3e35e805bb2f649c824162e226fd04a367379dfbebc2bc7d47a656c07cb5486",\n              "2e2250e98a32a4c737ea58991adfa8472cf1c3020f2335539badaedff616146a",\n              "62ae1c719a68c792b75e31f7f724eae75781222dfa8a3de50432bd884e6f8fc4",\n              "b1bd45c0dd7027fce95dbd0188764e9775547c06944aa28ade66f1c6c064373b",\n              "dbd902a84cbc5e9f018c484ec8dd5a4a2c0b6377aa6c4f742dbfb769c535afd9",\n              "7c4b6609f63fff584424b452bbfa9149cfd64335ca3005e677261315c0f503c6",\n              "7d1970e00fe99db108e209073f9151a5a631e3804a82282a713b60781ce2fde5",\n              "d55b4b0a5d85ea6413523832b12bc6205c6d5185e8846e9cdb5e49d29672e66e",\n              "68252e402258adc720d1cde870fb26af4470fecf05506c0f3cf846be1f65c91f",\n              "c9381a6e79573dd8fb56b4752b3b6c1f580a3856db250d9645ea8b0cc8f4a9af",\n              "1d4a8afab8b22aed71698959945258047b6fe508762918c3e306bff23db0c904",\n              "d43cbacab21ed52daf33603c975b8df24e848d1ea4ba0bb22463cd5e9d64a43c",\n              "090721c630ec0b11a83111e65c2345d8b92b1b6f90f74f298e132f72258581d4",\n              "71cacf236cdc3dc9538db554207e3c42cb459791e244c39efececac4bea78c1f",\n              "5eaf8c7048e4dd9657c45871aae17c640694b66ca8579527ce885528aec4e7ac",\n              "c50702546dfd93690c4416b9afd3fe8c1a9e1cd3c72c7a6f4859f7756160b649",\n              "849dfde445daf0b86280b20bee85a2d0f6e45fbc30ab76ded87480fc5c6e472c",\n              "a26ecff3a526de123b062c2969521c921912c9bb9a67bb6bf9f5c93870c2d104",\n              "70ae9f98bf78a1199c58b20239833e317a7cddd1cbed1d82b89aa996ad060e40",\n              "fa5f4671c24b7614b3b0854da1af003b6c14832b1c893324a43aff8f0608e792",\n              "590e2b95b8e2937738b8dcd67917333d51d6176e36ec0b00ed665ceb7fa5b293",\n              "585fd32a43c4df889939fe1e3c1fa02de4d55b7d37d949f3aff89c9edf1905c8",\n              "378cb020db4a0771b56d416b01c0c7b526e599c48d342d7fa520ddbe16c060d9",\n              "b1e0f997e615f781095265ade124d7ed4a3cdee6166047326c68b859a2e7735e",\n              "18bf41e956ce08044b805bcb1b800b96b5bb967a98a3fb27affbe11c73ebab25",\n              "a527afb2f503358343d2f16be030fc9918ee7aa7d3cb08ce1d907b502b84e4c8",\n              "681e50b9e134cd2c79991ca02249c3f1e14fbef0cc361e33170bbd2e666596d7",\n              "6ac0bf3b91e9461530edb0d4293dbf657b580744ddd96a60606164886e58d854",\n              "dcb8109a0e0da473ae1cca0cfed82b77e258ad4d2f67ca1aa3d40dcf1c9c4660",\n              "ef5dcaa4bf01fe62e553c98b4b25e2cc42520fc3db8e5d3bef64f2a55e9fcd27",\n              "ed22ecfdbb355bdb78c8d75e641501fa0775808b0e3ddcb591a9c25158ffa106",\n              "35cbe6d66a5d6db2bd9984d1171f17f4517a2562c3571d565688d766f5c3a970",\n              "cfeda5adb9180b2c45af59326da92e4e73a30d6bce3b2a5b9a37594a09d9792e",\n              "9f083da097778063a497ec1822e2aeeb7f16cd0cab80caff9f552d3cdb00649b"\n            ],\n            "slots": [\n              0,\n              8,\n              16,\n              24,\n              32,\n              41\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  21.5,\n                  21.890625\n                ],\n                "prediction": "B",\n                "lm_tokens": 2054,\n                "visual_tokens": 1960,\n                "prompt_sha256": "7a6e32c64cbe2431636d725bb25b40dc13f3f2f2d868ba744cd394c421a7d2ba"\n              },\n              "reversed": {\n                "logits": [\n                  21.640625,\n                  21.90625\n                ],\n                "prediction": "B",\n                "lm_tokens": 2054,\n                "visual_tokens": 1960,\n                "prompt_sha256": "267af125c31175965e247126fc0abbb896f40e6c1b833aa1d468ab6b9c6b8002"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  21.484375,\n                  22.015625\n                ],\n                "prediction": "B",\n                "prompt_sha256": "7a6e32c64cbe2431636d725bb25b40dc13f3f2f2d868ba744cd394c421a7d2ba",\n                "lm_tokens": 2054,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  21.625,\n                  22.125\n                ],\n                "prediction": "B",\n                "prompt_sha256": "267af125c31175965e247126fc0abbb896f40e6c1b833aa1d468ab6b9c6b8002",\n                "lm_tokens": 2054,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              0,\n              6,\n              12,\n              18,\n              24,\n              30,\n              36,\n              42,\n              48,\n              54,\n              60,\n              66,\n              72,\n              408,\n              414,\n              420,\n              426,\n              432,\n              438,\n              444,\n              450,\n              456,\n              462,\n              468,\n              474,\n              480,\n              576,\n              582,\n              588,\n              594,\n              600,\n              606,\n              612,\n              618,\n              624,\n              630,\n              636,\n              642,\n              648\n            ],\n            "hashes": [\n              "6217617ade9e407f40c6e660d18d6aacb8df22b96b4a83baab3aa89e689f6b87",\n              "1896bf7a92e8c7463a7a272b1828364ae8e79298acea9419a0624d817b7b6966",\n              "f29db48fed8aaf6df9081d00b8494640633ea10985f1a727a91511fb4e0b69ba",\n              "22b9633212c737e02eda2348defb39ba84e4ec21c8f22c10f53f4894d49386fc",\n              "0a0363b2298c2e51502642ee8822599c5928705e6108206e784edcf3d5343a5b",\n              "31327b6533e9bf92a2b5e8220ac5137d6d7eba6f2d42779e5649b1551573c8fb",\n              "60b3d73c7b498ed42b60f0c873a4bbae452fd2930656aef267a15680d8d963e7",\n              "9ea9ff1aa234c83fda5312e2915f6de8b3b1166a4feb8a38d43fb83ffae47f9c",\n              "f6457a1647d7429ac991f8826f56d3b5476f18786fddec6e22ed3b7ced0ba837",\n              "7ac9d4917e4f48ec118c15c2eb520b86533a4c0af210cd924259127fc94949d8",\n              "6ee3872aa04bd09b9ac13a9f4faa007990f15da653a16c0e38758499284ccdeb",\n              "eeab263ef6979faf3a0effc7cfce36ad188fced8e554af44842f836fd0ee6e3b",\n              "b8f57ec7fbc96ec7d5a4f3038ce329d705b625f08ddf1507d797505637581e4a",\n              "7c4b6609f63fff584424b452bbfa9149cfd64335ca3005e677261315c0f503c6",\n              "68e7ce553d45d2041341fcd712d6cc3f6b15e38739fc69c9b7f9970bc0e7bb19",\n              "43d3c0a778e85f6db171316b2a1a671f5feab47e9b7a65c72a2ef49b4206dc8a",\n              "b08eb94a1dedc502c6e9e6504ac757885f3c87bfebb2dde012750cc5c279dd8f",\n              "7d1970e00fe99db108e209073f9151a5a631e3804a82282a713b60781ce2fde5",\n              "d52706d72a49354ff450de3cceb6f263d0d134f8220ac6f8ff335490934573b6",\n              "eecae9b63892d551642b7253bfc6cc160033486dbc400a77d7dd65150bad8c52",\n              "5eab027d5123216bd315cac75a1bd489ef4e3ed2b922d3d8284ca037667de474",\n              "d55b4b0a5d85ea6413523832b12bc6205c6d5185e8846e9cdb5e49d29672e66e",\n              "aa0f8218a2b368fda002170fe2afac0829d3aa898565fb1a807145cee8e54c9f",\n              "73710825602abc518605bdd339f12fbb9947be8c88bf52541b5274c223e31dce",\n              "a9924f6bd43822f85f1e0fce4133c4e811e1a45c2b90ca7e5ed2e8d73ab1c9df",\n              "68252e402258adc720d1cde870fb26af4470fecf05506c0f3cf846be1f65c91f",\n              "090721c630ec0b11a83111e65c2345d8b92b1b6f90f74f298e132f72258581d4",\n              "f97a4eaec09721db23dc44c712279df3c8a32195871c9c2a75cc81c745c7c2a9",\n              "d4a535cd54220cc1edc799bef4d7e2d5a06f42407608022f92a766182fa2a22b",\n              "6e80c571eddb5130527a9dfaca675990557d862608f8655d2e0043cb56eb32ce",\n              "71cacf236cdc3dc9538db554207e3c42cb459791e244c39efececac4bea78c1f",\n              "897e40614cc042e10a4cabab24b696715ccb573c69c0b15410c6158107c032fc",\n              "8272dd7d27ac1c2516067db913745ce931dc976c6948456b59e9807b647fe1eb",\n              "a596056db50eaa83001c6e14bf671f099060de6252e84806143e07dd168c2dea",\n              "5eaf8c7048e4dd9657c45871aae17c640694b66ca8579527ce885528aec4e7ac",\n              "065d2b8da39c08457be6a5bd0294d2dcc42fed3ae33ec9c60343a8ccb0e9d0be",\n              "a75b134442d07303bc7b8aefb69d6447b287c52c231b11ca9dfd009d7ce1cffb",\n              "f0df5077fd2f79c8bbcee306422b3907d0285a063cc920ae9eaa5b8ce2308e94",\n              "c50702546dfd93690c4416b9afd3fe8c1a9e1cd3c72c7a6f4859f7756160b649"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": 0.21875,\n              "yes_no": {\n                "canonical": [\n                  21.609375,\n                  21.640625\n                ],\n                "reversed": [\n                  21.84375,\n                  21.375\n                ]\n              }\n            },\n            {\n              "slots": [\n                3,\n                6\n              ],\n              "relevance": 0.1953125,\n              "yes_no": {\n                "canonical": [\n                  21.765625,\n                  21.921875\n                ],\n                "reversed": [\n                  22.140625,\n                  21.59375\n                ]\n              }\n            },\n            {\n              "slots": [\n                6,\n                9\n              ],\n              "relevance": -0.390625,\n              "yes_no": {\n                "canonical": [\n                  21.515625,\n                  22.171875\n                ],\n                "reversed": [\n                  21.546875,\n                  21.671875\n                ]\n              }\n            },\n            {\n              "slots": [\n                10,\n                13\n              ],\n              "relevance": -0.3671875,\n              "yes_no": {\n                "canonical": [\n                  21.546875,\n                  22.171875\n                ],\n                "reversed": [\n                  21.5625,\n                  21.671875\n                ]\n              }\n            },\n            {\n              "slots": [\n                13,\n                16\n              ],\n              "relevance": -0.390625,\n              "yes_no": {\n                "canonical": [\n                  21.546875,\n                  22.1875\n                ],\n                "reversed": [\n                  21.546875,\n                  21.6875\n                ]\n              }\n            },\n            {\n              "slots": [\n                17,\n                20\n              ],\n              "relevance": -0.34375,\n              "yes_no": {\n                "canonical": [\n                  21.546875,\n                  22.125\n                ],\n                "reversed": [\n                  21.5625,\n                  21.671875\n                ]\n              }\n            },\n            {\n              "slots": [\n                20,\n                23\n              ],\n              "relevance": -0.3671875,\n              "yes_no": {\n                "canonical": [\n                  21.53125,\n                  22.15625\n                ],\n                "reversed": [\n                  21.546875,\n                  21.65625\n                ]\n              }\n            },\n            {\n              "slots": [\n                24,\n                27\n              ],\n              "relevance": -0.3515625,\n              "yes_no": {\n                "canonical": [\n                  21.515625,\n                  22.109375\n                ],\n                "reversed": [\n                  21.515625,\n                  21.625\n                ]\n              }\n            },\n            {\n              "slots": [\n                27,\n                30\n              ],\n              "relevance": -0.375,\n              "yes_no": {\n                "canonical": [\n                  21.53125,\n                  22.15625\n                ],\n                "reversed": [\n                  21.53125,\n                  21.65625\n                ]\n              }\n            },\n            {\n              "slots": [\n                31,\n                34\n              ],\n              "relevance": -0.3515625,\n              "yes_no": {\n                "canonical": [\n                  21.546875,\n                  22.140625\n                ],\n                "reversed": [\n                  21.53125,\n                  21.640625\n                ]\n              }\n            },\n            {\n              "slots": [\n                34,\n                37\n              ],\n              "relevance": -0.359375,\n              "yes_no": {\n                "canonical": [\n                  21.5,\n                  22.109375\n                ],\n                "reversed": [\n                  21.515625,\n                  21.625\n                ]\n              }\n            },\n            {\n              "slots": [\n                38,\n                41\n              ],\n              "relevance": -0.375,\n              "yes_no": {\n                "canonical": [\n                  21.53125,\n                  22.171875\n                ],\n                "reversed": [\n                  21.546875,\n                  21.65625\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                0,\n                3\n              ],\n              [\n                17,\n                20\n              ],\n              [\n                24,\n                27\n              ]\n            ],\n            "constrained_windows": [\n              [\n                3,\n                6\n              ],\n              [\n                17,\n                20\n              ],\n              [\n                24,\n                27\n              ]\n            ],\n            "indices": [\n              90,\n              132,\n              426,\n              462,\n              594,\n              630\n            ],\n            "scanned_indices": 150\n          }\n        },\n        "b": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480,\n              504,\n              528,\n              552,\n              576,\n              600,\n              624,\n              648,\n              672,\n              696,\n              720,\n              744,\n              768,\n              792,\n              816,\n              840,\n              864,\n              888,\n              912,\n              936,\n              960,\n              984,\n              1008,\n              1032,\n              1056,\n              1080\n            ],\n            "hashes": [\n              "b54683d2d225c1256e1a159a2e52fbb6538bd80bf911c6a3d83a2c652534cd30",\n              "5084c06df16fef200ff2fc0368b3397219f3e861afc78db61609e22c136abdb8",\n              "0d642ccba77c98cdcbb096264a390cc468813dbab70fcf11cbe309d118d6a872",\n              "3727e09920dd772a0533cf11399f8aae33271ca62df99f785ca28417287db44f",\n              "2f3c361d145baf13e2b052aeb21641e09d99a71a03938963401ab34486e4ea05",\n              "36fc91c3e244f761c850c8df17a80a2a882276270b05cd4dd393c7f748770ae7",\n              "94a70e604372242c5cd285ddcedfd8cd5c787bcbd026cbffa82ccce4499dd6b5",\n              "f7e3420afc80f74abd82bee8c9f7dd242b36079db6ab0e956e3175c8157dfbc9",\n              "26172b51ead5d075ebf86c96c503145170ec8ec493b401b9c7267a08a4acaf28",\n              "d340b77831787dd80c8402ac14ae8e0d60b5f03d1139958e60b6c0c8e173c38e",\n              "dcb494c232a7c57dcb1fc51963fb128a6e60a355b0de936d7d2b1ce422a05f4f",\n              "da535aab842e2ca693bf0815634b161b0ad0d3bcba68d0949b57d56bb1125dd3",\n              "f3e35e805bb2f649c824162e226fd04a367379dfbebc2bc7d47a656c07cb5486",\n              "2e2250e98a32a4c737ea58991adfa8472cf1c3020f2335539badaedff616146a",\n              "62ae1c719a68c792b75e31f7f724eae75781222dfa8a3de50432bd884e6f8fc4",\n              "b1bd45c0dd7027fce95dbd0188764e9775547c06944aa28ade66f1c6c064373b",\n              "dbd902a84cbc5e9f018c484ec8dd5a4a2c0b6377aa6c4f742dbfb769c535afd9",\n              "7c4b6609f63fff584424b452bbfa9149cfd64335ca3005e677261315c0f503c6",\n              "7d1970e00fe99db108e209073f9151a5a631e3804a82282a713b60781ce2fde5",\n              "d55b4b0a5d85ea6413523832b12bc6205c6d5185e8846e9cdb5e49d29672e66e",\n              "68252e402258adc720d1cde870fb26af4470fecf05506c0f3cf846be1f65c91f",\n              "c9381a6e79573dd8fb56b4752b3b6c1f580a3856db250d9645ea8b0cc8f4a9af",\n              "1d4a8afab8b22aed71698959945258047b6fe508762918c3e306bff23db0c904",\n              "d43cbacab21ed52daf33603c975b8df24e848d1ea4ba0bb22463cd5e9d64a43c",\n              "090721c630ec0b11a83111e65c2345d8b92b1b6f90f74f298e132f72258581d4",\n              "71cacf236cdc3dc9538db554207e3c42cb459791e244c39efececac4bea78c1f",\n              "5eaf8c7048e4dd9657c45871aae17c640694b66ca8579527ce885528aec4e7ac",\n              "c50702546dfd93690c4416b9afd3fe8c1a9e1cd3c72c7a6f4859f7756160b649",\n              "849dfde445daf0b86280b20bee85a2d0f6e45fbc30ab76ded87480fc5c6e472c",\n              "a26ecff3a526de123b062c2969521c921912c9bb9a67bb6bf9f5c93870c2d104",\n              "70ae9f98bf78a1199c58b20239833e317a7cddd1cbed1d82b89aa996ad060e40",\n              "fa5f4671c24b7614b3b0854da1af003b6c14832b1c893324a43aff8f0608e792",\n              "590e2b95b8e2937738b8dcd67917333d51d6176e36ec0b00ed665ceb7fa5b293",\n              "585fd32a43c4df889939fe1e3c1fa02de4d55b7d37d949f3aff89c9edf1905c8",\n              "378cb020db4a0771b56d416b01c0c7b526e599c48d342d7fa520ddbe16c060d9",\n              "b1e0f997e615f781095265ade124d7ed4a3cdee6166047326c68b859a2e7735e",\n              "18bf41e956ce08044b805bcb1b800b96b5bb967a98a3fb27affbe11c73ebab25",\n              "a527afb2f503358343d2f16be030fc9918ee7aa7d3cb08ce1d907b502b84e4c8",\n              "681e50b9e134cd2c79991ca02249c3f1e14fbef0cc361e33170bbd2e666596d7",\n              "6ac0bf3b91e9461530edb0d4293dbf657b580744ddd96a60606164886e58d854",\n              "dcb8109a0e0da473ae1cca0cfed82b77e258ad4d2f67ca1aa3d40dcf1c9c4660",\n              "ef5dcaa4bf01fe62e553c98b4b25e2cc42520fc3db8e5d3bef64f2a55e9fcd27",\n              "ed22ecfdbb355bdb78c8d75e641501fa0775808b0e3ddcb591a9c25158ffa106",\n              "35cbe6d66a5d6db2bd9984d1171f17f4517a2562c3571d565688d766f5c3a970",\n              "cfeda5adb9180b2c45af59326da92e4e73a30d6bce3b2a5b9a37594a09d9792e",\n              "9f083da097778063a497ec1822e2aeeb7f16cd0cab80caff9f552d3cdb00649b"\n            ],\n            "slots": [\n              0,\n              8,\n              16,\n              24,\n              32,\n              41\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  21.71875,\n                  21.625\n                ],\n                "prediction": "A",\n                "lm_tokens": 2054,\n                "visual_tokens": 1960,\n                "prompt_sha256": "7a6e32c64cbe2431636d725bb25b40dc13f3f2f2d868ba744cd394c421a7d2ba"\n              },\n              "reversed": {\n                "logits": [\n                  21.71875,\n                  21.671875\n                ],\n                "prediction": "A",\n                "lm_tokens": 2054,\n                "visual_tokens": 1960,\n                "prompt_sha256": "267af125c31175965e247126fc0abbb896f40e6c1b833aa1d468ab6b9c6b8002"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  22.15625,\n                  21.859375\n                ],\n                "prediction": "A",\n                "prompt_sha256": "7a6e32c64cbe2431636d725bb25b40dc13f3f2f2d868ba744cd394c421a7d2ba",\n                "lm_tokens": 2054,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  22.125,\n                  22.015625\n                ],\n                "prediction": "A",\n                "prompt_sha256": "267af125c31175965e247126fc0abbb896f40e6c1b833aa1d468ab6b9c6b8002",\n                "lm_tokens": 2054,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              0,\n              6,\n              12,\n              18,\n              24,\n              30,\n              36,\n              42,\n              48,\n              54,\n              60,\n              66,\n              72,\n              408,\n              414,\n              420,\n              426,\n              432,\n              438,\n              444,\n              450,\n              456,\n              462,\n              468,\n              474,\n              480,\n              576,\n              582,\n              588,\n              594,\n              600,\n              606,\n              612,\n              618,\n              624,\n              630,\n              636,\n              642,\n              648\n            ],\n            "hashes": [\n              "b54683d2d225c1256e1a159a2e52fbb6538bd80bf911c6a3d83a2c652534cd30",\n              "6b36c901a9eab3c118dd2c293304c7a517a1dfa0b335d1249b9683cfe80c767d",\n              "776dbc6c032560044ca7d363e89f7ceafde30d09d66261f7823cbdf5df5913f8",\n              "737d0b7eed51a9ce37936e17c6969905f7182df31a1e91b9d712754f1d1772a3",\n              "5084c06df16fef200ff2fc0368b3397219f3e861afc78db61609e22c136abdb8",\n              "e0be87034420bd6fe53be105fd4e734002c10655695e61445004564b91821883",\n              "571314ccde6821a481e50ca4ca1d39fd9ba9235c164c324c40445595bbd9ea92",\n              "d39d635937cd71fd8ca8a3280c94757b10b10f0a646d7b86989032676e1b2683",\n              "0d642ccba77c98cdcbb096264a390cc468813dbab70fcf11cbe309d118d6a872",\n              "6928047ceee46d45b1e4ed6011ef7600db2ac884945efa22190f8262c66e3bbb",\n              "e151abbdac40b87dc820bb15c35870fa5138af065ff8da33ea1e295a9d553e75",\n              "220a37bff855992cd8f031d6ef7c458d04f4c34291636102a9eb84b24909703c",\n              "3727e09920dd772a0533cf11399f8aae33271ca62df99f785ca28417287db44f",\n              "7c4b6609f63fff584424b452bbfa9149cfd64335ca3005e677261315c0f503c6",\n              "68e7ce553d45d2041341fcd712d6cc3f6b15e38739fc69c9b7f9970bc0e7bb19",\n              "43d3c0a778e85f6db171316b2a1a671f5feab47e9b7a65c72a2ef49b4206dc8a",\n              "b08eb94a1dedc502c6e9e6504ac757885f3c87bfebb2dde012750cc5c279dd8f",\n              "7d1970e00fe99db108e209073f9151a5a631e3804a82282a713b60781ce2fde5",\n              "d52706d72a49354ff450de3cceb6f263d0d134f8220ac6f8ff335490934573b6",\n              "eecae9b63892d551642b7253bfc6cc160033486dbc400a77d7dd65150bad8c52",\n              "5eab027d5123216bd315cac75a1bd489ef4e3ed2b922d3d8284ca037667de474",\n              "d55b4b0a5d85ea6413523832b12bc6205c6d5185e8846e9cdb5e49d29672e66e",\n              "aa0f8218a2b368fda002170fe2afac0829d3aa898565fb1a807145cee8e54c9f",\n              "73710825602abc518605bdd339f12fbb9947be8c88bf52541b5274c223e31dce",\n              "a9924f6bd43822f85f1e0fce4133c4e811e1a45c2b90ca7e5ed2e8d73ab1c9df",\n              "68252e402258adc720d1cde870fb26af4470fecf05506c0f3cf846be1f65c91f",\n              "090721c630ec0b11a83111e65c2345d8b92b1b6f90f74f298e132f72258581d4",\n              "f97a4eaec09721db23dc44c712279df3c8a32195871c9c2a75cc81c745c7c2a9",\n              "d4a535cd54220cc1edc799bef4d7e2d5a06f42407608022f92a766182fa2a22b",\n              "6e80c571eddb5130527a9dfaca675990557d862608f8655d2e0043cb56eb32ce",\n              "71cacf236cdc3dc9538db554207e3c42cb459791e244c39efececac4bea78c1f",\n              "897e40614cc042e10a4cabab24b696715ccb573c69c0b15410c6158107c032fc",\n              "8272dd7d27ac1c2516067db913745ce931dc976c6948456b59e9807b647fe1eb",\n              "a596056db50eaa83001c6e14bf671f099060de6252e84806143e07dd168c2dea",\n              "5eaf8c7048e4dd9657c45871aae17c640694b66ca8579527ce885528aec4e7ac",\n              "065d2b8da39c08457be6a5bd0294d2dcc42fed3ae33ec9c60343a8ccb0e9d0be",\n              "a75b134442d07303bc7b8aefb69d6447b287c52c231b11ca9dfd009d7ce1cffb",\n              "f0df5077fd2f79c8bbcee306422b3907d0285a063cc920ae9eaa5b8ce2308e94",\n              "c50702546dfd93690c4416b9afd3fe8c1a9e1cd3c72c7a6f4859f7756160b649"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": 0.484375,\n              "yes_no": {\n                "canonical": [\n                  22.09375,\n                  21.78125\n                ],\n                "reversed": [\n                  22.25,\n                  21.59375\n                ]\n              }\n            },\n            {\n              "slots": [\n                3,\n                6\n              ],\n              "relevance": 0.03125,\n              "yes_no": {\n                "canonical": [\n                  21.65625,\n                  22.03125\n                ],\n                "reversed": [\n                  22.015625,\n                  21.578125\n                ]\n              }\n            },\n            {\n              "slots": [\n                6,\n                9\n              ],\n              "relevance": -0.390625,\n              "yes_no": {\n                "canonical": [\n                  21.515625,\n                  22.171875\n                ],\n                "reversed": [\n                  21.546875,\n                  21.671875\n                ]\n              }\n            },\n            {\n              "slots": [\n                10,\n                13\n              ],\n              "relevance": -0.3671875,\n              "yes_no": {\n                "canonical": [\n                  21.546875,\n                  22.171875\n                ],\n                "reversed": [\n                  21.5625,\n                  21.671875\n                ]\n              }\n            },\n            {\n              "slots": [\n                13,\n                16\n              ],\n              "relevance": -0.390625,\n              "yes_no": {\n                "canonical": [\n                  21.546875,\n                  22.1875\n                ],\n                "reversed": [\n                  21.546875,\n                  21.6875\n                ]\n              }\n            },\n            {\n              "slots": [\n                17,\n                20\n              ],\n              "relevance": -0.34375,\n              "yes_no": {\n                "canonical": [\n                  21.546875,\n                  22.125\n                ],\n                "reversed": [\n                  21.5625,\n                  21.671875\n                ]\n              }\n            },\n            {\n              "slots": [\n                20,\n                23\n              ],\n              "relevance": -0.3671875,\n              "yes_no": {\n                "canonical": [\n                  21.53125,\n                  22.15625\n                ],\n                "reversed": [\n                  21.546875,\n                  21.65625\n                ]\n              }\n            },\n            {\n              "slots": [\n                24,\n                27\n              ],\n              "relevance": -0.3515625,\n              "yes_no": {\n                "canonical": [\n                  21.515625,\n                  22.109375\n                ],\n                "reversed": [\n                  21.515625,\n                  21.625\n                ]\n              }\n            },\n            {\n              "slots": [\n                27,\n                30\n              ],\n              "relevance": -0.375,\n              "yes_no": {\n                "canonical": [\n                  21.53125,\n                  22.15625\n                ],\n                "reversed": [\n                  21.53125,\n                  21.65625\n                ]\n              }\n            },\n            {\n              "slots": [\n                31,\n                34\n              ],\n              "relevance": -0.3515625,\n              "yes_no": {\n                "canonical": [\n                  21.546875,\n                  22.140625\n                ],\n                "reversed": [\n                  21.53125,\n                  21.640625\n                ]\n              }\n            },\n            {\n              "slots": [\n                34,\n                37\n              ],\n              "relevance": -0.359375,\n              "yes_no": {\n                "canonical": [\n                  21.5,\n                  22.109375\n                ],\n                "reversed": [\n                  21.515625,\n                  21.625\n                ]\n              }\n            },\n            {\n              "slots": [\n                38,\n                41\n              ],\n              "relevance": -0.375,\n              "yes_no": {\n                "canonical": [\n                  21.53125,\n                  22.171875\n                ],\n                "reversed": [\n                  21.546875,\n                  21.65625\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                0,\n                3\n              ],\n              [\n                17,\n                20\n              ],\n              [\n                24,\n                27\n              ]\n            ],\n            "constrained_windows": [\n              [\n                3,\n                6\n              ],\n              [\n                17,\n                20\n              ],\n              [\n                24,\n                27\n              ]\n            ],\n            "indices": [\n              90,\n              132,\n              426,\n              462,\n              594,\n              630\n            ],\n            "scanned_indices": 150\n          }\n        }\n      },\n      "media": {\n        "a": {\n          "file": "videos/twinpast_0118/A.mp4",\n          "bytes": 15711648,\n          "sha256": "1e2c077d7960adadaecd54d5eacb5ddbfdb9fa4e0d760f90d44da17ec0671d6c"\n        },\n        "b": {\n          "file": "videos/twinpast_0118/B.mp4",\n          "bytes": 15785368,\n          "sha256": "e58b4b79362be9876edb284a9cf441a7f2aac8f1f5ad98619eab2a37c550f479"\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0338",\n      "query_s": 39.208333333333336,\n      "question": "At the end of the earlier tiled-floor segment, immediately before the scene changes, was the small yellow box visible or hidden beneath the red cover?",\n      "options": [\n        "Hidden beneath the red cover.",\n        "Visible on the floor."\n      ],\n      "branches": {\n        "a": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480,\n              504,\n              528,\n              552,\n              576,\n              600,\n              624,\n              648,\n              672,\n              696,\n              720,\n              744,\n              768,\n              792,\n              816,\n              840,\n              864,\n              888,\n              912,\n              936\n            ],\n            "hashes": [\n              "c6aeeea2873bfa0d216b99ac80e427768dd09d62342430eab3cb813284100fb7",\n              "cbdfd4d61184acb9297ac84766511551ce8a7691ab8e70daed89c22acfd98f2d",\n              "f323155c634be0435ef6eac42e23fca6d2b24a63e062d605f22906f6d8276209",\n              "7c3b6dce3d05e7d8a0b8c4714383d567315d1adfbe6476b21e471a3db38900f0",\n              "25ac6a00dedf16b26f6aff2ece63b071de7bf1e1935ac1c8bc6841dcc529f151",\n              "be906b3398d3e3fcd13578c6ac450f4e002d6ba93658759a48f26d1a54df26c4",\n              "b19d4136e4e550356f5d84ef35337c59e0d230518cbfc1f3b71348dc03324474",\n              "ff77587119c6a8798f04801686026fe5d90d310119135f225a04bb6b2f464901",\n              "6a1685aeb048cffd1116ab978ade6a66abaf905534dac1c0f4ffd27ac56a5901",\n              "0741966a6de468544f358b98cc4517d291f724f44c1abbb6f1f6d3ba9242a178",\n              "cd49feb4db5fddfd69428633b181863d0291abfe3e42d2f9ae4ccd61b1645e05",\n              "0e56aebccefada3d092c9056d1b15dd09ca984a4a819c28b7f348c4ef51223a4",\n              "6151bfc1ed7f98f68056ec12182fdad5e08e1946c32a2e70e097b9ce3dc67367",\n              "c3f09ebeced95f5e92349bae7ff912161a0bdb5f2899b31327249bc3da0fff57",\n              "e5dffc1f43b0cc7a3c5723a98c9b39157aa0e6c87a3862c7a6b1b6f8beff6294",\n              "1970a2b27d9a708a8e929525c097d52bc36461aba608432962959d1394911bba",\n              "bfd48d711dddcba6f96f108412ce529a48fe2c92fb6eff0e1064daec78ca8998",\n              "d1e09e74b5ce7f334fa2e6aff95fc48d2a8c4359e28f2a7986a6ea38d201d6ff",\n              "d905da1a037c61570edf59d046207250fec0414643248b1b3adb86ffc0d87e8a",\n              "d6c9948612ab94e766d6a11f8d17035c33d24456ef6bce89d8f1808cbc005b94",\n              "2fe3d30ff6121d9aeb2b50a2c9bdbae8fa666a1d4185b3bcc3f028e529a5da6b",\n              "aad69b810232bcf28d7f5184c2f58bfe9ba66498228459f9aa650bb6c4a58ba1",\n              "b8fcec5c2205e618f367148ba8cf82cf6a4015c02dd8747f39cbb851699f718a",\n              "103cd72f1d8bf3525153f3fac178dcf49fbb8846712cb63e83df4758552eba14",\n              "7d91013b7208fe3853955487e5d78617dab99f43b80a5a49ac82e0b8b59928c6",\n              "2d0e578b0d29bdee94a146e848f3c6ca78db69532514c04bd66680fdfa12bd6a",\n              "70644fd5480fa0c535c092c6bfcf08dd943df14d760e9395d72869a0956de866",\n              "e7974c0129cee8f38cf9325db1dfcdf694cf9cff674fbb5e9436c22499ea58e0",\n              "9b8a145d17d0d3d4b1232c26f5b45943750a600242903696c75ba30f2802dfa2",\n              "8ef99da522dc5944c9bf271d3fb37149f16f1e294aecd0a5e00a01c1869690a7",\n              "46f2b5a4a160a6ccecbf26ab7ab082622899af14adbb634c52ea67d41c112ccd",\n              "fd8c4bd95220b962ba206e5eabe167485bc8b3f3a407d27dbe76baa6bf522899",\n              "da2602d244f1cc4ae1d614c4d51e1aa7f7806d4bd03dfac76cb21b108bb360bf",\n              "f9dbc6fec7958f3200a51a4d4c31e7a177ca7c80685d02dc62b5e8b53de31082",\n              "c31009568bef6b13f9345e605b3783fa6f9d68437019268ca673b152bd19be49",\n              "c3f135a0a0ca48d494dc23ffe076642d00d203f3453b1fa250f0541dca45eab7",\n              "1df9d114d0cae9670b97c2ff36b0975eeeb47d2d72eac309bb14b4629af5b4a8",\n              "50577a61830e56b94433223b2947205fb6e7c2f80942dccf00d360420f1ea7a3",\n              "0e6f437619760f812f033d7d60bd468d02ecc2e9c41be0e21c02fda33f968c37",\n              "60c398f36bebdfa26ea383265240eb970941b5c88e7505e8c8c685b249b0625e"\n            ],\n            "slots": [\n              0,\n              7,\n              14,\n              21,\n              28,\n              35\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  21.078125,\n                  21.859375\n                ],\n                "prediction": "B",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960,\n                "prompt_sha256": "d2537dbb0b1c2b652baf303bc1b8b410abf6e6b26f7a363935d38414ac88129b"\n              },\n              "reversed": {\n                "logits": [\n                  21.53125,\n                  21.0625\n                ],\n                "prediction": "A",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960,\n                "prompt_sha256": "0049fd7060241150d8ddcab600856257969942b4280d6d35e23fbfcf59be80af"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  22.25,\n                  21.109375\n                ],\n                "prediction": "A",\n                "prompt_sha256": "d2537dbb0b1c2b652baf303bc1b8b410abf6e6b26f7a363935d38414ac88129b",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  22.109375,\n                  20.234375\n                ],\n                "prediction": "A",\n                "prompt_sha256": "0049fd7060241150d8ddcab600856257969942b4280d6d35e23fbfcf59be80af",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              0,\n              6,\n              12,\n              18,\n              24,\n              30,\n              36,\n              42,\n              48,\n              54,\n              60,\n              66,\n              72,\n              264,\n              270,\n              276,\n              282,\n              288,\n              294,\n              300,\n              306,\n              312,\n              318,\n              324,\n              330,\n              336,\n              624,\n              630,\n              636,\n              642,\n              648,\n              654,\n              660,\n              666,\n              672,\n              678,\n              684,\n              690,\n              696\n            ],\n            "hashes": [\n              "c6aeeea2873bfa0d216b99ac80e427768dd09d62342430eab3cb813284100fb7",\n              "e715a53c58cdb0a5a80498b582e156b1e14a84e408aa928fad8ac1b5b3cfe88e",\n              "2e5d5aea2708747b9e5c619ead148a13af084c417f02b78a0d2be028cb3a6118",\n              "9fb5dd1975b822d30ccc46669f45b6775dc6f29ac874810eaaea31f33f949028",\n              "cbdfd4d61184acb9297ac84766511551ce8a7691ab8e70daed89c22acfd98f2d",\n              "547338f455b32ca21709f8ff2fef5adee348b986e99343e528d49b3f38343988",\n              "783bbf9f177eaf083a0ed85600720fd080cb6d483f8ef79f81f780f34c567c42",\n              "a7fba189b65de015f1c83bf411edf8ba9ef45787c84b7cbfaece179f5170f417",\n              "f323155c634be0435ef6eac42e23fca6d2b24a63e062d605f22906f6d8276209",\n              "3db8960a1e62eebc7963e7c55e0a2baeb3f1966704da60a54ab61d708c7850f7",\n              "6e920c0fe7cc581f59e96aadf8675387c2763f96583761e4b5145bc919cbd90c",\n              "bb2c51f438779ccfb6f720928a59ca1c7dcb38baf26204c04df10a4acea3da61",\n              "7c3b6dce3d05e7d8a0b8c4714383d567315d1adfbe6476b21e471a3db38900f0",\n              "0e56aebccefada3d092c9056d1b15dd09ca984a4a819c28b7f348c4ef51223a4",\n              "4ade3f76d8c9e1580a1430a35e0f202220c140dde57007da9317d1637a58535f",\n              "110562f3657aa18f458619fd68b12069c194c92fa879d280eaf95533b99bcb75",\n              "226179ed9c84caf8e95f88a977c8c7aecd8b690d793df779097061e66a301fde",\n              "6151bfc1ed7f98f68056ec12182fdad5e08e1946c32a2e70e097b9ce3dc67367",\n              "ca7099222067d7d22bdad72650ae06999a76c25704152abc32309d39728ec535",\n              "cc99e583714fd1af76c40db7e210fa438ca30390f671205df59ba12ce27914ab",\n              "ef1e0fabd901eae7289229f9fe7eb30d8e430ad8fbf34ce0276aa0b5f2a47eff",\n              "c3f09ebeced95f5e92349bae7ff912161a0bdb5f2899b31327249bc3da0fff57",\n              "a910038cf953fd351f9ce2c6573f55f16bc4cbd41893bc4bb3b186f2c3f2a3a9",\n              "82f2694cc56fbeaf7b6eb9353e05c5077ef07e6a53f1e59eba96d156b59891e5",\n              "98cc8671b44a11e0b63c4029cda405189559bfa028f6f1dd2a1ea513154abede",\n              "e5dffc1f43b0cc7a3c5723a98c9b39157aa0e6c87a3862c7a6b1b6f8beff6294",\n              "70644fd5480fa0c535c092c6bfcf08dd943df14d760e9395d72869a0956de866",\n              "6f388ba1cc90b30db2bc11d98cd7c89aa0936e905811f7caeeb67e4606fa2105",\n              "8b67166209154688438e1634ec5804b3753afa43e249f8a847f036266a7693b0",\n              "2e6c12de848b6e2d7a05e72290372afa6e348c3079785c21bc1f7efc342e7bc8",\n              "e7974c0129cee8f38cf9325db1dfcdf694cf9cff674fbb5e9436c22499ea58e0",\n              "0ae17d33549ca5fc697e280f2367eb7ee58ff86eb5bf4128e2cccd842395cb2e",\n              "699e35542a7fccaa31fe08dd41d213a94f886ea7cdd541164f2f8b6cdf6b2ee8",\n              "b530f6f11c70cf3ccf98c3f3bc3f3af2f56c16507df68f1b2f839a801333286f",\n              "9b8a145d17d0d3d4b1232c26f5b45943750a600242903696c75ba30f2802dfa2",\n              "0169e1124d4ab944b42b49f7a8703a1fc25e44c175ed668d7ee5169aebd37cb0",\n              "ec2850e878d4f2bb2574ba1f9e6415459ffa81e5fcba2114c5de213664fb829c",\n              "1646e003bf3e9aea15c4ea77632da8b76c20d1eda64151461d6687744b782529",\n              "8ef99da522dc5944c9bf271d3fb37149f16f1e294aecd0a5e00a01c1869690a7"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": -0.0859375,\n              "yes_no": {\n                "canonical": [\n                  22.015625,\n                  22.5\n                ],\n                "reversed": [\n                  22.34375,\n                  22.03125\n                ]\n              }\n            },\n            {\n              "slots": [\n                2,\n                5\n              ],\n              "relevance": -1.09375,\n              "yes_no": {\n                "canonical": [\n                  21.421875,\n                  22.890625\n                ],\n                "reversed": [\n                  21.796875,\n                  22.515625\n                ]\n              }\n            },\n            {\n              "slots": [\n                5,\n                8\n              ],\n              "relevance": -0.515625,\n              "yes_no": {\n                "canonical": [\n                  21.34375,\n                  22.140625\n                ],\n                "reversed": [\n                  21.625,\n                  21.859375\n                ]\n              }\n            },\n            {\n              "slots": [\n                8,\n                11\n              ],\n              "relevance": -0.40625,\n              "yes_no": {\n                "canonical": [\n                  21.359375,\n                  22.09375\n                ],\n                "reversed": [\n                  21.6875,\n                  21.765625\n                ]\n              }\n            },\n            {\n              "slots": [\n                11,\n                14\n              ],\n              "relevance": -0.359375,\n              "yes_no": {\n                "canonical": [\n                  21.296875,\n                  22.0\n                ],\n                "reversed": [\n                  21.640625,\n                  21.65625\n                ]\n              }\n            },\n            {\n              "slots": [\n                14,\n                17\n              ],\n              "relevance": -0.5,\n              "yes_no": {\n                "canonical": [\n                  21.40625,\n                  22.203125\n                ],\n                "reversed": [\n                  21.6875,\n                  21.890625\n                ]\n              }\n            },\n            {\n              "slots": [\n                17,\n                20\n              ],\n              "relevance": -0.5703125,\n              "yes_no": {\n                "canonical": [\n                  21.3125,\n                  22.203125\n                ],\n                "reversed": [\n                  21.625,\n                  21.875\n                ]\n              }\n            },\n            {\n              "slots": [\n                20,\n                23\n              ],\n              "relevance": -0.6328125,\n              "yes_no": {\n                "canonical": [\n                  21.25,\n                  22.203125\n                ],\n                "reversed": [\n                  21.609375,\n                  21.921875\n                ]\n              }\n            },\n            {\n              "slots": [\n                23,\n                26\n              ],\n              "relevance": -0.5859375,\n              "yes_no": {\n                "canonical": [\n                  21.34375,\n                  22.265625\n                ],\n                "reversed": [\n                  21.65625,\n                  21.90625\n                ]\n              }\n            },\n            {\n              "slots": [\n                26,\n                29\n              ],\n              "relevance": -0.5078125,\n              "yes_no": {\n                "canonical": [\n                  21.296875,\n                  22.125\n                ],\n                "reversed": [\n                  21.625,\n                  21.8125\n                ]\n              }\n            },\n            {\n              "slots": [\n                29,\n                32\n              ],\n              "relevance": -0.625,\n              "yes_no": {\n                "canonical": [\n                  21.28125,\n                  22.1875\n                ],\n                "reversed": [\n                  21.59375,\n                  21.9375\n                ]\n              }\n            },\n            {\n              "slots": [\n                32,\n                35\n              ],\n              "relevance": -0.53125,\n              "yes_no": {\n                "canonical": [\n                  21.28125,\n                  22.140625\n                ],\n                "reversed": [\n                  21.625,\n                  21.828125\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                0,\n                3\n              ],\n              [\n                11,\n                14\n              ],\n              [\n                26,\n                29\n              ]\n            ],\n            "constrained_windows": [\n              [\n                2,\n                5\n              ],\n              [\n                11,\n                14\n              ],\n              [\n                26,\n                29\n              ]\n            ],\n            "indices": [\n              66,\n              96,\n              282,\n              318,\n              642,\n              678\n            ],\n            "scanned_indices": 141\n          }\n        },\n        "b": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480,\n              504,\n              528,\n              552,\n              576,\n              600,\n              624,\n              648,\n              672,\n              696,\n              720,\n              744,\n              768,\n              792,\n              816,\n              840,\n              864,\n              888,\n              912,\n              936\n            ],\n            "hashes": [\n              "386855e29326f38ff54b64c54365c69b0d2f90b523cfe46827e93859e613af78",\n              "2a260075efbdf566293a2d63a21f3d6abc3c6cbd6f8e014233207c5008e59128",\n              "73e30c70dd62e725fb5f39c0c50fe720e7e709c2fd45303345877e2a22006f2b",\n              "2cfd832854d3d4fe341f02ceebbc2a2b625636cf832a7f2da96a1bf6dff5ea9c",\n              "1dfd30527b26f1c1d46cc7a8095f20734090bf36d4855a50cc0568847c93c774",\n              "be906b3398d3e3fcd13578c6ac450f4e002d6ba93658759a48f26d1a54df26c4",\n              "b19d4136e4e550356f5d84ef35337c59e0d230518cbfc1f3b71348dc03324474",\n              "ff77587119c6a8798f04801686026fe5d90d310119135f225a04bb6b2f464901",\n              "6a1685aeb048cffd1116ab978ade6a66abaf905534dac1c0f4ffd27ac56a5901",\n              "0741966a6de468544f358b98cc4517d291f724f44c1abbb6f1f6d3ba9242a178",\n              "cd49feb4db5fddfd69428633b181863d0291abfe3e42d2f9ae4ccd61b1645e05",\n              "0e56aebccefada3d092c9056d1b15dd09ca984a4a819c28b7f348c4ef51223a4",\n              "6151bfc1ed7f98f68056ec12182fdad5e08e1946c32a2e70e097b9ce3dc67367",\n              "c3f09ebeced95f5e92349bae7ff912161a0bdb5f2899b31327249bc3da0fff57",\n              "e5dffc1f43b0cc7a3c5723a98c9b39157aa0e6c87a3862c7a6b1b6f8beff6294",\n              "1970a2b27d9a708a8e929525c097d52bc36461aba608432962959d1394911bba",\n              "bfd48d711dddcba6f96f108412ce529a48fe2c92fb6eff0e1064daec78ca8998",\n              "d1e09e74b5ce7f334fa2e6aff95fc48d2a8c4359e28f2a7986a6ea38d201d6ff",\n              "d905da1a037c61570edf59d046207250fec0414643248b1b3adb86ffc0d87e8a",\n              "d6c9948612ab94e766d6a11f8d17035c33d24456ef6bce89d8f1808cbc005b94",\n              "2fe3d30ff6121d9aeb2b50a2c9bdbae8fa666a1d4185b3bcc3f028e529a5da6b",\n              "aad69b810232bcf28d7f5184c2f58bfe9ba66498228459f9aa650bb6c4a58ba1",\n              "b8fcec5c2205e618f367148ba8cf82cf6a4015c02dd8747f39cbb851699f718a",\n              "103cd72f1d8bf3525153f3fac178dcf49fbb8846712cb63e83df4758552eba14",\n              "7d91013b7208fe3853955487e5d78617dab99f43b80a5a49ac82e0b8b59928c6",\n              "2d0e578b0d29bdee94a146e848f3c6ca78db69532514c04bd66680fdfa12bd6a",\n              "70644fd5480fa0c535c092c6bfcf08dd943df14d760e9395d72869a0956de866",\n              "e7974c0129cee8f38cf9325db1dfcdf694cf9cff674fbb5e9436c22499ea58e0",\n              "9b8a145d17d0d3d4b1232c26f5b45943750a600242903696c75ba30f2802dfa2",\n              "8ef99da522dc5944c9bf271d3fb37149f16f1e294aecd0a5e00a01c1869690a7",\n              "46f2b5a4a160a6ccecbf26ab7ab082622899af14adbb634c52ea67d41c112ccd",\n              "fd8c4bd95220b962ba206e5eabe167485bc8b3f3a407d27dbe76baa6bf522899",\n              "da2602d244f1cc4ae1d614c4d51e1aa7f7806d4bd03dfac76cb21b108bb360bf",\n              "f9dbc6fec7958f3200a51a4d4c31e7a177ca7c80685d02dc62b5e8b53de31082",\n              "c31009568bef6b13f9345e605b3783fa6f9d68437019268ca673b152bd19be49",\n              "c3f135a0a0ca48d494dc23ffe076642d00d203f3453b1fa250f0541dca45eab7",\n              "1df9d114d0cae9670b97c2ff36b0975eeeb47d2d72eac309bb14b4629af5b4a8",\n              "50577a61830e56b94433223b2947205fb6e7c2f80942dccf00d360420f1ea7a3",\n              "0e6f437619760f812f033d7d60bd468d02ecc2e9c41be0e21c02fda33f968c37",\n              "60c398f36bebdfa26ea383265240eb970941b5c88e7505e8c8c685b249b0625e"\n            ],\n            "slots": [\n              0,\n              7,\n              14,\n              21,\n              28,\n              35\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  21.8125,\n                  21.5625\n                ],\n                "prediction": "A",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960,\n                "prompt_sha256": "d2537dbb0b1c2b652baf303bc1b8b410abf6e6b26f7a363935d38414ac88129b"\n              },\n              "reversed": {\n                "logits": [\n                  21.984375,\n                  20.609375\n                ],\n                "prediction": "A",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960,\n                "prompt_sha256": "0049fd7060241150d8ddcab600856257969942b4280d6d35e23fbfcf59be80af"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  21.78125,\n                  21.46875\n                ],\n                "prediction": "A",\n                "prompt_sha256": "d2537dbb0b1c2b652baf303bc1b8b410abf6e6b26f7a363935d38414ac88129b",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  21.96875,\n                  20.546875\n                ],\n                "prediction": "A",\n                "prompt_sha256": "0049fd7060241150d8ddcab600856257969942b4280d6d35e23fbfcf59be80af",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              0,\n              6,\n              12,\n              18,\n              24,\n              30,\n              36,\n              42,\n              48,\n              54,\n              60,\n              66,\n              72,\n              264,\n              270,\n              276,\n              282,\n              288,\n              294,\n              300,\n              306,\n              312,\n              318,\n              324,\n              330,\n              336,\n              624,\n              630,\n              636,\n              642,\n              648,\n              654,\n              660,\n              666,\n              672,\n              678,\n              684,\n              690,\n              696\n            ],\n            "hashes": [\n              "386855e29326f38ff54b64c54365c69b0d2f90b523cfe46827e93859e613af78",\n              "816d89ca6f14c83d130ce1d9c30ec1e79110f21231b8f5d13cb14fd730cbbbb9",\n              "62ab5bd96dba21095691187ee6c210b5078c881abde3f42586b5c5446d0bccfd",\n              "c820b7bc9944e80dbe9f6a4f234221b1505f23565c18f715a8eb1d94c0a5685b",\n              "2a260075efbdf566293a2d63a21f3d6abc3c6cbd6f8e014233207c5008e59128",\n              "24d84d9f7f3bf067476ad8b643c1e05a5171bda974e2a24dcafd124aa4e47044",\n              "a8c65073f8694e1844ba8d73d48daec0c4c52c2bc77c034c399528407876bf39",\n              "72b94c90cb80cde68c04d2517fcc09747968d4b114ba8d5377981ddd877d320a",\n              "73e30c70dd62e725fb5f39c0c50fe720e7e709c2fd45303345877e2a22006f2b",\n              "0067e6b44390f0d222c0ac3c8ba131594a2457ccf639768f65616496d8a300ea",\n              "7190c8fd9a46c417db92476df43721f95d0dae204278d225de72751e08d33b26",\n              "ea151c3c7a9dfb8ceb517ba92b0b24960a267a9ef477b004512c74b56011667a",\n              "2cfd832854d3d4fe341f02ceebbc2a2b625636cf832a7f2da96a1bf6dff5ea9c",\n              "0e56aebccefada3d092c9056d1b15dd09ca984a4a819c28b7f348c4ef51223a4",\n              "4ade3f76d8c9e1580a1430a35e0f202220c140dde57007da9317d1637a58535f",\n              "110562f3657aa18f458619fd68b12069c194c92fa879d280eaf95533b99bcb75",\n              "226179ed9c84caf8e95f88a977c8c7aecd8b690d793df779097061e66a301fde",\n              "6151bfc1ed7f98f68056ec12182fdad5e08e1946c32a2e70e097b9ce3dc67367",\n              "ca7099222067d7d22bdad72650ae06999a76c25704152abc32309d39728ec535",\n              "cc99e583714fd1af76c40db7e210fa438ca30390f671205df59ba12ce27914ab",\n              "ef1e0fabd901eae7289229f9fe7eb30d8e430ad8fbf34ce0276aa0b5f2a47eff",\n              "c3f09ebeced95f5e92349bae7ff912161a0bdb5f2899b31327249bc3da0fff57",\n              "a910038cf953fd351f9ce2c6573f55f16bc4cbd41893bc4bb3b186f2c3f2a3a9",\n              "82f2694cc56fbeaf7b6eb9353e05c5077ef07e6a53f1e59eba96d156b59891e5",\n              "98cc8671b44a11e0b63c4029cda405189559bfa028f6f1dd2a1ea513154abede",\n              "e5dffc1f43b0cc7a3c5723a98c9b39157aa0e6c87a3862c7a6b1b6f8beff6294",\n              "70644fd5480fa0c535c092c6bfcf08dd943df14d760e9395d72869a0956de866",\n              "6f388ba1cc90b30db2bc11d98cd7c89aa0936e905811f7caeeb67e4606fa2105",\n              "8b67166209154688438e1634ec5804b3753afa43e249f8a847f036266a7693b0",\n              "2e6c12de848b6e2d7a05e72290372afa6e348c3079785c21bc1f7efc342e7bc8",\n              "e7974c0129cee8f38cf9325db1dfcdf694cf9cff674fbb5e9436c22499ea58e0",\n              "0ae17d33549ca5fc697e280f2367eb7ee58ff86eb5bf4128e2cccd842395cb2e",\n              "699e35542a7fccaa31fe08dd41d213a94f886ea7cdd541164f2f8b6cdf6b2ee8",\n              "b530f6f11c70cf3ccf98c3f3bc3f3af2f56c16507df68f1b2f839a801333286f",\n              "9b8a145d17d0d3d4b1232c26f5b45943750a600242903696c75ba30f2802dfa2",\n              "0169e1124d4ab944b42b49f7a8703a1fc25e44c175ed668d7ee5169aebd37cb0",\n              "ec2850e878d4f2bb2574ba1f9e6415459ffa81e5fcba2114c5de213664fb829c",\n              "1646e003bf3e9aea15c4ea77632da8b76c20d1eda64151461d6687744b782529",\n              "8ef99da522dc5944c9bf271d3fb37149f16f1e294aecd0a5e00a01c1869690a7"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": 0.171875,\n              "yes_no": {\n                "canonical": [\n                  22.125,\n                  22.390625\n                ],\n                "reversed": [\n                  22.625,\n                  22.015625\n                ]\n              }\n            },\n            {\n              "slots": [\n                2,\n                5\n              ],\n              "relevance": -0.953125,\n              "yes_no": {\n                "canonical": [\n                  21.4375,\n                  22.734375\n                ],\n                "reversed": [\n                  21.78125,\n                  22.390625\n                ]\n              }\n            },\n            {\n              "slots": [\n                5,\n                8\n              ],\n              "relevance": -0.515625,\n              "yes_no": {\n                "canonical": [\n                  21.34375,\n                  22.140625\n                ],\n                "reversed": [\n                  21.625,\n                  21.859375\n                ]\n              }\n            },\n            {\n              "slots": [\n                8,\n                11\n              ],\n              "relevance": -0.40625,\n              "yes_no": {\n                "canonical": [\n                  21.359375,\n                  22.09375\n                ],\n                "reversed": [\n                  21.6875,\n                  21.765625\n                ]\n              }\n            },\n            {\n              "slots": [\n                11,\n                14\n              ],\n              "relevance": -0.359375,\n              "yes_no": {\n                "canonical": [\n                  21.296875,\n                  22.0\n                ],\n                "reversed": [\n                  21.640625,\n                  21.65625\n                ]\n              }\n            },\n            {\n              "slots": [\n                14,\n                17\n              ],\n              "relevance": -0.5,\n              "yes_no": {\n                "canonical": [\n                  21.40625,\n                  22.203125\n                ],\n                "reversed": [\n                  21.6875,\n                  21.890625\n                ]\n              }\n            },\n            {\n              "slots": [\n                17,\n                20\n              ],\n              "relevance": -0.5703125,\n              "yes_no": {\n                "canonical": [\n                  21.3125,\n                  22.203125\n                ],\n                "reversed": [\n                  21.625,\n                  21.875\n                ]\n              }\n            },\n            {\n              "slots": [\n                20,\n                23\n              ],\n              "relevance": -0.6328125,\n              "yes_no": {\n                "canonical": [\n                  21.25,\n                  22.203125\n                ],\n                "reversed": [\n                  21.609375,\n                  21.921875\n                ]\n              }\n            },\n            {\n              "slots": [\n                23,\n                26\n              ],\n              "relevance": -0.5859375,\n              "yes_no": {\n                "canonical": [\n                  21.34375,\n                  22.265625\n                ],\n                "reversed": [\n                  21.65625,\n                  21.90625\n                ]\n              }\n            },\n            {\n              "slots": [\n                26,\n                29\n              ],\n              "relevance": -0.5078125,\n              "yes_no": {\n                "canonical": [\n                  21.296875,\n                  22.125\n                ],\n                "reversed": [\n                  21.625,\n                  21.8125\n                ]\n              }\n            },\n            {\n              "slots": [\n                29,\n                32\n              ],\n              "relevance": -0.625,\n              "yes_no": {\n                "canonical": [\n                  21.28125,\n                  22.1875\n                ],\n                "reversed": [\n                  21.59375,\n                  21.9375\n                ]\n              }\n            },\n            {\n              "slots": [\n                32,\n                35\n              ],\n              "relevance": -0.53125,\n              "yes_no": {\n                "canonical": [\n                  21.28125,\n                  22.140625\n                ],\n                "reversed": [\n                  21.625,\n                  21.828125\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                0,\n                3\n              ],\n              [\n                11,\n                14\n              ],\n              [\n                26,\n                29\n              ]\n            ],\n            "constrained_windows": [\n              [\n                2,\n                5\n              ],\n              [\n                11,\n                14\n              ],\n              [\n                26,\n                29\n              ]\n            ],\n            "indices": [\n              66,\n              96,\n              282,\n              318,\n              642,\n              678\n            ],\n            "scanned_indices": 141\n          }\n        }\n      },\n      "media": {\n        "a": {\n          "file": "videos/twinpast_0338/A.mp4",\n          "bytes": 11553424,\n          "sha256": "f9ba9c04ee2b3ddaee8e05ea0a09cf596aea1943204cf70e41e8ccab68ef2b34"\n        },\n        "b": {\n          "file": "videos/twinpast_0338/B.mp4",\n          "bytes": 11528085,\n          "sha256": "33e0c317de9f05651a3640adb419a5ad76a694056022446609db18d0c09c7e82"\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0164",\n      "query_s": 22.041666666666668,\n      "question": "At the end of the close-up of the orange publication on the stone counter, was the publication open or closed?",\n      "options": [\n        "The publication was open.",\n        "The publication was closed."\n      ],\n      "branches": {\n        "a": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480,\n              504,\n              528\n            ],\n            "hashes": [\n              "f937135d6b24e015e4aaeefd08b20fa4337c36c9404e48e0ceb54edb625ce977",\n              "35dcc9b1b976cdfd1f0dc4cc98c67bf5f58a9171094acc6c5e979a70e072ede6",\n              "e099c88e4784d9ebb1a838d7e504010adf2c8a469225536d368cec529fb38f27",\n              "4a7b8611c047e2a34f9d0e9ae857a97d26500798ec57cedd2ff567ce4916bd97",\n              "48937e9f06c79f5f11eebc9ff5263d5a11472fa792f5583ad68da03794f3dfcf",\n              "6494ad4b5781558d03c20962f62fcac938340c2f3ad3f1f9c7adb6ef56b76560",\n              "e9484b82e2fc73684671a7b378b71212b11044583ea3ea122cbf2a4f24d44967",\n              "8ba3b0cde767e2bac9e4baa79bdc8d486c0d893cf898eda3e57d3748d17c85d3",\n              "45e0dc0c84f106fd66d126bd3ee6b8a37e67842260284e4ccf51aba5dee57982",\n              "b10ef77744a56b3ffac9fdaeff4c437b01dc365a0ee03e93ea50468d1e666d95",\n              "2a4d6b7b8f17f82e272631c0e8af6f4afd8dad66f79ef35665408c98680794a4",\n              "790461702a3c2221e42a7f8b8a2011ea2c0377f33088590cafb8b108cf1bec2d",\n              "3190ba3aaff18eee5c4cb76513f809ecf27e2b181484eeaf01350d6bdba4001b",\n              "2c81ba5ca7dab4c0ef47fb4dcbcab2b5b62de1fe58e74e01cd36223654f3b73a",\n              "1f547746ca04e6cfb8d4d9c1b850ce06bd9ed4b7d5836058bb1a0ca41d937b8a",\n              "47862f62e3efa0ffa42a1fb6fb4d30ebe1bbe6b58120b7b28a1261d90e74957d",\n              "bd6779c523c21c47deab4192f7311bf5bb081d9664ec5113df903159c5fa0d30",\n              "5d5a9b2b4b725281c17942c0ad737d1bc10880fce32c708f65401e06f5059abc",\n              "7a376463461448e44d0b7b3c6f2ffab9446cd8e9318c3823a3a09dffcbf5bbfc",\n              "2030dafcc99a761d2e332eb883eed3ea9c70a964cfdfb5553b0a9142ae866d7d",\n              "733ff23ec4f1abc3349db93f551d4e5e40ec1da292879412a0be9b8051ee75d8",\n              "2083b7e586eab4f38ea012fee73d305a8020ccea50a50e22193d1db1c2e4fbea",\n              "5232ec91ca88078f61b35a459dcc29b2f9d8dbeec5e7d6f165e4a8fd164bf588"\n            ],\n            "slots": [\n              0,\n              3,\n              7,\n              10,\n              14,\n              18\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  22.125,\n                  22.265625\n                ],\n                "prediction": "B",\n                "lm_tokens": 2039,\n                "visual_tokens": 1960,\n                "prompt_sha256": "27f27144bc972b676ddf4980d2594c7926a542cdda76897ec122bc2057c356e9"\n              },\n              "reversed": {\n                "logits": [\n                  22.296875,\n                  21.6875\n                ],\n                "prediction": "A",\n                "lm_tokens": 2039,\n                "visual_tokens": 1960,\n                "prompt_sha256": "57f7288e55c47b3339f4ab07dfcb204a7546cc78f397cab4a8e9acc0bd274ba3"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  21.125,\n                  21.640625\n                ],\n                "prediction": "B",\n                "prompt_sha256": "27f27144bc972b676ddf4980d2594c7926a542cdda76897ec122bc2057c356e9",\n                "lm_tokens": 2039,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  21.609375,\n                  21.265625\n                ],\n                "prediction": "A",\n                "prompt_sha256": "57f7288e55c47b3339f4ab07dfcb204a7546cc78f397cab4a8e9acc0bd274ba3",\n                "lm_tokens": 2039,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              96,\n              102,\n              108,\n              114,\n              120,\n              126,\n              132,\n              138,\n              144,\n              150,\n              156,\n              162,\n              168,\n              192,\n              198,\n              204,\n              210,\n              216,\n              222,\n              228,\n              234,\n              240,\n              246,\n              252,\n              258,\n              264,\n              288,\n              294,\n              300,\n              306,\n              312,\n              318,\n              324,\n              330,\n              336,\n              342,\n              348,\n              354,\n              360\n            ],\n            "hashes": [\n              "48937e9f06c79f5f11eebc9ff5263d5a11472fa792f5583ad68da03794f3dfcf",\n              "4439f04191912e438e5ec615aad0b970cf1c1425da90d5b77dd2e1ff5d868b9f",\n              "798c9d53a463101b8ee2d885886cdd24eab6a62f473986546003fac29c0e4fb5",\n              "e6981996e4debc0a201b5eb614b8a20a378ac215d69f6d6701c151ca0e208dbe",\n              "6494ad4b5781558d03c20962f62fcac938340c2f3ad3f1f9c7adb6ef56b76560",\n              "c2da9a4d48f39cd9133d10c55585a2009d3877018162aad16d5f2c57f506d4eb",\n              "c758c3d8b9451a4e048f4d38d07ccb804a3bb785ee46942cf208f556cba68fa5",\n              "0f8087ab1a53bcc545b304ceb9eef75e5c4e70458226a2688d110a31cb44ffe0",\n              "e9484b82e2fc73684671a7b378b71212b11044583ea3ea122cbf2a4f24d44967",\n              "8cb32336d79df03149c15ee6627119132e3a15f4f13d4129fa830bab251b1bf7",\n              "16d02d552f26ca82b0ca97182afaaa6e9b2bfecce19a60ce0c995787e8ffa76d",\n              "096dac6e22c568a6ae792fdddca9e21b87af8a42edbdf88f9ba65b47788b731a",\n              "8ba3b0cde767e2bac9e4baa79bdc8d486c0d893cf898eda3e57d3748d17c85d3",\n              "45e0dc0c84f106fd66d126bd3ee6b8a37e67842260284e4ccf51aba5dee57982",\n              "47a66baa4727bab7d6f431abdc9e5def89e4431d693fd7bcabd7382b07b8ecc4",\n              "bd90fb09ce376adfe1540b1a47e7542aa8958c789738cc720a77fb7f929689f9",\n              "20638e4c31dc97ac61321349a0d3a9a84cc9a0d95c55e1e572b21fbf0d69b2a1",\n              "b10ef77744a56b3ffac9fdaeff4c437b01dc365a0ee03e93ea50468d1e666d95",\n              "6edcf66ff81899a52fece02055790fa3ebd504557990427abf4a931932dbb632",\n              "2e50a3a072ae4b4758b22c548905ce521ee8c0f7dd25cbe0287b86813bba1c57",\n              "799851158e63ab6183cd9ebcd3e53307747ba6e63687b7e6b29554a87517cf07",\n              "2a4d6b7b8f17f82e272631c0e8af6f4afd8dad66f79ef35665408c98680794a4",\n              "1ed0dd0d7a8f667ab676575788e411a2e6fbf272a7907b5ed88c0ac50086a7cd",\n              "1f6e621b6d56282e0595d473082a189602f6faa81ede5c28503b87468552f8f7",\n              "db179a785d54e2ab5f3b0a949779f7e69a253067077d9dcf2a18ed2e35736aab",\n              "790461702a3c2221e42a7f8b8a2011ea2c0377f33088590cafb8b108cf1bec2d",\n              "3190ba3aaff18eee5c4cb76513f809ecf27e2b181484eeaf01350d6bdba4001b",\n              "513eed7c910dde735dbb375aafdf06de48ac620e887cd2b099bbac52098260ac",\n              "7e087e2c9aac68970872e99e2a55e8849b35aa76a5cd1eddff49b498c7e1b9cb",\n              "a86526d3551b7298f4504f4f7d65e06b0724ded5295f15707f2039fcf440f044",\n              "2c81ba5ca7dab4c0ef47fb4dcbcab2b5b62de1fe58e74e01cd36223654f3b73a",\n              "e130f224ed3306ae1dc9bfada7c2fe6ba048904d427730bcbc085387e68464d3",\n              "3ec64d250ffaab51f809be0919cbcacfa1af04f4b56863fef566bdb76b9575e6",\n              "650c24fd3077e173a00fbc95c21c5367d67155581513fbc0a0d55d38a1f9ca81",\n              "1f547746ca04e6cfb8d4d9c1b850ce06bd9ed4b7d5836058bb1a0ca41d937b8a",\n              "bcc1d125f395b759b6315261bc4204220a675eaae9494cb7a8cd358097433726",\n              "c7095a16c70c413aab88f586216987eee23e8288622906bdb662238b52c36894",\n              "69f91ea22362da6a77dcff480bef1c1a92cb5723bc8cefba8a64d39beaab7712",\n              "47862f62e3efa0ffa42a1fb6fb4d30ebe1bbe6b58120b7b28a1261d90e74957d"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": -0.4453125,\n              "yes_no": {\n                "canonical": [\n                  21.875,\n                  22.671875\n                ],\n                "reversed": [\n                  22.0625,\n                  22.15625\n                ]\n              }\n            },\n            {\n              "slots": [\n                1,\n                4\n              ],\n              "relevance": -0.1875,\n              "yes_no": {\n                "canonical": [\n                  21.84375,\n                  22.375\n                ],\n                "reversed": [\n                  22.0625,\n                  21.90625\n                ]\n              }\n            },\n            {\n              "slots": [\n                2,\n                5\n              ],\n              "relevance": -0.125,\n              "yes_no": {\n                "canonical": [\n                  22.09375,\n                  22.5\n                ],\n                "reversed": [\n                  22.296875,\n                  22.140625\n                ]\n              }\n            },\n            {\n              "slots": [\n                4,\n                7\n              ],\n              "relevance": 0.3125,\n              "yes_no": {\n                "canonical": [\n                  22.1875,\n                  22.078125\n                ],\n                "reversed": [\n                  22.40625,\n                  21.890625\n                ]\n              }\n            },\n            {\n              "slots": [\n                5,\n                8\n              ],\n              "relevance": -0.3125,\n              "yes_no": {\n                "canonical": [\n                  21.78125,\n                  22.265625\n                ],\n                "reversed": [\n                  21.765625,\n                  21.90625\n                ]\n              }\n            },\n            {\n              "slots": [\n                6,\n                9\n              ],\n              "relevance": -0.2734375,\n              "yes_no": {\n                "canonical": [\n                  21.796875,\n                  22.234375\n                ],\n                "reversed": [\n                  21.78125,\n                  21.890625\n                ]\n              }\n            },\n            {\n              "slots": [\n                8,\n                11\n              ],\n              "relevance": -0.2265625,\n              "yes_no": {\n                "canonical": [\n                  21.828125,\n                  22.21875\n                ],\n                "reversed": [\n                  21.828125,\n                  21.890625\n                ]\n              }\n            },\n            {\n              "slots": [\n                9,\n                12\n              ],\n              "relevance": -0.2421875,\n              "yes_no": {\n                "canonical": [\n                  21.859375,\n                  22.28125\n                ],\n                "reversed": [\n                  21.890625,\n                  21.953125\n                ]\n              }\n            },\n            {\n              "slots": [\n                10,\n                13\n              ],\n              "relevance": -0.1875,\n              "yes_no": {\n                "canonical": [\n                  21.859375,\n                  22.234375\n                ],\n                "reversed": [\n                  21.921875,\n                  21.921875\n                ]\n              }\n            },\n            {\n              "slots": [\n                12,\n                15\n              ],\n              "relevance": -0.1796875,\n              "yes_no": {\n                "canonical": [\n                  21.890625,\n                  22.203125\n                ],\n                "reversed": [\n                  21.90625,\n                  21.953125\n                ]\n              }\n            },\n            {\n              "slots": [\n                13,\n                16\n              ],\n              "relevance": -0.1875,\n              "yes_no": {\n                "canonical": [\n                  21.875,\n                  22.203125\n                ],\n                "reversed": [\n                  21.90625,\n                  21.953125\n                ]\n              }\n            },\n            {\n              "slots": [\n                15,\n                18\n              ],\n              "relevance": -0.21875,\n              "yes_no": {\n                "canonical": [\n                  21.859375,\n                  22.234375\n                ],\n                "reversed": [\n                  21.90625,\n                  21.96875\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                4,\n                7\n              ],\n              [\n                8,\n                11\n              ],\n              [\n                12,\n                15\n              ]\n            ],\n            "constrained_windows": [\n              [\n                4,\n                7\n              ],\n              [\n                8,\n                11\n              ],\n              [\n                12,\n                15\n              ]\n            ],\n            "indices": [\n              108,\n              150,\n              210,\n              246,\n              306,\n              342\n            ],\n            "scanned_indices": 73\n          }\n        },\n        "b": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480,\n              504,\n              528\n            ],\n            "hashes": [\n              "c401e424a66f8a3c03c2b5377edc3ecf067af9c2c7b35dfcccf30bf6811443c1",\n              "b8e1ac2d16d3c370667ee603117caf8c1e0e7884d04f38f01dab184a6a4e5700",\n              "03b143c20d360610603c147e3d524a6bf34f2b4891fb041ce45c28e8acd45c26",\n              "353e6e8970d1dddd454fbc5b5732504da37b0b8f953f58306d5f9e69358a42e8",\n              "4fa8a4ba78175361caa606cb7dd45d1a3870572eaf1edda8d2464b3be2b811a2",\n              "6494ad4b5781558d03c20962f62fcac938340c2f3ad3f1f9c7adb6ef56b76560",\n              "e9484b82e2fc73684671a7b378b71212b11044583ea3ea122cbf2a4f24d44967",\n              "8ba3b0cde767e2bac9e4baa79bdc8d486c0d893cf898eda3e57d3748d17c85d3",\n              "45e0dc0c84f106fd66d126bd3ee6b8a37e67842260284e4ccf51aba5dee57982",\n              "b10ef77744a56b3ffac9fdaeff4c437b01dc365a0ee03e93ea50468d1e666d95",\n              "2a4d6b7b8f17f82e272631c0e8af6f4afd8dad66f79ef35665408c98680794a4",\n              "790461702a3c2221e42a7f8b8a2011ea2c0377f33088590cafb8b108cf1bec2d",\n              "3190ba3aaff18eee5c4cb76513f809ecf27e2b181484eeaf01350d6bdba4001b",\n              "2c81ba5ca7dab4c0ef47fb4dcbcab2b5b62de1fe58e74e01cd36223654f3b73a",\n              "1f547746ca04e6cfb8d4d9c1b850ce06bd9ed4b7d5836058bb1a0ca41d937b8a",\n              "47862f62e3efa0ffa42a1fb6fb4d30ebe1bbe6b58120b7b28a1261d90e74957d",\n              "bd6779c523c21c47deab4192f7311bf5bb081d9664ec5113df903159c5fa0d30",\n              "5d5a9b2b4b725281c17942c0ad737d1bc10880fce32c708f65401e06f5059abc",\n              "7a376463461448e44d0b7b3c6f2ffab9446cd8e9318c3823a3a09dffcbf5bbfc",\n              "2030dafcc99a761d2e332eb883eed3ea9c70a964cfdfb5553b0a9142ae866d7d",\n              "733ff23ec4f1abc3349db93f551d4e5e40ec1da292879412a0be9b8051ee75d8",\n              "2083b7e586eab4f38ea012fee73d305a8020ccea50a50e22193d1db1c2e4fbea",\n              "5232ec91ca88078f61b35a459dcc29b2f9d8dbeec5e7d6f165e4a8fd164bf588"\n            ],\n            "slots": [\n              0,\n              3,\n              7,\n              10,\n              14,\n              18\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  22.171875,\n                  22.078125\n                ],\n                "prediction": "A",\n                "lm_tokens": 2039,\n                "visual_tokens": 1960,\n                "prompt_sha256": "27f27144bc972b676ddf4980d2594c7926a542cdda76897ec122bc2057c356e9"\n              },\n              "reversed": {\n                "logits": [\n                  22.34375,\n                  21.5\n                ],\n                "prediction": "A",\n                "lm_tokens": 2039,\n                "visual_tokens": 1960,\n                "prompt_sha256": "57f7288e55c47b3339f4ab07dfcb204a7546cc78f397cab4a8e9acc0bd274ba3"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  22.15625,\n                  22.234375\n                ],\n                "prediction": "B",\n                "prompt_sha256": "27f27144bc972b676ddf4980d2594c7926a542cdda76897ec122bc2057c356e9",\n                "lm_tokens": 2039,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  22.546875,\n                  21.65625\n                ],\n                "prediction": "A",\n                "prompt_sha256": "57f7288e55c47b3339f4ab07dfcb204a7546cc78f397cab4a8e9acc0bd274ba3",\n                "lm_tokens": 2039,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              0,\n              6,\n              12,\n              18,\n              24,\n              30,\n              36,\n              42,\n              48,\n              54,\n              60,\n              66,\n              72,\n              96,\n              102,\n              108,\n              114,\n              120,\n              126,\n              132,\n              138,\n              144,\n              150,\n              156,\n              162,\n              168,\n              288,\n              294,\n              300,\n              306,\n              312,\n              318,\n              324,\n              330,\n              336,\n              342,\n              348,\n              354,\n              360\n            ],\n            "hashes": [\n              "c401e424a66f8a3c03c2b5377edc3ecf067af9c2c7b35dfcccf30bf6811443c1",\n              "dee0e44463758bd66d8c99f8beabec237667362b74549738841e3067fdc593eb",\n              "a1801be8793c4a4156971e3a28e3598d21132834cb0123c5f0b52d43b1587beb",\n              "127e8d4cbe9977e50a2e1fbbe3855109c21b233e6b4897a863cfa32eaa36a7fc",\n              "b8e1ac2d16d3c370667ee603117caf8c1e0e7884d04f38f01dab184a6a4e5700",\n              "3f97332559b502e59897a393de76d17287f4ba203e35aee8a70b7799aa4cc8b8",\n              "64753b809cb4f538b2022e727ec6b165f32a68e91dd396c821c8da3b66106027",\n              "635df564531ad804e79ed88f3d1932b7506b89f54b8685e5bd6c37579b3f0583",\n              "03b143c20d360610603c147e3d524a6bf34f2b4891fb041ce45c28e8acd45c26",\n              "cd8f53057d9cda8bd669800c06ac2a4055871d012fd589d86fac97ed5ade85c4",\n              "b036118ad9af9783bafb8510a319973b5be3d4012b9ddb5ec1c0653a208fba1d",\n              "eb2397cd0914c325e9ea75e46141c16a175f8d1ed0aa1272a22edde74f67ca78",\n              "353e6e8970d1dddd454fbc5b5732504da37b0b8f953f58306d5f9e69358a42e8",\n              "4fa8a4ba78175361caa606cb7dd45d1a3870572eaf1edda8d2464b3be2b811a2",\n              "bbd227cea1b3d52e62cea4e55e5b0ca7417d9bcbf81a01d0d1f6d0adf69f090e",\n              "f6c3156f92f62625543ba5cc0518ac6726c7bcedf48bb6c7fd28f88b4bd369f1",\n              "e6981996e4debc0a201b5eb614b8a20a378ac215d69f6d6701c151ca0e208dbe",\n              "6494ad4b5781558d03c20962f62fcac938340c2f3ad3f1f9c7adb6ef56b76560",\n              "c2da9a4d48f39cd9133d10c55585a2009d3877018162aad16d5f2c57f506d4eb",\n              "c758c3d8b9451a4e048f4d38d07ccb804a3bb785ee46942cf208f556cba68fa5",\n              "0f8087ab1a53bcc545b304ceb9eef75e5c4e70458226a2688d110a31cb44ffe0",\n              "e9484b82e2fc73684671a7b378b71212b11044583ea3ea122cbf2a4f24d44967",\n              "8cb32336d79df03149c15ee6627119132e3a15f4f13d4129fa830bab251b1bf7",\n              "16d02d552f26ca82b0ca97182afaaa6e9b2bfecce19a60ce0c995787e8ffa76d",\n              "096dac6e22c568a6ae792fdddca9e21b87af8a42edbdf88f9ba65b47788b731a",\n              "8ba3b0cde767e2bac9e4baa79bdc8d486c0d893cf898eda3e57d3748d17c85d3",\n              "3190ba3aaff18eee5c4cb76513f809ecf27e2b181484eeaf01350d6bdba4001b",\n              "513eed7c910dde735dbb375aafdf06de48ac620e887cd2b099bbac52098260ac",\n              "7e087e2c9aac68970872e99e2a55e8849b35aa76a5cd1eddff49b498c7e1b9cb",\n              "a86526d3551b7298f4504f4f7d65e06b0724ded5295f15707f2039fcf440f044",\n              "2c81ba5ca7dab4c0ef47fb4dcbcab2b5b62de1fe58e74e01cd36223654f3b73a",\n              "e130f224ed3306ae1dc9bfada7c2fe6ba048904d427730bcbc085387e68464d3",\n              "3ec64d250ffaab51f809be0919cbcacfa1af04f4b56863fef566bdb76b9575e6",\n              "650c24fd3077e173a00fbc95c21c5367d67155581513fbc0a0d55d38a1f9ca81",\n              "1f547746ca04e6cfb8d4d9c1b850ce06bd9ed4b7d5836058bb1a0ca41d937b8a",\n              "bcc1d125f395b759b6315261bc4204220a675eaae9494cb7a8cd358097433726",\n              "c7095a16c70c413aab88f586216987eee23e8288622906bdb662238b52c36894",\n              "69f91ea22362da6a77dcff480bef1c1a92cb5723bc8cefba8a64d39beaab7712",\n              "47862f62e3efa0ffa42a1fb6fb4d30ebe1bbe6b58120b7b28a1261d90e74957d"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": 0.1953125,\n              "yes_no": {\n                "canonical": [\n                  22.4375,\n                  22.515625\n                ],\n                "reversed": [\n                  22.765625,\n                  22.296875\n                ]\n              }\n            },\n            {\n              "slots": [\n                1,\n                4\n              ],\n              "relevance": 0.25,\n              "yes_no": {\n                "canonical": [\n                  22.234375,\n                  22.328125\n                ],\n                "reversed": [\n                  22.609375,\n                  22.015625\n                ]\n              }\n            },\n            {\n              "slots": [\n                2,\n                5\n              ],\n              "relevance": -0.2578125,\n              "yes_no": {\n                "canonical": [\n                  21.859375,\n                  22.4375\n                ],\n                "reversed": [\n                  22.140625,\n                  22.078125\n                ]\n              }\n            },\n            {\n              "slots": [\n                4,\n                7\n              ],\n              "relevance": -0.125,\n              "yes_no": {\n                "canonical": [\n                  22.140625,\n                  22.59375\n                ],\n                "reversed": [\n                  22.4375,\n                  22.234375\n                ]\n              }\n            },\n            {\n              "slots": [\n                5,\n                8\n              ],\n              "relevance": -0.3125,\n              "yes_no": {\n                "canonical": [\n                  21.78125,\n                  22.265625\n                ],\n                "reversed": [\n                  21.765625,\n                  21.90625\n                ]\n              }\n            },\n            {\n              "slots": [\n                6,\n                9\n              ],\n              "relevance": -0.2734375,\n              "yes_no": {\n                "canonical": [\n                  21.796875,\n                  22.234375\n                ],\n                "reversed": [\n                  21.78125,\n                  21.890625\n                ]\n              }\n            },\n            {\n              "slots": [\n                8,\n                11\n              ],\n              "relevance": -0.2265625,\n              "yes_no": {\n                "canonical": [\n                  21.828125,\n                  22.21875\n                ],\n                "reversed": [\n                  21.828125,\n                  21.890625\n                ]\n              }\n            },\n            {\n              "slots": [\n                9,\n                12\n              ],\n              "relevance": -0.2421875,\n              "yes_no": {\n                "canonical": [\n                  21.859375,\n                  22.28125\n                ],\n                "reversed": [\n                  21.890625,\n                  21.953125\n                ]\n              }\n            },\n            {\n              "slots": [\n                10,\n                13\n              ],\n              "relevance": -0.1875,\n              "yes_no": {\n                "canonical": [\n                  21.859375,\n                  22.234375\n                ],\n                "reversed": [\n                  21.921875,\n                  21.921875\n                ]\n              }\n            },\n            {\n              "slots": [\n                12,\n                15\n              ],\n              "relevance": -0.1796875,\n              "yes_no": {\n                "canonical": [\n                  21.890625,\n                  22.203125\n                ],\n                "reversed": [\n                  21.90625,\n                  21.953125\n                ]\n              }\n            },\n            {\n              "slots": [\n                13,\n                16\n              ],\n              "relevance": -0.1875,\n              "yes_no": {\n                "canonical": [\n                  21.875,\n                  22.203125\n                ],\n                "reversed": [\n                  21.90625,\n                  21.953125\n                ]\n              }\n            },\n            {\n              "slots": [\n                15,\n                18\n              ],\n              "relevance": -0.21875,\n              "yes_no": {\n                "canonical": [\n                  21.859375,\n                  22.234375\n                ],\n                "reversed": [\n                  21.90625,\n                  21.96875\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                0,\n                3\n              ],\n              [\n                4,\n                7\n              ],\n              [\n                12,\n                15\n              ]\n            ],\n            "constrained_windows": [\n              [\n                0,\n                3\n              ],\n              [\n                4,\n                7\n              ],\n              [\n                12,\n                15\n              ]\n            ],\n            "indices": [\n              18,\n              54,\n              108,\n              150,\n              306,\n              342\n            ],\n            "scanned_indices": 73\n          }\n        }\n      },\n      "media": {\n        "a": {\n          "file": "videos/twinpast_0164/A.mp4",\n          "bytes": 7044647,\n          "sha256": "2aae7b2a1d8717fa50800c33f3345b69f12f100121ba7b742d122514d24e0a82"\n        },\n        "b": {\n          "file": "videos/twinpast_0164/B.mp4",\n          "bytes": 7090016,\n          "sha256": "80db2aca755771b394ba1d72f875663144c80e637d7ccfd8fb35b8605fc87d3d"\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0120",\n      "query_s": 42.958333333333336,\n      "question": "At the end of the box-packing scene, before the video moves on to packing a backpack, what is the visible state of the box top?",\n      "options": [\n        "The top flaps lay flat across the opening, covering the contents.",\n        "The top flaps remained raised and the contents were exposed."\n      ],\n      "branches": {\n        "a": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480,\n              504,\n              528,\n              552,\n              576,\n              600,\n              624,\n              648,\n              672,\n              696,\n              720,\n              744,\n              768,\n              792,\n              816,\n              840,\n              864,\n              888,\n              912,\n              936,\n              960,\n              984,\n              1008\n            ],\n            "hashes": [\n              "dbae5de11fe4336afb4a1868478560724108abf69dda82c50702503f7c9d3ccb",\n              "f53682374bc8a4792c44e1d6432e7a5aec12e288a2dc4b52cf2857c46fb3e4b8",\n              "6bc8c3623431417c1a720d8d51488e5c5f7eb447036bf59f43800ea12680aa2d",\n              "5a416f7182ff57746737ff484a5993152485ce83352ec5eb5bc1521bbaf099d1",\n              "b80814daaf4d102f1b71456cbc9ad49f1a0f48525bae6ba2edcf96cc454c87c7",\n              "b4d8800a0ec9b8c3f6bbac29f1daa9ead6744927478deff55afb48d841007b93",\n              "a26b0cf7efd673608cc0cbdc32d711764c25054411d852d8ceb1138a33191d90",\n              "d50a1929d3e0c9c033867e8b702d2cf03542daf9338f5019bdf79f1f9755c5b0",\n              "a7f88be2525061b1bbb2add5435b3fd2e34f74c30cdaa40285c2e1d475a03a19",\n              "87377fae990fcafb83baa5954a7855a4713cfba04c411efa87a0495ac70d4e65",\n              "4755811a7b8e22184357eae0708dcd23f29e7ba1cf7257a60011c1a7f9e342a1",\n              "20fbf79d73d5d895c3326535b03f810a6849d2dc71afaaaee1817ac2c50c51d9",\n              "32a6e2ca2c5fcceb4a12ab46c3180aa684bd0869ef07ff4f6a231ce2695b5427",\n              "28d240e3079db296bcbf092cf0d4a3080f6684ca357dbfa14500f5c483103c10",\n              "b08de1cb45bd669a76e84c1fc56ca9e6ecce6269caf4e1da239617ec2df5a4d3",\n              "650c97d64ee8e397358ab7809c19cb4b64b843b0b5b6a67c0e125c81d83aca28",\n              "b870ac9cec8c7f6244e9e8ca958e5383fc260a52cb80409f3887be9b8cff7ed4",\n              "a85a0a53f336c62e49d0ac9ef2f495899af05831366d87bad788506466486a2e",\n              "1ccb14f3d7ef85613c4e66d4d8b13cf15ac5b3ccb81d32af2c96d46a3cfa6ebf",\n              "3fc48253d49b2b394dd68521ec221ecc6a07caed855c8dd6410978ce27990913",\n              "3f09591e6a402355c644caee1774b412157797eca17f74da3344a9f70a04191a",\n              "45bd725d64edf693e51000b856e01dc2ccebb246cfacf5848d669d5306ef09c7",\n              "9aa1e0260a093c08c4719974fe38737e0738c465d2a94be714e96cc749e74ea8",\n              "57df8c110367f9e8737dcb40d6b0e0d5086b92a01f6b0f4d15f0ec40685cc48b",\n              "d1eda72cfb20b60bc5bdb35d052dfbe5a858b2dcd4c1522e310c8af7c2de4449",\n              "a0d25150b45cbcd8ff11ad88f94fa62241276c6713a6e738a8d7901b4be4ac4a",\n              "4585e17b99acdbabe582c62bc1d29199da12e00599600bf5bd7bcc33d88661aa",\n              "4fae67a9481634b680f92ce744d588ca63f337c509f840d810a944140cf6dad8",\n              "ceac7fe1a9dcb8d25968f0c670a49b5c4aeb3f1e76d226794f0598df9fdb7b09",\n              "50accf0534fef6631f0fcea7b07880cacd52a9b72b2d5fc999e86d8153d6551d",\n              "c736dddccc83c432269bfa542c322beae15cda58cc516b73034bba544e7a3e55",\n              "f768fdb1625e52a71beced00ba2999073af469709a605d831b36386a7f64298b",\n              "e22f6463626bc79f9025fe79640fa6ea6274a46616f2fe17f5749f4b4254bc30",\n              "6fc07e4d85bda21335b1cc39b6459c3ef0f363488d1958efad222df07e7d22e6",\n              "d80fbd7c25150579c2902c7d9dbefe8f515ab399516bb332fce365c776841771",\n              "596addf18c298caf4b00d54a55ef164770ee9f93d8b7796cc304ea14adab1577",\n              "5c871cfd816e5b104b9cc53105bef19d7435ccea3bb6d13efd3dfc6aca0c5025",\n              "06a0c6e139770b1029ca481d82281c3db7d846346671d9f5696b71542690cee6",\n              "ee933af39df2ca720a214a2d2a0d3b2d88032d0b2c391ba9d6a274b64d714375",\n              "98ea07a9d39f208e8a21650d42079d8e931952a97e3d72702912121a5e86b21e",\n              "75737b3cdaa341c803e38be375d4413508360d9e7e96b8bbfb65e29d0f036633",\n              "ab5426fdf5526a76465a76dabea1088ffffbda13d17c5115f2bf4dd15415c3bf",\n              "54f26b680b6bfb7b31e37ba09e3e0eee97a6d888840c10b6e83f947bee2d46ce"\n            ],\n            "slots": [\n              0,\n              7,\n              15,\n              22,\n              30,\n              38\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  20.875,\n                  21.859375\n                ],\n                "prediction": "B",\n                "lm_tokens": 2062,\n                "visual_tokens": 1960,\n                "prompt_sha256": "ab6b493a0b922e9dffb71cb7a5a724993ff219038d87a6b4c312f496f2366724"\n              },\n              "reversed": {\n                "logits": [\n                  21.515625,\n                  21.1875\n                ],\n                "prediction": "A",\n                "lm_tokens": 2062,\n                "visual_tokens": 1960,\n                "prompt_sha256": "375abbbd647b8fdd93e91c24bde95328101491168b8562b3bb543f808db3b10f"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  21.203125,\n                  22.078125\n                ],\n                "prediction": "B",\n                "prompt_sha256": "ab6b493a0b922e9dffb71cb7a5a724993ff219038d87a6b4c312f496f2366724",\n                "lm_tokens": 2062,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  21.875,\n                  21.5\n                ],\n                "prediction": "A",\n                "prompt_sha256": "375abbbd647b8fdd93e91c24bde95328101491168b8562b3bb543f808db3b10f",\n                "lm_tokens": 2062,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              0,\n              6,\n              12,\n              18,\n              24,\n              30,\n              36,\n              42,\n              48,\n              54,\n              60,\n              66,\n              72,\n              144,\n              150,\n              156,\n              162,\n              168,\n              174,\n              180,\n              186,\n              192,\n              198,\n              204,\n              210,\n              216,\n              528,\n              534,\n              540,\n              546,\n              552,\n              558,\n              564,\n              570,\n              576,\n              582,\n              588,\n              594,\n              600\n            ],\n            "hashes": [\n              "dbae5de11fe4336afb4a1868478560724108abf69dda82c50702503f7c9d3ccb",\n              "4afd6f01aec1d44fe6366d65e23c8cdd1991cca78461cf721d410530f5ed3500",\n              "548e9eba6ccb598b9dc17dbeefd71d5a1b0a27dd620bdb1425885dbd005251aa",\n              "b3e092e78e17fd5d75f1c5d4cffdfa22bafaf339d0180ad179e3d2aa97861f75",\n              "f53682374bc8a4792c44e1d6432e7a5aec12e288a2dc4b52cf2857c46fb3e4b8",\n              "761bafa0fcc2d620945add5e82b23708c7457515bf15491d3bfdb58e180f609b",\n              "b27d50c069b8e5e563c8fcec66aba1ca6f07fdb630741405c015fe2d34b796e1",\n              "48911da89a8f079d97079b9ca0ccbb722c0b2646e94a5976899bce1ce7a4e5d6",\n              "6bc8c3623431417c1a720d8d51488e5c5f7eb447036bf59f43800ea12680aa2d",\n              "aeb710046dd20e003508b51ddf7deb489b6bf5df403e0e77b17002de73f1ff95",\n              "7f86134d91f707294b6fbf92fcbf12591b1bd46e8c2f4451b8474f9fab328787",\n              "88a515d80f0f777ee08e907b39abc63549e407a1eec3db1ed04fccb844bed454",\n              "5a416f7182ff57746737ff484a5993152485ce83352ec5eb5bc1521bbaf099d1",\n              "a26b0cf7efd673608cc0cbdc32d711764c25054411d852d8ceb1138a33191d90",\n              "554c3899563daefd199efbc17bdf2c8ced031cedc337676580e82c2fcd9fd657",\n              "6e93faa06495e304d49599dd986b863a654a6d481dede0109771dc966a444ccb",\n              "589689db30a1edcfa93e532f78bedff475d6d91a1675b58a0b74055c975c4e76",\n              "d50a1929d3e0c9c033867e8b702d2cf03542daf9338f5019bdf79f1f9755c5b0",\n              "8191e511b34034c1d38506ee1f0d758d233b7b8467646bc92400230c4bda791e",\n              "8b134c5d2ec39979c2ad28c1c8a5b4afed58257ec576f84e2cd5ea027ff92ba0",\n              "26ffa2b1ceb37bbf7e44148d059fbeeda4d4d71712415246560deaf1654ce61c",\n              "a7f88be2525061b1bbb2add5435b3fd2e34f74c30cdaa40285c2e1d475a03a19",\n              "874729f26cc275afb0809d54b039955f144d2d1f85b6884bd5aa7d214cb58eb9",\n              "577d86f505e420225f74552a1e097422e85f08a94d0dba74d1f9e656cdbd958b",\n              "207fc5a915f7e9465b5942c6ee583f1c4415e49fe1ec116c28321d5f68c78dbe",\n              "87377fae990fcafb83baa5954a7855a4713cfba04c411efa87a0495ac70d4e65",\n              "9aa1e0260a093c08c4719974fe38737e0738c465d2a94be714e96cc749e74ea8",\n              "7f895bd0bce35845e90d274f0824e5b7aea28b75d131864ca5d1e3a8a9e70afa",\n              "5e1c7a201c716bb562c62a0059f1143c0b726ef09337806112bbc316386612d8",\n              "bb0c50709ba9042da43df2105a9a79191464a73413717760352ffdf8a123ab19",\n              "57df8c110367f9e8737dcb40d6b0e0d5086b92a01f6b0f4d15f0ec40685cc48b",\n              "92e6f8f9e02b92910993f806a138dc4f594d9910d931845ecc6b1de849db3f98",\n              "33704b05bddb01d6c2439f53f118ca6047928d1e57a2cc0f3ec717a51307b560",\n              "da187ca5db6faf3cb59b5c28e48c26ba13d71585848b6412f5abbf05b9a36621",\n              "d1eda72cfb20b60bc5bdb35d052dfbe5a858b2dcd4c1522e310c8af7c2de4449",\n              "39cc3c07a0c744d9441a8e8d7b631148191829ee987193345c8e9fa1cd4275af",\n              "0d2464de565587e3e58533ef06d0c836d61def95f3a11caade82fa8bbee431a6",\n              "2c978b116c6b7b6f613203eefd07918e9c274aeb59dd49358a9e8f53739832eb",\n              "a0d25150b45cbcd8ff11ad88f94fa62241276c6713a6e738a8d7901b4be4ac4a"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": -0.0703125,\n              "yes_no": {\n                "canonical": [\n                  22.0625,\n                  22.3125\n                ],\n                "reversed": [\n                  21.78125,\n                  21.671875\n                ]\n              }\n            },\n            {\n              "slots": [\n                3,\n                6\n              ],\n              "relevance": 0.1796875,\n              "yes_no": {\n                "canonical": [\n                  22.1875,\n                  22.15625\n                ],\n                "reversed": [\n                  21.90625,\n                  21.578125\n                ]\n              }\n            },\n            {\n              "slots": [\n                6,\n                9\n              ],\n              "relevance": 0.2890625,\n              "yes_no": {\n                "canonical": [\n                  22.203125,\n                  22.0625\n                ],\n                "reversed": [\n                  22.03125,\n                  21.59375\n                ]\n              }\n            },\n            {\n              "slots": [\n                9,\n                12\n              ],\n              "relevance": -0.1015625,\n              "yes_no": {\n                "canonical": [\n                  21.9375,\n                  22.28125\n                ],\n                "reversed": [\n                  21.703125,\n                  21.5625\n                ]\n              }\n            },\n            {\n              "slots": [\n                12,\n                15\n              ],\n              "relevance": -0.2109375,\n              "yes_no": {\n                "canonical": [\n                  21.890625,\n                  22.375\n                ],\n                "reversed": [\n                  21.671875,\n                  21.609375\n                ]\n              }\n            },\n            {\n              "slots": [\n                15,\n                18\n              ],\n              "relevance": -0.234375,\n              "yes_no": {\n                "canonical": [\n                  21.890625,\n                  22.4375\n                ],\n                "reversed": [\n                  21.671875,\n                  21.59375\n                ]\n              }\n            },\n            {\n              "slots": [\n                19,\n                22\n              ],\n              "relevance": -0.234375,\n              "yes_no": {\n                "canonical": [\n                  21.890625,\n                  22.453125\n                ],\n                "reversed": [\n                  21.703125,\n                  21.609375\n                ]\n              }\n            },\n            {\n              "slots": [\n                22,\n                25\n              ],\n              "relevance": -0.1015625,\n              "yes_no": {\n                "canonical": [\n                  21.875,\n                  22.265625\n                ],\n                "reversed": [\n                  21.71875,\n                  21.53125\n                ]\n              }\n            },\n            {\n              "slots": [\n                25,\n                28\n              ],\n              "relevance": -0.25,\n              "yes_no": {\n                "canonical": [\n                  21.96875,\n                  22.5\n                ],\n                "reversed": [\n                  21.703125,\n                  21.671875\n                ]\n              }\n            },\n            {\n              "slots": [\n                28,\n                31\n              ],\n              "relevance": -0.1328125,\n              "yes_no": {\n                "canonical": [\n                  22.078125,\n                  22.46875\n                ],\n                "reversed": [\n                  21.8125,\n                  21.6875\n                ]\n              }\n            },\n            {\n              "slots": [\n                31,\n                34\n              ],\n              "relevance": -0.3828125,\n              "yes_no": {\n                "canonical": [\n                  21.671875,\n                  22.40625\n                ],\n                "reversed": [\n                  21.625,\n                  21.65625\n                ]\n              }\n            },\n            {\n              "slots": [\n                35,\n                38\n              ],\n              "relevance": -0.171875,\n              "yes_no": {\n                "canonical": [\n                  21.96875,\n                  22.421875\n                ],\n                "reversed": [\n                  21.75,\n                  21.640625\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                0,\n                3\n              ],\n              [\n                6,\n                9\n              ],\n              [\n                22,\n                25\n              ]\n            ],\n            "constrained_windows": [\n              [\n                0,\n                3\n              ],\n              [\n                6,\n                9\n              ],\n              [\n                22,\n                25\n              ]\n            ],\n            "indices": [\n              18,\n              54,\n              162,\n              186,\n              546,\n              582\n            ],\n            "scanned_indices": 147\n          }\n        },\n        "b": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480,\n              504,\n              528,\n              552,\n              576,\n              600,\n              624,\n              648,\n              672,\n              696,\n              720,\n              744,\n              768,\n              792,\n              816,\n              840,\n              864,\n              888,\n              912,\n              936,\n              960,\n              984,\n              1008\n            ],\n            "hashes": [\n              "4e40a18b9f85816df76f7d24056c909fcc90c081bddc6bbcbb8e049cf3eb69ec",\n              "74afc619976a38febfaad4476d9de346b63ec5101645c10c8040e95c6f252a3d",\n              "aee5b50fb1709588a122c331667ffe6d9fee2bda417c8731909d4b4bd72d99b8",\n              "8aee29f35afddee054440281805090d31975396c167a12a3707cc93885c6ecfc",\n              "3b61cdc6cd5fb2e9217a034500b2b958cba12ca8ee9757efd4165209506cbdd4",\n              "154b15086b088ac1690b8c0b5ba5c23fccb4a842fadf1803e8bf05a3d61be84b",\n              "d6057f919b8c95a6da7b978445dd86fe63b95948ee74316944ef7cc78851813b",\n              "20546aa3c39d1693b3da9c66255c4df89b82a097b2e4e9efcd6e55264c40caaf",\n              "a7f88be2525061b1bbb2add5435b3fd2e34f74c30cdaa40285c2e1d475a03a19",\n              "87377fae990fcafb83baa5954a7855a4713cfba04c411efa87a0495ac70d4e65",\n              "4755811a7b8e22184357eae0708dcd23f29e7ba1cf7257a60011c1a7f9e342a1",\n              "20fbf79d73d5d895c3326535b03f810a6849d2dc71afaaaee1817ac2c50c51d9",\n              "32a6e2ca2c5fcceb4a12ab46c3180aa684bd0869ef07ff4f6a231ce2695b5427",\n              "28d240e3079db296bcbf092cf0d4a3080f6684ca357dbfa14500f5c483103c10",\n              "b08de1cb45bd669a76e84c1fc56ca9e6ecce6269caf4e1da239617ec2df5a4d3",\n              "650c97d64ee8e397358ab7809c19cb4b64b843b0b5b6a67c0e125c81d83aca28",\n              "b870ac9cec8c7f6244e9e8ca958e5383fc260a52cb80409f3887be9b8cff7ed4",\n              "a85a0a53f336c62e49d0ac9ef2f495899af05831366d87bad788506466486a2e",\n              "1ccb14f3d7ef85613c4e66d4d8b13cf15ac5b3ccb81d32af2c96d46a3cfa6ebf",\n              "3fc48253d49b2b394dd68521ec221ecc6a07caed855c8dd6410978ce27990913",\n              "3f09591e6a402355c644caee1774b412157797eca17f74da3344a9f70a04191a",\n              "45bd725d64edf693e51000b856e01dc2ccebb246cfacf5848d669d5306ef09c7",\n              "9aa1e0260a093c08c4719974fe38737e0738c465d2a94be714e96cc749e74ea8",\n              "57df8c110367f9e8737dcb40d6b0e0d5086b92a01f6b0f4d15f0ec40685cc48b",\n              "d1eda72cfb20b60bc5bdb35d052dfbe5a858b2dcd4c1522e310c8af7c2de4449",\n              "a0d25150b45cbcd8ff11ad88f94fa62241276c6713a6e738a8d7901b4be4ac4a",\n              "4585e17b99acdbabe582c62bc1d29199da12e00599600bf5bd7bcc33d88661aa",\n              "4fae67a9481634b680f92ce744d588ca63f337c509f840d810a944140cf6dad8",\n              "ceac7fe1a9dcb8d25968f0c670a49b5c4aeb3f1e76d226794f0598df9fdb7b09",\n              "50accf0534fef6631f0fcea7b07880cacd52a9b72b2d5fc999e86d8153d6551d",\n              "c736dddccc83c432269bfa542c322beae15cda58cc516b73034bba544e7a3e55",\n              "f768fdb1625e52a71beced00ba2999073af469709a605d831b36386a7f64298b",\n              "e22f6463626bc79f9025fe79640fa6ea6274a46616f2fe17f5749f4b4254bc30",\n              "6fc07e4d85bda21335b1cc39b6459c3ef0f363488d1958efad222df07e7d22e6",\n              "d80fbd7c25150579c2902c7d9dbefe8f515ab399516bb332fce365c776841771",\n              "596addf18c298caf4b00d54a55ef164770ee9f93d8b7796cc304ea14adab1577",\n              "5c871cfd816e5b104b9cc53105bef19d7435ccea3bb6d13efd3dfc6aca0c5025",\n              "06a0c6e139770b1029ca481d82281c3db7d846346671d9f5696b71542690cee6",\n              "ee933af39df2ca720a214a2d2a0d3b2d88032d0b2c391ba9d6a274b64d714375",\n              "98ea07a9d39f208e8a21650d42079d8e931952a97e3d72702912121a5e86b21e",\n              "75737b3cdaa341c803e38be375d4413508360d9e7e96b8bbfb65e29d0f036633",\n              "ab5426fdf5526a76465a76dabea1088ffffbda13d17c5115f2bf4dd15415c3bf",\n              "54f26b680b6bfb7b31e37ba09e3e0eee97a6d888840c10b6e83f947bee2d46ce"\n            ],\n            "slots": [\n              0,\n              7,\n              15,\n              22,\n              30,\n              38\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  21.015625,\n                  21.90625\n                ],\n                "prediction": "B",\n                "lm_tokens": 2062,\n                "visual_tokens": 1960,\n                "prompt_sha256": "ab6b493a0b922e9dffb71cb7a5a724993ff219038d87a6b4c312f496f2366724"\n              },\n              "reversed": {\n                "logits": [\n                  21.546875,\n                  21.546875\n                ],\n                "prediction": "A",\n                "lm_tokens": 2062,\n                "visual_tokens": 1960,\n                "prompt_sha256": "375abbbd647b8fdd93e91c24bde95328101491168b8562b3bb543f808db3b10f"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  21.328125,\n                  21.46875\n                ],\n                "prediction": "B",\n                "prompt_sha256": "ab6b493a0b922e9dffb71cb7a5a724993ff219038d87a6b4c312f496f2366724",\n                "lm_tokens": 2062,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  21.4375,\n                  20.921875\n                ],\n                "prediction": "A",\n                "prompt_sha256": "375abbbd647b8fdd93e91c24bde95328101491168b8562b3bb543f808db3b10f",\n                "lm_tokens": 2062,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              216,\n              222,\n              228,\n              234,\n              240,\n              246,\n              252,\n              258,\n              264,\n              270,\n              276,\n              282,\n              288,\n              528,\n              534,\n              540,\n              546,\n              552,\n              558,\n              564,\n              570,\n              576,\n              582,\n              588,\n              594,\n              600,\n              672,\n              678,\n              684,\n              690,\n              696,\n              702,\n              708,\n              714,\n              720,\n              726,\n              732,\n              738,\n              744\n            ],\n            "hashes": [\n              "87377fae990fcafb83baa5954a7855a4713cfba04c411efa87a0495ac70d4e65",\n              "214da728a8b7d3f979e9a38df192bd78b0a0e341ae9c1d46c2fc053abd0e9f90",\n              "f419653d6e56f6fe26fbd92098340f1ec0cf00bf9652178ceea12803128dd0eb",\n              "ba025dc51441e2e5986c82cbabbc9220ec847068ed7ecfbdf69f762e6cc93af2",\n              "4755811a7b8e22184357eae0708dcd23f29e7ba1cf7257a60011c1a7f9e342a1",\n              "fe1ae70194270e28ae3b0c5e4cdc7c9f316298ff768eceab8c3881eff37ca8de",\n              "6721e4099cedaa10cb360acb9af17e6d9f5930fbdbf5bb899dffe5bbeeaf7f4c",\n              "bbf9da1293606245b882ae76a027b4c4ea3f3842364f1746a946cc547db3a641",\n              "20fbf79d73d5d895c3326535b03f810a6849d2dc71afaaaee1817ac2c50c51d9",\n              "6059b6a556433bb8fc30eb931985dc4e0551b3ce8d073b130466e84452d338dd",\n              "00b5bac95d90783a4c4e5352874bf8729d4efd7e3b59a1bb71e9e4c097b6f35b",\n              "9ec6539fcedc4391f35147e3bb7a36d8ea06c8d90d68aeedda3cf6add4d99248",\n              "32a6e2ca2c5fcceb4a12ab46c3180aa684bd0869ef07ff4f6a231ce2695b5427",\n              "9aa1e0260a093c08c4719974fe38737e0738c465d2a94be714e96cc749e74ea8",\n              "7f895bd0bce35845e90d274f0824e5b7aea28b75d131864ca5d1e3a8a9e70afa",\n              "5e1c7a201c716bb562c62a0059f1143c0b726ef09337806112bbc316386612d8",\n              "bb0c50709ba9042da43df2105a9a79191464a73413717760352ffdf8a123ab19",\n              "57df8c110367f9e8737dcb40d6b0e0d5086b92a01f6b0f4d15f0ec40685cc48b",\n              "92e6f8f9e02b92910993f806a138dc4f594d9910d931845ecc6b1de849db3f98",\n              "33704b05bddb01d6c2439f53f118ca6047928d1e57a2cc0f3ec717a51307b560",\n              "da187ca5db6faf3cb59b5c28e48c26ba13d71585848b6412f5abbf05b9a36621",\n              "d1eda72cfb20b60bc5bdb35d052dfbe5a858b2dcd4c1522e310c8af7c2de4449",\n              "39cc3c07a0c744d9441a8e8d7b631148191829ee987193345c8e9fa1cd4275af",\n              "0d2464de565587e3e58533ef06d0c836d61def95f3a11caade82fa8bbee431a6",\n              "2c978b116c6b7b6f613203eefd07918e9c274aeb59dd49358a9e8f53739832eb",\n              "a0d25150b45cbcd8ff11ad88f94fa62241276c6713a6e738a8d7901b4be4ac4a",\n              "ceac7fe1a9dcb8d25968f0c670a49b5c4aeb3f1e76d226794f0598df9fdb7b09",\n              "244973f9cd7c177a2f1e5242b44d0bdd90dc851c5238d3ef5ee9e0594d056350",\n              "3fa3a50bbc593087a5cdebbe4cbd566580283f6dba3311ff321def529a81afcc",\n              "4cfe1fc2544183970e68c733f1092ac70edfe325a6db1799b66824222d05f6ca",\n              "50accf0534fef6631f0fcea7b07880cacd52a9b72b2d5fc999e86d8153d6551d",\n              "0fd193b7ac08f76a7c462dac03e2089fe40413300098966163074714dbc1e9f8",\n              "9b97c245820c8eb76998a3a0d35fa536a40b80acc274580429dac11f9e777001",\n              "83a1d0a1077b43b6d7ed723fe656f9220e696d4f87215c74a1b601c68728b2d7",\n              "c736dddccc83c432269bfa542c322beae15cda58cc516b73034bba544e7a3e55",\n              "7042544857354a78fc220ac498f43cab0176a8d0b217f4f58cfa1f81cad15570",\n              "ea71f17301d06d498505c1428519ccc16d89d804d043c7bb914900648489106c",\n              "4755725bb3fff36a78e8f58f652f6887b339d9f07b1933ed2143f70973583143",\n              "f768fdb1625e52a71beced00ba2999073af469709a605d831b36386a7f64298b"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": -0.3515625,\n              "yes_no": {\n                "canonical": [\n                  22.09375,\n                  22.65625\n                ],\n                "reversed": [\n                  21.796875,\n                  21.9375\n                ]\n              }\n            },\n            {\n              "slots": [\n                3,\n                6\n              ],\n              "relevance": -0.625,\n              "yes_no": {\n                "canonical": [\n                  21.921875,\n                  22.890625\n                ],\n                "reversed": [\n                  21.75,\n                  22.03125\n                ]\n              }\n            },\n            {\n              "slots": [\n                6,\n                9\n              ],\n              "relevance": -0.3046875,\n              "yes_no": {\n                "canonical": [\n                  21.921875,\n                  22.609375\n                ],\n                "reversed": [\n                  21.875,\n                  21.796875\n                ]\n              }\n            },\n            {\n              "slots": [\n                9,\n                12\n              ],\n              "relevance": -0.1015625,\n              "yes_no": {\n                "canonical": [\n                  21.9375,\n                  22.28125\n                ],\n                "reversed": [\n                  21.703125,\n                  21.5625\n                ]\n              }\n            },\n            {\n              "slots": [\n                12,\n                15\n              ],\n              "relevance": -0.2109375,\n              "yes_no": {\n                "canonical": [\n                  21.890625,\n                  22.375\n                ],\n                "reversed": [\n                  21.671875,\n                  21.609375\n                ]\n              }\n            },\n            {\n              "slots": [\n                15,\n                18\n              ],\n              "relevance": -0.234375,\n              "yes_no": {\n                "canonical": [\n                  21.890625,\n                  22.4375\n                ],\n                "reversed": [\n                  21.671875,\n                  21.59375\n                ]\n              }\n            },\n            {\n              "slots": [\n                19,\n                22\n              ],\n              "relevance": -0.234375,\n              "yes_no": {\n                "canonical": [\n                  21.890625,\n                  22.453125\n                ],\n                "reversed": [\n                  21.703125,\n                  21.609375\n                ]\n              }\n            },\n            {\n              "slots": [\n                22,\n                25\n              ],\n              "relevance": -0.1015625,\n              "yes_no": {\n                "canonical": [\n                  21.875,\n                  22.265625\n                ],\n                "reversed": [\n                  21.71875,\n                  21.53125\n                ]\n              }\n            },\n            {\n              "slots": [\n                25,\n                28\n              ],\n              "relevance": -0.25,\n              "yes_no": {\n                "canonical": [\n                  21.96875,\n                  22.5\n                ],\n                "reversed": [\n                  21.703125,\n                  21.671875\n                ]\n              }\n            },\n            {\n              "slots": [\n                28,\n                31\n              ],\n              "relevance": -0.1328125,\n              "yes_no": {\n                "canonical": [\n                  22.078125,\n                  22.46875\n                ],\n                "reversed": [\n                  21.8125,\n                  21.6875\n                ]\n              }\n            },\n            {\n              "slots": [\n                31,\n                34\n              ],\n              "relevance": -0.3828125,\n              "yes_no": {\n                "canonical": [\n                  21.671875,\n                  22.40625\n                ],\n                "reversed": [\n                  21.625,\n                  21.65625\n                ]\n              }\n            },\n            {\n              "slots": [\n                35,\n                38\n              ],\n              "relevance": -0.171875,\n              "yes_no": {\n                "canonical": [\n                  21.96875,\n                  22.421875\n                ],\n                "reversed": [\n                  21.75,\n                  21.640625\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                9,\n                12\n              ],\n              [\n                22,\n                25\n              ],\n              [\n                28,\n                31\n              ]\n            ],\n            "constrained_windows": [\n              [\n                6,\n                9\n              ],\n              [\n                22,\n                25\n              ],\n              [\n                28,\n                31\n              ]\n            ],\n            "indices": [\n              162,\n              186,\n              546,\n              582,\n              690,\n              726\n            ],\n            "scanned_indices": 147\n          }\n        }\n      },\n      "media": {\n        "a": {\n          "file": "videos/twinpast_0120/A.mp4",\n          "bytes": 25283725,\n          "sha256": "9bcbd0ff510845f5385ecb9aecd0e63f160fb3f70619dbc57dc7a9acb8fd70b3"\n        },\n        "b": {\n          "file": "videos/twinpast_0120/B.mp4",\n          "bytes": 26609694,\n          "sha256": "1f701a8e5746e67c3aa6abded420da371339d069998cd8e743f856a96f8cc199"\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0085",\n      "query_s": 39.041666666666664,\n      "question": "At the end of the earlier bedding segment, immediately before the scene changes, was the white pillow exposed or covered by the gray sheet?",\n      "options": [\n        "Covered by the gray sheet.",\n        "Exposed beside the gray sheet."\n      ],\n      "branches": {\n        "a": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480,\n              504,\n              528,\n              552,\n              576,\n              600,\n              624,\n              648,\n              672,\n              696,\n              720,\n              744,\n              768,\n              792,\n              816,\n              840,\n              864,\n              888,\n              912,\n              936\n            ],\n            "hashes": [\n              "4abd2f0f9678cede8f56514b8c5357787b173e6973e64afe2d7d045c78de01f3",\n              "eaf94b74172b97e7456b7131a02a06c480df0be3586c02bb63d889c6b235a9ca",\n              "2ec9c6d6a749555997488a00d0a8ef171c5f93e5cb822cea6f8f6821e86827d7",\n              "0031a4b84dceca5517b2650d53cdd9dc4275647a1f5ff150ad24ac6cdcfe5bc4",\n              "7433a3f69f5bb5289fd7b40a7538292544a6f7aef99f3ba521b3ef4e35e3e067",\n              "f511bb7e84361dfabc2195f403813ebf6ea6255af4c460a83f786e68b390c8f7",\n              "98ba1c29f299c948cf03ba5f01666a555330f0fa7f92187c500e953044173b2c",\n              "30dbfeb35536d8df2c7d0506a3bb369e6c3e17435316409953c069dc6af66d4d",\n              "ab7b1324535423b10b4ace3a1b73e0eb9335591d334a7ae05a4e7eec0b93a2ac",\n              "d6698dde30a26a1171b37f207bc8829ea6301fd85fcde21118570e13615615e0",\n              "6b5e69e3266b3e88271b5b1dcccef8f949d16247ea2ec34e30372731fd22c594",\n              "2687beaa880114ae0349c8975f78907a296b65d0e50fea43b81d69e0630679ff",\n              "726f608ef89e415b0f9939fc0f3f8fa50f46527c5ee0d20317b4348eed6dc1c3",\n              "46960ebd2da846b31b63febb221ab2dfacab7d85e25f651818f539720b80fff3",\n              "30a260c6df1233200b0a307f26cb76aadfe8cab496f76dc251715212ca6e4f54",\n              "eb7b4f720aacbe60f214aa087d60bdda90de07f0f0f26a2fda38ab14faf2f457",\n              "9c4d53dbd00f9686e3cc1c1b883bb6ab74de7d9668d086fdc0b6a0e8c2c99e73",\n              "0ed15455e0e1686bd7d2f9f6202a126c371964acee78a6a9742bff8a135890ab",\n              "e7d350b5e8fc2a4447b32e26c8edeb30d8fb57c153370543c62db97059206665",\n              "ba36a1527bb5f4526676b4283b9874d55913aff69d5c10112034958575707280",\n              "34549fc874e70bd56bb1c95855c8b34718b3bd208ce29e81fef876f836a1666e",\n              "6a3ef476afc0741535b2613c1586c44b9add31c42f6920f08fb8687a31413265",\n              "6c82f57f57df92fc5565487f961abc2c759be335da04e59d217fbb2e427202a6",\n              "5de57aa7c0a7dc330c7310a2d9532af2f0d5da4e33dc8046053bf74740e371f8",\n              "7a957f7e381b62d35faa5997d8939c4bf388a7e261544b4f2885195be6efd03a",\n              "e55858172070c0728b09758408e88d8dc4e30e6814d784588b2b1d11ad00e356",\n              "ab6461a5ab3cc02dbd1d661bb43d2bf1f6bf6ac01f3bdb868b1ab159ca440054",\n              "95a20fcc8fbe37c3b56ca7e64424b23ac6f0e5d214c7133585199d5f42867578",\n              "fda1639005e1477b806be1f412e0783abd522757085f63ad95bec2e6f5b225cb",\n              "ebe907fcc52174db70868cc8f3981116f3ed43cfc8ab1822849027acf1985438",\n              "ef2677c31e4b4f0c93cf83bb2bcdeca3060f760f6f9747b544ccab83ec1952af",\n              "a3775688936ec1a72e10009993f311177cbbb596a8704fffcd3fb8383e8f8828",\n              "7d9277c63a3e9af772436943575f091011ad61c00415e3795cb17f561edb0d57",\n              "4a416955343f16824c6c84d5ddbafec7d2ba8ac9e6a48ddbb34614560f11ec97",\n              "d95879d9f127024629f020dc1b379b1dfa64d13475297bd582e240ccfaeb2d55",\n              "d42058cac3f4485516809d7e43d41561f6bf5026a09dd2a141b4a817dcefb768",\n              "1ebd320ccf92650088fafb4dea9dbe4deec08caf31e07bec368ff222c52af0d9",\n              "ecf2155c824011556c8bb72987f1f5d9d6de490564627a2318055ffc75749ae5",\n              "4f9a17077361e1fc3aff550f12232f44dd95df6df8578bf9336b00570af1b385",\n              "0be5eac53cdbd5fdb55a7ea303998b666724277e037a6ebd91e35f5839bc0b68"\n            ],\n            "slots": [\n              0,\n              7,\n              14,\n              21,\n              28,\n              35\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  21.59375,\n                  22.328125\n                ],\n                "prediction": "B",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960,\n                "prompt_sha256": "1514452f426d2625fe36b67bf70434b83452c0ddacbbb70cc9f489c0ff504c39"\n              },\n              "reversed": {\n                "logits": [\n                  21.75,\n                  22.171875\n                ],\n                "prediction": "B",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960,\n                "prompt_sha256": "62b3987507ecd683d9059cb67ca4421bfb9cb52eefa80c8f7687ee251ea363be"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  22.046875,\n                  21.59375\n                ],\n                "prediction": "A",\n                "prompt_sha256": "1514452f426d2625fe36b67bf70434b83452c0ddacbbb70cc9f489c0ff504c39",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  21.75,\n                  21.6875\n                ],\n                "prediction": "A",\n                "prompt_sha256": "62b3987507ecd683d9059cb67ca4421bfb9cb52eefa80c8f7687ee251ea363be",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              48,\n              54,\n              60,\n              66,\n              72,\n              78,\n              84,\n              90,\n              96,\n              102,\n              108,\n              114,\n              120,\n              336,\n              342,\n              348,\n              354,\n              360,\n              366,\n              372,\n              378,\n              384,\n              390,\n              396,\n              402,\n              408,\n              480,\n              486,\n              492,\n              498,\n              504,\n              510,\n              516,\n              522,\n              528,\n              534,\n              540,\n              546,\n              552\n            ],\n            "hashes": [\n              "2ec9c6d6a749555997488a00d0a8ef171c5f93e5cb822cea6f8f6821e86827d7",\n              "579df9dc7c52069de489381c7b8d978df1b13bb753adae965fb239553b98f8f5",\n              "45d1bcebb7041fa4c3eb84cf1029b197581ef85b1ff670545544d694ac3c9c0c",\n              "4a9b965213a9fd9dad66ed30e99ced23e447a8f530e11e210eb6ca9520053ec0",\n              "0031a4b84dceca5517b2650d53cdd9dc4275647a1f5ff150ad24ac6cdcfe5bc4",\n              "76ef6f71922d4e1551d8a1d320b854a6d9e6c24ae9e7e54ed02273c87653b8c3",\n              "d463e4eaa96864d1d0007462599d8546d9ca979afddc4f5982063af34bc33cc5",\n              "e79ac1f6ae7e038463909faa54bed6ff8fb1c31bd56a5406006052dc5b073900",\n              "7433a3f69f5bb5289fd7b40a7538292544a6f7aef99f3ba521b3ef4e35e3e067",\n              "9adc61d0c554928dd33cef72e82acb52e99d7c7665b8effcc9774257f23a29e8",\n              "5588ec9acf0b3cc684f86f4847d338fa6f5e644945b488212a3a80f1bf1a3463",\n              "40c4e9a01a7fbfbfa306eb6763daedc2103707604bf58c53440fb47fe4e15925",\n              "f511bb7e84361dfabc2195f403813ebf6ea6255af4c460a83f786e68b390c8f7",\n              "30a260c6df1233200b0a307f26cb76aadfe8cab496f76dc251715212ca6e4f54",\n              "112f0a2ab15c11d8883fd241a442831eb54356760d1b62730d79c1abad8cd504",\n              "5f685b7f5a9f959cb740fdbb0a52ff84665e962e489f3304b5975561cc78a78b",\n              "cd222e3425a95c446e827aabd9cd9957fe82804862deedf5d279ffa0d584b562",\n              "eb7b4f720aacbe60f214aa087d60bdda90de07f0f0f26a2fda38ab14faf2f457",\n              "06910c31fcacae81d34008afc3d6cfb6a1737adc6cc8b8170ebe1eb5e8aa06ba",\n              "c833299cf75f506ff7596ed721ee643049db5a97ae7e0ad9554161b5c57f3f4f",\n              "84751ed68d5864bb94a11c8b3d5695dc0bfa2da8212c58b7f3d3b1fd05eb411f",\n              "9c4d53dbd00f9686e3cc1c1b883bb6ab74de7d9668d086fdc0b6a0e8c2c99e73",\n              "184a9492b2df7974ffd1df76283d556347f9dc3532cf57db14663b6f77ab22a7",\n              "fbf69bf3dcc3a2dc49492d78622663d58443ad632b2bd58a300086b6f40a2b88",\n              "89dab137138640e16add36366b2002bae6c9d1d57da5b1bd8c55a3304a5cf16a",\n              "0ed15455e0e1686bd7d2f9f6202a126c371964acee78a6a9742bff8a135890ab",\n              "34549fc874e70bd56bb1c95855c8b34718b3bd208ce29e81fef876f836a1666e",\n              "862ba21d4a585e42bf5c3ba2de7246e7553db003e8cfd3f4ceaf231ed4be4b6b",\n              "44072c5ded1b8a80ca97309ec397307c472b81a33ae6fda4fd0277469d2b2200",\n              "794d1374dbd6d450aafd71ca7ed16a37606ac9f165afd776c64ec00c68126a63",\n              "6a3ef476afc0741535b2613c1586c44b9add31c42f6920f08fb8687a31413265",\n              "96e6e6adb41973c1f29fb826a62ec62d1d165c9fa47642a71c168447ad9d76b2",\n              "1758e9b97bf0828f750c55a111e74992056567502fd313132dc8c7623a2ad4c3",\n              "a1c8401c87d873bc5b22bb42e34566518af7129149152f6b014d16a0567952de",\n              "6c82f57f57df92fc5565487f961abc2c759be335da04e59d217fbb2e427202a6",\n              "4fe89422cac4c84590d6a3de808dae3635df7ecd607349b7b197bce0bc44122e",\n              "c5782e659c5a6cfea4bbf4fb58bb55ac18dc2a6718601d08ff7e4feed82d2267",\n              "e0d5949a323c9b75a63dcc0f8fb9b8d6408f927f6863530981e9782cbe9d3ebd",\n              "5de57aa7c0a7dc330c7310a2d9532af2f0d5da4e33dc8046053bf74740e371f8"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": -0.078125,\n              "yes_no": {\n                "canonical": [\n                  22.0625,\n                  22.421875\n                ],\n                "reversed": [\n                  22.125,\n                  21.921875\n                ]\n              }\n            },\n            {\n              "slots": [\n                2,\n                5\n              ],\n              "relevance": 0.03125,\n              "yes_no": {\n                "canonical": [\n                  22.265625,\n                  22.546875\n                ],\n                "reversed": [\n                  22.3125,\n                  21.96875\n                ]\n              }\n            },\n            {\n              "slots": [\n                5,\n                8\n              ],\n              "relevance": -0.5,\n              "yes_no": {\n                "canonical": [\n                  21.671875,\n                  22.453125\n                ],\n                "reversed": [\n                  21.8125,\n                  22.03125\n                ]\n              }\n            },\n            {\n              "slots": [\n                8,\n                11\n              ],\n              "relevance": -0.0390625,\n              "yes_no": {\n                "canonical": [\n                  21.609375,\n                  21.984375\n                ],\n                "reversed": [\n                  21.9375,\n                  21.640625\n                ]\n              }\n            },\n            {\n              "slots": [\n                11,\n                14\n              ],\n              "relevance": -0.0234375,\n              "yes_no": {\n                "canonical": [\n                  21.6875,\n                  22.0\n                ],\n                "reversed": [\n                  21.9375,\n                  21.671875\n                ]\n              }\n            },\n            {\n              "slots": [\n                14,\n                17\n              ],\n              "relevance": 0.0859375,\n              "yes_no": {\n                "canonical": [\n                  21.96875,\n                  22.171875\n                ],\n                "reversed": [\n                  22.203125,\n                  21.828125\n                ]\n              }\n            },\n            {\n              "slots": [\n                17,\n                20\n              ],\n              "relevance": 0.078125,\n              "yes_no": {\n                "canonical": [\n                  21.890625,\n                  22.140625\n                ],\n                "reversed": [\n                  22.15625,\n                  21.75\n                ]\n              }\n            },\n            {\n              "slots": [\n                20,\n                23\n              ],\n              "relevance": 0.09375,\n              "yes_no": {\n                "canonical": [\n                  21.921875,\n                  22.15625\n                ],\n                "reversed": [\n                  22.203125,\n                  21.78125\n                ]\n              }\n            },\n            {\n              "slots": [\n                23,\n                26\n              ],\n              "relevance": 0.0,\n              "yes_no": {\n                "canonical": [\n                  22.078125,\n                  22.375\n                ],\n                "reversed": [\n                  22.265625,\n                  21.96875\n                ]\n              }\n            },\n            {\n              "slots": [\n                26,\n                29\n              ],\n              "relevance": 0.0078125,\n              "yes_no": {\n                "canonical": [\n                  22.109375,\n                  22.375\n                ],\n                "reversed": [\n                  22.3125,\n                  22.03125\n                ]\n              }\n            },\n            {\n              "slots": [\n                29,\n                32\n              ],\n              "relevance": 0.03125,\n              "yes_no": {\n                "canonical": [\n                  21.625,\n                  21.953125\n                ],\n                "reversed": [\n                  22.015625,\n                  21.625\n                ]\n              }\n            },\n            {\n              "slots": [\n                32,\n                35\n              ],\n              "relevance": -0.21875,\n              "yes_no": {\n                "canonical": [\n                  21.8125,\n                  22.34375\n                ],\n                "reversed": [\n                  22.046875,\n                  21.953125\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                2,\n                5\n              ],\n              [\n                14,\n                17\n              ],\n              [\n                20,\n                23\n              ]\n            ],\n            "constrained_windows": [\n              [\n                2,\n                5\n              ],\n              [\n                14,\n                17\n              ],\n              [\n                20,\n                23\n              ]\n            ],\n            "indices": [\n              66,\n              96,\n              354,\n              390,\n              498,\n              534\n            ],\n            "scanned_indices": 141\n          }\n        },\n        "b": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480,\n              504,\n              528,\n              552,\n              576,\n              600,\n              624,\n              648,\n              672,\n              696,\n              720,\n              744,\n              768,\n              792,\n              816,\n              840,\n              864,\n              888,\n              912,\n              936\n            ],\n            "hashes": [\n              "425a648f03ef71a480151db987862ebaf0a73337d03f57e52c2b03574fcee282",\n              "907451265dbd22bd7e3df6f3fd84223db047b9b5fda0fc6432e484ac91e51969",\n              "edddd5f68bad5d14426e15e36b58874a5581a1b9f5dad71735a54b02fc0837fe",\n              "de23900a9e32ed123174e20f9412ec9573511b09feb0f4b167e8b4951363e712",\n              "7fc4f7474e1f7688260689f7c2f9f17eb1e4db559fbc7afd05dc27de684408c5",\n              "f511bb7e84361dfabc2195f403813ebf6ea6255af4c460a83f786e68b390c8f7",\n              "98ba1c29f299c948cf03ba5f01666a555330f0fa7f92187c500e953044173b2c",\n              "30dbfeb35536d8df2c7d0506a3bb369e6c3e17435316409953c069dc6af66d4d",\n              "ab7b1324535423b10b4ace3a1b73e0eb9335591d334a7ae05a4e7eec0b93a2ac",\n              "d6698dde30a26a1171b37f207bc8829ea6301fd85fcde21118570e13615615e0",\n              "6b5e69e3266b3e88271b5b1dcccef8f949d16247ea2ec34e30372731fd22c594",\n              "2687beaa880114ae0349c8975f78907a296b65d0e50fea43b81d69e0630679ff",\n              "726f608ef89e415b0f9939fc0f3f8fa50f46527c5ee0d20317b4348eed6dc1c3",\n              "46960ebd2da846b31b63febb221ab2dfacab7d85e25f651818f539720b80fff3",\n              "30a260c6df1233200b0a307f26cb76aadfe8cab496f76dc251715212ca6e4f54",\n              "eb7b4f720aacbe60f214aa087d60bdda90de07f0f0f26a2fda38ab14faf2f457",\n              "9c4d53dbd00f9686e3cc1c1b883bb6ab74de7d9668d086fdc0b6a0e8c2c99e73",\n              "0ed15455e0e1686bd7d2f9f6202a126c371964acee78a6a9742bff8a135890ab",\n              "e7d350b5e8fc2a4447b32e26c8edeb30d8fb57c153370543c62db97059206665",\n              "ba36a1527bb5f4526676b4283b9874d55913aff69d5c10112034958575707280",\n              "34549fc874e70bd56bb1c95855c8b34718b3bd208ce29e81fef876f836a1666e",\n              "6a3ef476afc0741535b2613c1586c44b9add31c42f6920f08fb8687a31413265",\n              "6c82f57f57df92fc5565487f961abc2c759be335da04e59d217fbb2e427202a6",\n              "5de57aa7c0a7dc330c7310a2d9532af2f0d5da4e33dc8046053bf74740e371f8",\n              "7a957f7e381b62d35faa5997d8939c4bf388a7e261544b4f2885195be6efd03a",\n              "e55858172070c0728b09758408e88d8dc4e30e6814d784588b2b1d11ad00e356",\n              "ab6461a5ab3cc02dbd1d661bb43d2bf1f6bf6ac01f3bdb868b1ab159ca440054",\n              "95a20fcc8fbe37c3b56ca7e64424b23ac6f0e5d214c7133585199d5f42867578",\n              "fda1639005e1477b806be1f412e0783abd522757085f63ad95bec2e6f5b225cb",\n              "ebe907fcc52174db70868cc8f3981116f3ed43cfc8ab1822849027acf1985438",\n              "ef2677c31e4b4f0c93cf83bb2bcdeca3060f760f6f9747b544ccab83ec1952af",\n              "a3775688936ec1a72e10009993f311177cbbb596a8704fffcd3fb8383e8f8828",\n              "7d9277c63a3e9af772436943575f091011ad61c00415e3795cb17f561edb0d57",\n              "4a416955343f16824c6c84d5ddbafec7d2ba8ac9e6a48ddbb34614560f11ec97",\n              "d95879d9f127024629f020dc1b379b1dfa64d13475297bd582e240ccfaeb2d55",\n              "d42058cac3f4485516809d7e43d41561f6bf5026a09dd2a141b4a817dcefb768",\n              "1ebd320ccf92650088fafb4dea9dbe4deec08caf31e07bec368ff222c52af0d9",\n              "ecf2155c824011556c8bb72987f1f5d9d6de490564627a2318055ffc75749ae5",\n              "4f9a17077361e1fc3aff550f12232f44dd95df6df8578bf9336b00570af1b385",\n              "0be5eac53cdbd5fdb55a7ea303998b666724277e037a6ebd91e35f5839bc0b68"\n            ],\n            "slots": [\n              0,\n              7,\n              14,\n              21,\n              28,\n              35\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  21.984375,\n                  22.25\n                ],\n                "prediction": "B",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960,\n                "prompt_sha256": "1514452f426d2625fe36b67bf70434b83452c0ddacbbb70cc9f489c0ff504c39"\n              },\n              "reversed": {\n                "logits": [\n                  21.921875,\n                  22.34375\n                ],\n                "prediction": "B",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960,\n                "prompt_sha256": "62b3987507ecd683d9059cb67ca4421bfb9cb52eefa80c8f7687ee251ea363be"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  22.09375,\n                  21.703125\n                ],\n                "prediction": "A",\n                "prompt_sha256": "1514452f426d2625fe36b67bf70434b83452c0ddacbbb70cc9f489c0ff504c39",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  22.0,\n                  21.65625\n                ],\n                "prediction": "A",\n                "prompt_sha256": "62b3987507ecd683d9059cb67ca4421bfb9cb52eefa80c8f7687ee251ea363be",\n                "lm_tokens": 2046,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              0,\n              6,\n              12,\n              18,\n              24,\n              30,\n              36,\n              42,\n              48,\n              54,\n              60,\n              66,\n              72,\n              336,\n              342,\n              348,\n              354,\n              360,\n              366,\n              372,\n              378,\n              384,\n              390,\n              396,\n              402,\n              408,\n              480,\n              486,\n              492,\n              498,\n              504,\n              510,\n              516,\n              522,\n              528,\n              534,\n              540,\n              546,\n              552\n            ],\n            "hashes": [\n              "425a648f03ef71a480151db987862ebaf0a73337d03f57e52c2b03574fcee282",\n              "e23cf380b42fd77474a9d62dbbbaf2daead62bce467d64c515765c68720598f6",\n              "1457b73819f9a168673e0520d8c8d028ee2ea28fafbc7491570669058e6d4906",\n              "8296ccfd9d59db6394737b4acfb57a205caccc6c1c285b6a2f6fca574ca0df4d",\n              "907451265dbd22bd7e3df6f3fd84223db047b9b5fda0fc6432e484ac91e51969",\n              "ba40dd5e145c100b713a50a2e4676f948936d997c45b6bfb3e36227fe5a605ca",\n              "4ecc69657734b88167f1066f8f43a0b090c526b37e889d3ca9cd05775bf57e26",\n              "aadde8f1d3fa86439d8c6986387349fa0085d0696b44c3d5f0026c86a5456886",\n              "edddd5f68bad5d14426e15e36b58874a5581a1b9f5dad71735a54b02fc0837fe",\n              "eea983492c132043299e4b0ce81f7d059e3cdce351f06792215b1901f546d5f9",\n              "05069a7f0e6260841bd45474ca898d77a1c7d1aaca21f7d83b87da24b9d628d1",\n              "512f98d2afe4df9e0da13867149b2e80f4ea45e9f4d090cd80d5c801327be5bc",\n              "de23900a9e32ed123174e20f9412ec9573511b09feb0f4b167e8b4951363e712",\n              "30a260c6df1233200b0a307f26cb76aadfe8cab496f76dc251715212ca6e4f54",\n              "112f0a2ab15c11d8883fd241a442831eb54356760d1b62730d79c1abad8cd504",\n              "5f685b7f5a9f959cb740fdbb0a52ff84665e962e489f3304b5975561cc78a78b",\n              "cd222e3425a95c446e827aabd9cd9957fe82804862deedf5d279ffa0d584b562",\n              "eb7b4f720aacbe60f214aa087d60bdda90de07f0f0f26a2fda38ab14faf2f457",\n              "06910c31fcacae81d34008afc3d6cfb6a1737adc6cc8b8170ebe1eb5e8aa06ba",\n              "c833299cf75f506ff7596ed721ee643049db5a97ae7e0ad9554161b5c57f3f4f",\n              "84751ed68d5864bb94a11c8b3d5695dc0bfa2da8212c58b7f3d3b1fd05eb411f",\n              "9c4d53dbd00f9686e3cc1c1b883bb6ab74de7d9668d086fdc0b6a0e8c2c99e73",\n              "184a9492b2df7974ffd1df76283d556347f9dc3532cf57db14663b6f77ab22a7",\n              "fbf69bf3dcc3a2dc49492d78622663d58443ad632b2bd58a300086b6f40a2b88",\n              "89dab137138640e16add36366b2002bae6c9d1d57da5b1bd8c55a3304a5cf16a",\n              "0ed15455e0e1686bd7d2f9f6202a126c371964acee78a6a9742bff8a135890ab",\n              "34549fc874e70bd56bb1c95855c8b34718b3bd208ce29e81fef876f836a1666e",\n              "862ba21d4a585e42bf5c3ba2de7246e7553db003e8cfd3f4ceaf231ed4be4b6b",\n              "44072c5ded1b8a80ca97309ec397307c472b81a33ae6fda4fd0277469d2b2200",\n              "794d1374dbd6d450aafd71ca7ed16a37606ac9f165afd776c64ec00c68126a63",\n              "6a3ef476afc0741535b2613c1586c44b9add31c42f6920f08fb8687a31413265",\n              "96e6e6adb41973c1f29fb826a62ec62d1d165c9fa47642a71c168447ad9d76b2",\n              "1758e9b97bf0828f750c55a111e74992056567502fd313132dc8c7623a2ad4c3",\n              "a1c8401c87d873bc5b22bb42e34566518af7129149152f6b014d16a0567952de",\n              "6c82f57f57df92fc5565487f961abc2c759be335da04e59d217fbb2e427202a6",\n              "4fe89422cac4c84590d6a3de808dae3635df7ecd607349b7b197bce0bc44122e",\n              "c5782e659c5a6cfea4bbf4fb58bb55ac18dc2a6718601d08ff7e4feed82d2267",\n              "e0d5949a323c9b75a63dcc0f8fb9b8d6408f927f6863530981e9782cbe9d3ebd",\n              "5de57aa7c0a7dc330c7310a2d9532af2f0d5da4e33dc8046053bf74740e371f8"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": 0.859375,\n              "yes_no": {\n                "canonical": [\n                  22.546875,\n                  21.859375\n                ],\n                "reversed": [\n                  22.671875,\n                  21.640625\n                ]\n              }\n            },\n            {\n              "slots": [\n                2,\n                5\n              ],\n              "relevance": -0.03125,\n              "yes_no": {\n                "canonical": [\n                  22.296875,\n                  22.59375\n                ],\n                "reversed": [\n                  22.296875,\n                  22.0625\n                ]\n              }\n            },\n            {\n              "slots": [\n                5,\n                8\n              ],\n              "relevance": -0.5,\n              "yes_no": {\n                "canonical": [\n                  21.671875,\n                  22.453125\n                ],\n                "reversed": [\n                  21.8125,\n                  22.03125\n                ]\n              }\n            },\n            {\n              "slots": [\n                8,\n                11\n              ],\n              "relevance": -0.0390625,\n              "yes_no": {\n                "canonical": [\n                  21.609375,\n                  21.984375\n                ],\n                "reversed": [\n                  21.9375,\n                  21.640625\n                ]\n              }\n            },\n            {\n              "slots": [\n                11,\n                14\n              ],\n              "relevance": -0.0234375,\n              "yes_no": {\n                "canonical": [\n                  21.6875,\n                  22.0\n                ],\n                "reversed": [\n                  21.9375,\n                  21.671875\n                ]\n              }\n            },\n            {\n              "slots": [\n                14,\n                17\n              ],\n              "relevance": 0.0859375,\n              "yes_no": {\n                "canonical": [\n                  21.96875,\n                  22.171875\n                ],\n                "reversed": [\n                  22.203125,\n                  21.828125\n                ]\n              }\n            },\n            {\n              "slots": [\n                17,\n                20\n              ],\n              "relevance": 0.078125,\n              "yes_no": {\n                "canonical": [\n                  21.890625,\n                  22.140625\n                ],\n                "reversed": [\n                  22.15625,\n                  21.75\n                ]\n              }\n            },\n            {\n              "slots": [\n                20,\n                23\n              ],\n              "relevance": 0.09375,\n              "yes_no": {\n                "canonical": [\n                  21.921875,\n                  22.15625\n                ],\n                "reversed": [\n                  22.203125,\n                  21.78125\n                ]\n              }\n            },\n            {\n              "slots": [\n                23,\n                26\n              ],\n              "relevance": 0.0,\n              "yes_no": {\n                "canonical": [\n                  22.078125,\n                  22.375\n                ],\n                "reversed": [\n                  22.265625,\n                  21.96875\n                ]\n              }\n            },\n            {\n              "slots": [\n                26,\n                29\n              ],\n              "relevance": 0.0078125,\n              "yes_no": {\n                "canonical": [\n                  22.109375,\n                  22.375\n                ],\n                "reversed": [\n                  22.3125,\n                  22.03125\n                ]\n              }\n            },\n            {\n              "slots": [\n                29,\n                32\n              ],\n              "relevance": 0.03125,\n              "yes_no": {\n                "canonical": [\n                  21.625,\n                  21.953125\n                ],\n                "reversed": [\n                  22.015625,\n                  21.625\n                ]\n              }\n            },\n            {\n              "slots": [\n                32,\n                35\n              ],\n              "relevance": -0.21875,\n              "yes_no": {\n                "canonical": [\n                  21.8125,\n                  22.34375\n                ],\n                "reversed": [\n                  22.046875,\n                  21.953125\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                0,\n                3\n              ],\n              [\n                14,\n                17\n              ],\n              [\n                20,\n                23\n              ]\n            ],\n            "constrained_windows": [\n              [\n                2,\n                5\n              ],\n              [\n                14,\n                17\n              ],\n              [\n                20,\n                23\n              ]\n            ],\n            "indices": [\n              66,\n              96,\n              354,\n              390,\n              498,\n              534\n            ],\n            "scanned_indices": 141\n          }\n        }\n      },\n      "media": {\n        "a": {\n          "file": "videos/twinpast_0085/A.mp4",\n          "bytes": 4608254,\n          "sha256": "23909dece7fad9af70b53e8b1b333057600095ceaf999d1d2d4c485b077348ca"\n        },\n        "b": {\n          "file": "videos/twinpast_0085/B.mp4",\n          "bytes": 4651289,\n          "sha256": "06ea2b4e280cabc773f7b2c0566f8f7890f52d8bc293e74a85d757aa6b256494"\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0206",\n      "query_s": 20.958333333333332,\n      "question": "In the wooden-table scene with the green bottle, what was the large round blue lid resting on just before the scene changed?",\n      "options": [\n        "The tall translucent pitcher.",\n        "The small rectangular clear container."\n      ],\n      "branches": {\n        "a": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480\n            ],\n            "hashes": [\n              "22a7a86a33e68988745d55071be01e25fda48acfff69d7b9b33709a20aef3475",\n              "266eab0745bed76e598625b2c5a44b33749291b71db3bb8fd45d22ae4f825565",\n              "82b2b3feaecb667c8e65f8c5cdb11ac760e431da65be4dd4969a0b71022cd5d0",\n              "d1c09f3fa47a943916c69ec682407c8fb0a3e596cea1acabf503023ccb27e199",\n              "a58eee3ef0870deb471637c3ad9477543e082662a0ab4b3bddc9fb242f146f10",\n              "5823a7f5bbcfb14aa8ffe1db560e9d7eb6ab32954896fb1ef7e781e7d078d567",\n              "abaceb5d2215d545132fb14b59c47540e0886aeb6af6b84265f43859b8d09113",\n              "f7c2cd4adb341ea2b0a7c823cf103971127ff927c3089924d07a65508925c8f0",\n              "dd2a5452b7fa204cd55aea838d7436dfb135620786d6463939604d973463e54c",\n              "01acf28a20a45f83d632d9c2c9838e8bd7806199bf79ee7b3e59306108fbbf7c",\n              "2e7c338b82720f4ed942390a515ac0fda893588810e26231fd13afc117a64fa9",\n              "0f541be7cf9ef7db2d78c4d72199a532d6229a3f56d4922eb83d90dfd31ea798",\n              "3ea235ed2dc9b4cd858c2ee3a587b61f3fcbf455e89819fba15f70ceed4f3ef6",\n              "6517bc5800924515df0981b2ebb70ea5b6c79ae19e8cd1e1de4088a99e250638",\n              "89775a0a254521d6d17e46267c92fa9bbb64fd09a643c93435f352b81685ee65",\n              "fa9f4c28ef23fbd84f65aa2dd26b3d4b9bd250ff99fcff5ec2e26a36ae52d766",\n              "c1dec169288d20917828329342caefb07b7d1fbb8d82e4b8e8269175a3085a50",\n              "fad555f6aa802c1d59ca70165cf9e983519d5d0eff361d06d7a94fd103476d8d",\n              "d80c14ac3f4ee56d5dc03c43f4e31f810f1e40f2e0a44e5720a805318341e395",\n              "64c20ae4bb8d583f3fecae050f049d83f883a1fa95d301a3b1051742d93eb787",\n              "ae62d0d132093fafa04fd73fe1ccbd6dfe06ebf8923d469d7e230cffbdf11b3d"\n            ],\n            "slots": [\n              0,\n              3,\n              6,\n              9,\n              12,\n              16\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  21.578125,\n                  21.5625\n                ],\n                "prediction": "A",\n                "lm_tokens": 2042,\n                "visual_tokens": 1960,\n                "prompt_sha256": "5cbc84128f75b44d6a431c668c9048836f402dbd8c8b871bbb673a67f9a4c967"\n              },\n              "reversed": {\n                "logits": [\n                  21.6875,\n                  21.0625\n                ],\n                "prediction": "A",\n                "lm_tokens": 2042,\n                "visual_tokens": 1960,\n                "prompt_sha256": "20f66a02ebf3a94fd7f7fe1f4738e90f1c37cceb4a0e9ffb42ccfd8ebd7fdbbb"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  20.640625,\n                  21.953125\n                ],\n                "prediction": "B",\n                "prompt_sha256": "5cbc84128f75b44d6a431c668c9048836f402dbd8c8b871bbb673a67f9a4c967",\n                "lm_tokens": 2042,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  20.5625,\n                  22.109375\n                ],\n                "prediction": "B",\n                "prompt_sha256": "20f66a02ebf3a94fd7f7fe1f4738e90f1c37cceb4a0e9ffb42ccfd8ebd7fdbbb",\n                "lm_tokens": 2042,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              24,\n              30,\n              36,\n              42,\n              48,\n              54,\n              60,\n              66,\n              72,\n              78,\n              84,\n              90,\n              96,\n              120,\n              126,\n              132,\n              138,\n              144,\n              150,\n              156,\n              162,\n              168,\n              174,\n              180,\n              186,\n              192,\n              264,\n              270,\n              276,\n              282,\n              288,\n              294,\n              300,\n              306,\n              312,\n              318,\n              324,\n              330,\n              336\n            ],\n            "hashes": [\n              "266eab0745bed76e598625b2c5a44b33749291b71db3bb8fd45d22ae4f825565",\n              "bb98e75ffce932482e846f2ab966bc7bf3c2490063c8adc8d5d0e98bf0cc4336",\n              "761bc507ede009353edd7da7b55ee7e8b3f9e2b837b02deeda8a6cc18a597e3c",\n              "c830d6f2ef4fbd94b38076ecc3e572acd4aec1d4610a3be139ece14222beb458",\n              "82b2b3feaecb667c8e65f8c5cdb11ac760e431da65be4dd4969a0b71022cd5d0",\n              "7e31b3c24177cc9a672e14145fb3652afffd55e586c8f5300d53dd5d3e002678",\n              "1645be8a9a2312628b93536c0f9ac98085da1ac52bb028641a96e082ef9db07f",\n              "94177c12dd681929a7f57f5a2f417d905ba7c5bea5fb3388bf6c5c16d5fa3d11",\n              "d1c09f3fa47a943916c69ec682407c8fb0a3e596cea1acabf503023ccb27e199",\n              "99ba015dcc16f67c7837a53550a09d2108c254ba9651ccf538154d303551f536",\n              "91e3f23207b394666efecb4856d55323327714d8f31996b84d35b6077506161e",\n              "2dc9258f573119badf65889c2229901a1c6d7866b315ff1db3b41c675d60f618",\n              "a58eee3ef0870deb471637c3ad9477543e082662a0ab4b3bddc9fb242f146f10",\n              "5823a7f5bbcfb14aa8ffe1db560e9d7eb6ab32954896fb1ef7e781e7d078d567",\n              "3de764631411192b0ad9b0ca7fee6a6b0340ebddc477fffa6ab7e236063bd2a9",\n              "e77a690ddfa6ceb7b98ad74f41630c4d418c4668e1dd80442ea93e07b21c6c39",\n              "581ff3bf538a28cb1a3ad0da05751390b18e627aa0894c127f4fbf646790a999",\n              "abaceb5d2215d545132fb14b59c47540e0886aeb6af6b84265f43859b8d09113",\n              "6f7955ba6bdb558f883ebcfd0156a1a2733ec0c44f86ccb2e2fe7f0771891d57",\n              "a82eb8b8d94bd80e612622534fb5f9491e4fc89acf61ad7bcc157de78c1a1e69",\n              "cb513ba27fee25bfae8667bbbb50346a93c6975b331d767863d90e53eb05735a",\n              "f7c2cd4adb341ea2b0a7c823cf103971127ff927c3089924d07a65508925c8f0",\n              "115924b9b11b849fe191bcdb393e2a6478c5fea15044953f4b658dfcb6794f0f",\n              "64c4b05a3f374c8322ca10660afca4567cb3480763db6d1c8e8c920de9ff333d",\n              "b1e94c43e85633147c6ff7b648eb66ac67a3174c8ff892fdec207180c2e46c15",\n              "dd2a5452b7fa204cd55aea838d7436dfb135620786d6463939604d973463e54c",\n              "0f541be7cf9ef7db2d78c4d72199a532d6229a3f56d4922eb83d90dfd31ea798",\n              "6cf283cb4091b00f4a2a8cb53a351831c71af3d80772d4bb78e23eb63629ee0a",\n              "4b536c4bbc2b3b71c225f5436b4f33b24809b6f9f3045e4571ee00f708e293c8",\n              "6879e9fdb15dcfc617753f45c12ea25a898df92689495631d5c0d274ac4552f1",\n              "3ea235ed2dc9b4cd858c2ee3a587b61f3fcbf455e89819fba15f70ceed4f3ef6",\n              "922a5871ca10793ba582982a77ab1539f7758604e6dc1f46b751b45caa74a182",\n              "8ccaaffb73a8c8e4efb015753f67a6d0535ef250f8c4ceef9fd8cb5adf7af77b",\n              "55e41f18ecd84f3864106cfc5ae9ce2ba3cf731ccd204491295708413ec1fa1e",\n              "6517bc5800924515df0981b2ebb70ea5b6c79ae19e8cd1e1de4088a99e250638",\n              "5a227fd6e90006ab990ea88185bd665d046ebb50bca052db92073c0043f2018a",\n              "9bbccadc45232b547ba5f7113f19601535832e4e56c601b669c91f475a6877c2",\n              "510112942775bde7b1f4de945590436c98c7b28a5939fc5c6f822245891542e6",\n              "89775a0a254521d6d17e46267c92fa9bbb64fd09a643c93435f352b81685ee65"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": -0.2734375,\n              "yes_no": {\n                "canonical": [\n                  21.546875,\n                  22.109375\n                ],\n                "reversed": [\n                  21.65625,\n                  21.640625\n                ]\n              }\n            },\n            {\n              "slots": [\n                1,\n                4\n              ],\n              "relevance": -0.1171875,\n              "yes_no": {\n                "canonical": [\n                  21.640625,\n                  22.015625\n                ],\n                "reversed": [\n                  21.71875,\n                  21.578125\n                ]\n              }\n            },\n            {\n              "slots": [\n                2,\n                5\n              ],\n              "relevance": -0.3046875,\n              "yes_no": {\n                "canonical": [\n                  21.578125,\n                  22.171875\n                ],\n                "reversed": [\n                  21.734375,\n                  21.75\n                ]\n              }\n            },\n            {\n              "slots": [\n                3,\n                6\n              ],\n              "relevance": -0.2578125,\n              "yes_no": {\n                "canonical": [\n                  21.46875,\n                  22.03125\n                ],\n                "reversed": [\n                  21.640625,\n                  21.59375\n                ]\n              }\n            },\n            {\n              "slots": [\n                4,\n                7\n              ],\n              "relevance": -0.0625,\n              "yes_no": {\n                "canonical": [\n                  21.671875,\n                  22.015625\n                ],\n                "reversed": [\n                  21.796875,\n                  21.578125\n                ]\n              }\n            },\n            {\n              "slots": [\n                5,\n                8\n              ],\n              "relevance": -0.1328125,\n              "yes_no": {\n                "canonical": [\n                  21.578125,\n                  22.0\n                ],\n                "reversed": [\n                  21.75,\n                  21.59375\n                ]\n              }\n            },\n            {\n              "slots": [\n                7,\n                10\n              ],\n              "relevance": -0.3828125,\n              "yes_no": {\n                "canonical": [\n                  21.703125,\n                  22.375\n                ],\n                "reversed": [\n                  21.8125,\n                  21.90625\n                ]\n              }\n            },\n            {\n              "slots": [\n                8,\n                11\n              ],\n              "relevance": -0.5,\n              "yes_no": {\n                "canonical": [\n                  21.546875,\n                  22.375\n                ],\n                "reversed": [\n                  21.703125,\n                  21.875\n                ]\n              }\n            },\n            {\n              "slots": [\n                9,\n                12\n              ],\n              "relevance": 0.34375,\n              "yes_no": {\n                "canonical": [\n                  22.109375,\n                  21.953125\n                ],\n                "reversed": [\n                  22.125,\n                  21.59375\n                ]\n              }\n            },\n            {\n              "slots": [\n                10,\n                13\n              ],\n              "relevance": 0.4296875,\n              "yes_no": {\n                "canonical": [\n                  21.953125,\n                  21.71875\n                ],\n                "reversed": [\n                  22.0,\n                  21.375\n                ]\n              }\n            },\n            {\n              "slots": [\n                11,\n                14\n              ],\n              "relevance": 0.640625,\n              "yes_no": {\n                "canonical": [\n                  21.953125,\n                  21.46875\n                ],\n                "reversed": [\n                  22.03125,\n                  21.234375\n                ]\n              }\n            },\n            {\n              "slots": [\n                13,\n                16\n              ],\n              "relevance": -0.59375,\n              "yes_no": {\n                "canonical": [\n                  21.484375,\n                  22.359375\n                ],\n                "reversed": [\n                  21.609375,\n                  21.921875\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                1,\n                4\n              ],\n              [\n                5,\n                8\n              ],\n              [\n                11,\n                14\n              ]\n            ],\n            "constrained_windows": [\n              [\n                1,\n                4\n              ],\n              [\n                5,\n                8\n              ],\n              [\n                11,\n                14\n              ]\n            ],\n            "indices": [\n              42,\n              78,\n              138,\n              174,\n              282,\n              318\n            ],\n            "scanned_indices": 65\n          }\n        },\n        "b": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480\n            ],\n            "hashes": [\n              "22a7a86a33e68988745d55071be01e25fda48acfff69d7b9b33709a20aef3475",\n              "266eab0745bed76e598625b2c5a44b33749291b71db3bb8fd45d22ae4f825565",\n              "82b2b3feaecb667c8e65f8c5cdb11ac760e431da65be4dd4969a0b71022cd5d0",\n              "d1c09f3fa47a943916c69ec682407c8fb0a3e596cea1acabf503023ccb27e199",\n              "a58eee3ef0870deb471637c3ad9477543e082662a0ab4b3bddc9fb242f146f10",\n              "5823a7f5bbcfb14aa8ffe1db560e9d7eb6ab32954896fb1ef7e781e7d078d567",\n              "abaceb5d2215d545132fb14b59c47540e0886aeb6af6b84265f43859b8d09113",\n              "f7c2cd4adb341ea2b0a7c823cf103971127ff927c3089924d07a65508925c8f0",\n              "dd2a5452b7fa204cd55aea838d7436dfb135620786d6463939604d973463e54c",\n              "631eb501b3a1bb37afc004a171fe54143656cc2fd8b9048fc0e84f27672eb382",\n              "190f9db6316cb188b1a43d567e68f91758462f36161aaa49e73811f43ffb674e",\n              "3e2a9a3a2a3d75e513e29d08fc0ba5b2f24cee9d13e2dea69d6444d561275d96",\n              "3ea235ed2dc9b4cd858c2ee3a587b61f3fcbf455e89819fba15f70ceed4f3ef6",\n              "6517bc5800924515df0981b2ebb70ea5b6c79ae19e8cd1e1de4088a99e250638",\n              "89775a0a254521d6d17e46267c92fa9bbb64fd09a643c93435f352b81685ee65",\n              "fa9f4c28ef23fbd84f65aa2dd26b3d4b9bd250ff99fcff5ec2e26a36ae52d766",\n              "c1dec169288d20917828329342caefb07b7d1fbb8d82e4b8e8269175a3085a50",\n              "fad555f6aa802c1d59ca70165cf9e983519d5d0eff361d06d7a94fd103476d8d",\n              "d80c14ac3f4ee56d5dc03c43f4e31f810f1e40f2e0a44e5720a805318341e395",\n              "64c20ae4bb8d583f3fecae050f049d83f883a1fa95d301a3b1051742d93eb787",\n              "ae62d0d132093fafa04fd73fe1ccbd6dfe06ebf8923d469d7e230cffbdf11b3d"\n            ],\n            "slots": [\n              0,\n              3,\n              6,\n              9,\n              12,\n              16\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  22.015625,\n                  20.734375\n                ],\n                "prediction": "A",\n                "lm_tokens": 2042,\n                "visual_tokens": 1960,\n                "prompt_sha256": "5cbc84128f75b44d6a431c668c9048836f402dbd8c8b871bbb673a67f9a4c967"\n              },\n              "reversed": {\n                "logits": [\n                  22.140625,\n                  20.25\n                ],\n                "prediction": "A",\n                "lm_tokens": 2042,\n                "visual_tokens": 1960,\n                "prompt_sha256": "20f66a02ebf3a94fd7f7fe1f4738e90f1c37cceb4a0e9ffb42ccfd8ebd7fdbbb"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  21.3125,\n                  21.546875\n                ],\n                "prediction": "B",\n                "prompt_sha256": "5cbc84128f75b44d6a431c668c9048836f402dbd8c8b871bbb673a67f9a4c967",\n                "lm_tokens": 2042,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  21.453125,\n                  21.234375\n                ],\n                "prediction": "A",\n                "prompt_sha256": "20f66a02ebf3a94fd7f7fe1f4738e90f1c37cceb4a0e9ffb42ccfd8ebd7fdbbb",\n                "lm_tokens": 2042,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              24,\n              30,\n              36,\n              42,\n              48,\n              54,\n              60,\n              66,\n              72,\n              78,\n              84,\n              90,\n              96,\n              168,\n              174,\n              180,\n              186,\n              192,\n              198,\n              204,\n              210,\n              216,\n              222,\n              228,\n              234,\n              240,\n              264,\n              270,\n              276,\n              282,\n              288,\n              294,\n              300,\n              306,\n              312,\n              318,\n              324,\n              330,\n              336\n            ],\n            "hashes": [\n              "266eab0745bed76e598625b2c5a44b33749291b71db3bb8fd45d22ae4f825565",\n              "bb98e75ffce932482e846f2ab966bc7bf3c2490063c8adc8d5d0e98bf0cc4336",\n              "761bc507ede009353edd7da7b55ee7e8b3f9e2b837b02deeda8a6cc18a597e3c",\n              "c830d6f2ef4fbd94b38076ecc3e572acd4aec1d4610a3be139ece14222beb458",\n              "82b2b3feaecb667c8e65f8c5cdb11ac760e431da65be4dd4969a0b71022cd5d0",\n              "7e31b3c24177cc9a672e14145fb3652afffd55e586c8f5300d53dd5d3e002678",\n              "1645be8a9a2312628b93536c0f9ac98085da1ac52bb028641a96e082ef9db07f",\n              "94177c12dd681929a7f57f5a2f417d905ba7c5bea5fb3388bf6c5c16d5fa3d11",\n              "d1c09f3fa47a943916c69ec682407c8fb0a3e596cea1acabf503023ccb27e199",\n              "99ba015dcc16f67c7837a53550a09d2108c254ba9651ccf538154d303551f536",\n              "91e3f23207b394666efecb4856d55323327714d8f31996b84d35b6077506161e",\n              "2dc9258f573119badf65889c2229901a1c6d7866b315ff1db3b41c675d60f618",\n              "a58eee3ef0870deb471637c3ad9477543e082662a0ab4b3bddc9fb242f146f10",\n              "f7c2cd4adb341ea2b0a7c823cf103971127ff927c3089924d07a65508925c8f0",\n              "115924b9b11b849fe191bcdb393e2a6478c5fea15044953f4b658dfcb6794f0f",\n              "64c4b05a3f374c8322ca10660afca4567cb3480763db6d1c8e8c920de9ff333d",\n              "b1e94c43e85633147c6ff7b648eb66ac67a3174c8ff892fdec207180c2e46c15",\n              "dd2a5452b7fa204cd55aea838d7436dfb135620786d6463939604d973463e54c",\n              "e9df89d2920a7781cc383a29f919c8f74edb77430e19cac55875334529f13325",\n              "4d40986e185d18d4e7605139784493391dc90246828c550c55426b5513ca2ce4",\n              "c060bf99445df39f995e491ad7f204f65d1a1236ad727c2c9a3559712c7e4c50",\n              "631eb501b3a1bb37afc004a171fe54143656cc2fd8b9048fc0e84f27672eb382",\n              "cd192d6ad736194e249a34d1868808d4e8fc760980d74b87261cb38aebba6599",\n              "852b0857a6409432106e17ca2ca36835f37d17bbb425b63c26154d8d2ad3fb5e",\n              "2ff2f933a3afe8ddd098455c6cd9a53e98cef79988a76f83f006fc4b643bc7aa",\n              "190f9db6316cb188b1a43d567e68f91758462f36161aaa49e73811f43ffb674e",\n              "3e2a9a3a2a3d75e513e29d08fc0ba5b2f24cee9d13e2dea69d6444d561275d96",\n              "04a5adfde45fc09e03b1f57d6a4bf2b5eae05b9d8f12be1e8c83f434f340bfaa",\n              "cf90f2f256e11c9e578a99b4e1d6f6b95a311f89cb98132f391ede292585ae79",\n              "8f62b0e9631cb0cb748b74e546c0f61178d70443008ad0f1b7e0281ca71d39f7",\n              "3ea235ed2dc9b4cd858c2ee3a587b61f3fcbf455e89819fba15f70ceed4f3ef6",\n              "922a5871ca10793ba582982a77ab1539f7758604e6dc1f46b751b45caa74a182",\n              "8ccaaffb73a8c8e4efb015753f67a6d0535ef250f8c4ceef9fd8cb5adf7af77b",\n              "55e41f18ecd84f3864106cfc5ae9ce2ba3cf731ccd204491295708413ec1fa1e",\n              "6517bc5800924515df0981b2ebb70ea5b6c79ae19e8cd1e1de4088a99e250638",\n              "5a227fd6e90006ab990ea88185bd665d046ebb50bca052db92073c0043f2018a",\n              "9bbccadc45232b547ba5f7113f19601535832e4e56c601b669c91f475a6877c2",\n              "510112942775bde7b1f4de945590436c98c7b28a5939fc5c6f822245891542e6",\n              "89775a0a254521d6d17e46267c92fa9bbb64fd09a643c93435f352b81685ee65"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": -0.2734375,\n              "yes_no": {\n                "canonical": [\n                  21.546875,\n                  22.109375\n                ],\n                "reversed": [\n                  21.65625,\n                  21.640625\n                ]\n              }\n            },\n            {\n              "slots": [\n                1,\n                4\n              ],\n              "relevance": -0.1171875,\n              "yes_no": {\n                "canonical": [\n                  21.640625,\n                  22.015625\n                ],\n                "reversed": [\n                  21.71875,\n                  21.578125\n                ]\n              }\n            },\n            {\n              "slots": [\n                2,\n                5\n              ],\n              "relevance": -0.3046875,\n              "yes_no": {\n                "canonical": [\n                  21.578125,\n                  22.171875\n                ],\n                "reversed": [\n                  21.734375,\n                  21.75\n                ]\n              }\n            },\n            {\n              "slots": [\n                3,\n                6\n              ],\n              "relevance": -0.2578125,\n              "yes_no": {\n                "canonical": [\n                  21.46875,\n                  22.03125\n                ],\n                "reversed": [\n                  21.640625,\n                  21.59375\n                ]\n              }\n            },\n            {\n              "slots": [\n                4,\n                7\n              ],\n              "relevance": -0.0625,\n              "yes_no": {\n                "canonical": [\n                  21.671875,\n                  22.015625\n                ],\n                "reversed": [\n                  21.796875,\n                  21.578125\n                ]\n              }\n            },\n            {\n              "slots": [\n                5,\n                8\n              ],\n              "relevance": -0.1328125,\n              "yes_no": {\n                "canonical": [\n                  21.578125,\n                  22.0\n                ],\n                "reversed": [\n                  21.75,\n                  21.59375\n                ]\n              }\n            },\n            {\n              "slots": [\n                7,\n                10\n              ],\n              "relevance": 0.0859375,\n              "yes_no": {\n                "canonical": [\n                  21.53125,\n                  21.734375\n                ],\n                "reversed": [\n                  21.6875,\n                  21.3125\n                ]\n              }\n            },\n            {\n              "slots": [\n                8,\n                11\n              ],\n              "relevance": -0.0078125,\n              "yes_no": {\n                "canonical": [\n                  21.65625,\n                  21.984375\n                ],\n                "reversed": [\n                  21.796875,\n                  21.484375\n                ]\n              }\n            },\n            {\n              "slots": [\n                9,\n                12\n              ],\n              "relevance": -0.5078125,\n              "yes_no": {\n                "canonical": [\n                  21.65625,\n                  22.53125\n                ],\n                "reversed": [\n                  21.90625,\n                  22.046875\n                ]\n              }\n            },\n            {\n              "slots": [\n                10,\n                13\n              ],\n              "relevance": -0.1875,\n              "yes_no": {\n                "canonical": [\n                  21.71875,\n                  22.21875\n                ],\n                "reversed": [\n                  21.9375,\n                  21.8125\n                ]\n              }\n            },\n            {\n              "slots": [\n                11,\n                14\n              ],\n              "relevance": -0.296875,\n              "yes_no": {\n                "canonical": [\n                  21.71875,\n                  22.359375\n                ],\n                "reversed": [\n                  21.984375,\n                  21.9375\n                ]\n              }\n            },\n            {\n              "slots": [\n                13,\n                16\n              ],\n              "relevance": -0.59375,\n              "yes_no": {\n                "canonical": [\n                  21.484375,\n                  22.359375\n                ],\n                "reversed": [\n                  21.609375,\n                  21.921875\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                1,\n                4\n              ],\n              [\n                7,\n                10\n              ],\n              [\n                11,\n                14\n              ]\n            ],\n            "constrained_windows": [\n              [\n                1,\n                4\n              ],\n              [\n                7,\n                10\n              ],\n              [\n                11,\n                14\n              ]\n            ],\n            "indices": [\n              42,\n              78,\n              186,\n              222,\n              282,\n              318\n            ],\n            "scanned_indices": 65\n          }\n        }\n      },\n      "media": {\n        "a": {\n          "file": "videos/twinpast_0206/A.mp4",\n          "bytes": 7534621,\n          "sha256": "dec07694bf90c232ef4b26ddf824e42fd805c5e3b6ffe99821fe4c7d3bacefbb"\n        },\n        "b": {\n          "file": "videos/twinpast_0206/B.mp4",\n          "bytes": 7541950,\n          "sha256": "e0799ab8db777a0d67a0ea176177b8937bfd750c318fafaa6a0793a4a923a1e1"\n        }\n      }\n    },\n    {\n      "pair_id": "twinpast_0021",\n      "query_s": 66.95833333333333,\n      "question": "In the last view of the orange bowl resting on the counter before the eggs are poured into the frying pan, are separate yolks still visible, or have the eggs been beaten into a blended mixture?",\n      "options": [\n        "Separate yolks were still visible.",\n        "The eggs had been beaten into a blended mixture."\n      ],\n      "branches": {\n        "a": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480,\n              504,\n              528,\n              552,\n              576,\n              600,\n              624,\n              648,\n              672,\n              696,\n              720,\n              744,\n              768,\n              792,\n              816,\n              840,\n              864,\n              888,\n              912,\n              936,\n              960,\n              984,\n              1008,\n              1032,\n              1056,\n              1080,\n              1104,\n              1128,\n              1152,\n              1176,\n              1200,\n              1224,\n              1248,\n              1272,\n              1296,\n              1320,\n              1344,\n              1368,\n              1392,\n              1416,\n              1440,\n              1464,\n              1488,\n              1512,\n              1536,\n              1560,\n              1584\n            ],\n            "hashes": [\n              "010ded0367a7efddd8a6a9035597984d1b82d616e38204dc874dd00e040dc01a",\n              "d465e54b4e6b8042f59342b6ec09425580b369b603c8d0bbfb370ce3cd68cac5",\n              "e1c7840c75bcd121687a5e4bd365e7001e017e30946fd3103d53044af025884d",\n              "72f3d812035f5555894fa5d317cf0f5e187a08b24a9ff68df21767b49f6e9d75",\n              "52d4f0c163acf3122f82dd9cc7a2297c9d287cd67a11e43d8daf818bc476137f",\n              "fa7c92a8c7855f7901f38aca034f18d272eba6387aa89126712672d7a4bbb698",\n              "b7c4ef7d083c6033a2f471cd9f90ab3c09ec082a5cdcb8f44c1493dc6b1df7af",\n              "c11bfe529f6f22104db01d1b6ab7d3eca0273f2165d43ba025e4c1d7ba99f996",\n              "34c0648f09e00160fb784483cf2a4df3e3eb39beabfb1898f2f5fd1420ea01bf",\n              "cdf0aad89e83ab3898d23ea10941026d6db412133aa311391d5901ce5b1006de",\n              "ae8428564dcadc822a12bbf4e711df7e377f31a8efabfc4dd9a30099238f8cea",\n              "173b437522748862d08c19a32b9e08c9e273ab2be15462f7ae05c3cf46855a55",\n              "dc15c565026a855fe1605e4bc5aceddcecd87e6b3ab544e4f1841195ca091503",\n              "57f97d9f4617f968457a61545570d969a53d3302f7b1e6c8a63a4ac974d67dc8",\n              "093e75ed0b7fce34df1284b165e47fb47c14a6d9d97a7dde18d803361849f19c",\n              "1e4e5eb6ba8da3fb0d150c1bab500e65e41a3fb11865f618cfdd6b2b103f3659",\n              "04e96508a03a9f7d371db2e07287d06e9f3801ade4add206239b8671c41af0ba",\n              "913a6d5d915deeb8240d5903539441827aa6f2eae22fdd246fc44b0862163e43",\n              "9764c2be238d1c8bc46bea3bd3bed70007f287d1b1990ec7807bd5708359d592",\n              "5d87c25f1e84bbe80a8f921a0210d05ba301e78a316661a86fc509c9a22bcd2d",\n              "b5ad03faa8987f18a97c637ea298e189e063c1a028bb52c68f269012fa5a071e",\n              "d7f642e87ae4fc19d206bf3f56005231f9fd50ddacaa5e2c5e736ea1e76df315",\n              "c166bb25a40e23bbef9e7d8bac271f797c14e72ca822fc626c25a06db4167180",\n              "d3d179be89c17c049ecb3e8b334e13d87b4c5629303934a9bca188ff4e71e4a4",\n              "f128ac92973cf96d0f891daea041a597895dfdc62df5954e23b4094bf99bd940",\n              "04af3c96bda43e79a9141d91607bcfe78e0f8ca12623545bcde28fa50e61ff5f",\n              "76eb1d72588fda9f0017b704be8a76c243c42320a024939108d8759867e37767",\n              "d7856893f900340c415cb9263fba7b4aff541a99766f87894dffd3d7420ac98b",\n              "056fbe01e3a8e352a44972844d41f5241c7df00806733dd5f4c20b7e7acfe2c8",\n              "e9cefc6db5004b6e01154ae61ac93812c3cc0d2813397c5cd98cfee4b83bce7e",\n              "5bf25d77f7bff8e6bb767a4c1e127eb7556c9447328b48201b89c31915b522be",\n              "8786afd84e899868f31d768a0e2dca9bd51e2e5ac03122211f7c19f750dcce96",\n              "321bace629d850b589580b9963bba87a30a43bfe717934569c1d643c522614ec",\n              "e029bcd4016852960beb9a3f07584f01c49971f1ce830649ffc3561f51a0b47f",\n              "f213739918ebb2649dd97a375b8f26f99a4911336f38c95d43b7184fb447fea2",\n              "ec925f9fda9a3f168a65eafeffc5a4b55e66f8d10864e9603b57e61d6cb61c3c",\n              "0e27863181f59832065a487a04f61bda9516d95485929187f750ca03318ee3b8",\n              "8bba6fb74992d1dc8ed8bd551c033251a61cca041a45f284014e40343eefd2cc",\n              "fd453ab40a34217d3faf9a669e593b5a34e055943dd695092c360f26a8df76f1",\n              "96b1e80c00f5b13cc5af6bd32a60280aee93cbf40d4999080891ac44b7f82787",\n              "a6341d3c97df58ddc714cdde3fdfcbb8d1d1ddb9f1dee59e2c4282bf4c86418f",\n              "0fc810e8d8d5680814ced0187a473046eb897d738b8b1f1ed69f320d63361291",\n              "b9e675e8a3c406baf20a42cc1634f101b6f10f82b0560ea05b9b76c4a0ccde15",\n              "917d59da2f65eec6e03a1981f6f58ca249a3c0c8bdc482ab806041d3a0d158cc",\n              "25fd3a8e28eb437e5171d9fff6cea0e0b5fc19cfba3a3f0d47cf094fc000eea6",\n              "722e47fdb243bce61192039c30c77d4f886e516872977e06675a384836c163b1",\n              "99824eb673a84453461cce366e14ac9c284f2f63070843614841801ac69867b1",\n              "38b6d8bedad6ce59741412ab98a396a1b8d2684eacc72a34d3c4762987bf0762",\n              "ff4f0c0bcd01c5617b2d476b36253c1f2fceaca00b743db92f08799a66dc173c",\n              "68a69c8225695dad86e82ca4e064d1b997776404231a22dbd5092a19a585ca1b",\n              "ac4bdaca14a029a651c2373fa156aa557284828164f6d316c1118716266de0a0",\n              "99f0ce76ae123bf4a6d0704766a12b6f594a18d9cae72d869da374fd860db8f9",\n              "56cde5d448da438d6b257f99e5e9b529647a01f5b598e95a74ac2e4112fa0448",\n              "e8794554263e63206e66e358d0f7ba42e584e0668c90a47886cb91b803801a7c",\n              "bcca351806231f25339f62fe4814a31571324ea72a0563c9c0eee5d1d918ed8d",\n              "36f7b23877d32039b40cf88407732d2edf274b3a52b4628a18c150c667512b4b",\n              "8be277ccc53257208d81d8f9629919f9a2db307180f48835fe63c9558d11fa0e",\n              "f20a682d32dccb66d4f59d6fb31148388f6054f99d7192e421a8fc7e58a0d835",\n              "13f98d416623cff7d1de30dfee77fd0a094d3d88385847117614b161914bd7ae",\n              "22fa12976a15061694e24614e3942496a910521adc26d82b27e4c0a1b732e2e2",\n              "8465e4ae653d14cc479dbc9de665f2bb505bf08333357b07352e932b37b61547",\n              "ee298333dee53864b18a01b58a717e7e78a78c237427c682e4e71dbd5c0f2fff",\n              "4a9bc00b08403c06f5123105775c7f0624a7a4451cfb5b1cbd75c0deebe6ca2a",\n              "3de34c51691676b656a329fc130ea17cb635185f9e070e2e3fcb344e47007344",\n              "a060d1d1dedc354281aa7332f8623d1d2bdc32e9186710af21f8e7c78e4b80db",\n              "3fd333ec7946cfac3131f26e6d2ffcb6af45094b1ecae0c1932a0ce1515d05c3",\n              "984fd146f6cb4775df93ee3a4a7995619abc3ad686d6f5616b2f36594145d503"\n            ],\n            "slots": [\n              0,\n              12,\n              24,\n              37,\n              49,\n              62\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  20.015625,\n                  22.171875\n                ],\n                "prediction": "B",\n                "lm_tokens": 2063,\n                "visual_tokens": 1960,\n                "prompt_sha256": "8454c18515049578828ceb0abf747cbd2d3b147467ae4e94c40b05c8df2e953b"\n              },\n              "reversed": {\n                "logits": [\n                  20.65625,\n                  22.328125\n                ],\n                "prediction": "B",\n                "lm_tokens": 2063,\n                "visual_tokens": 1960,\n                "prompt_sha256": "21e46534011b13f3b7b8c51d3262080017e5d9f90773e42ef65da8f5cf278f6d"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  20.34375,\n                  22.234375\n                ],\n                "prediction": "B",\n                "prompt_sha256": "8454c18515049578828ceb0abf747cbd2d3b147467ae4e94c40b05c8df2e953b",\n                "lm_tokens": 2063,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  20.765625,\n                  22.34375\n                ],\n                "prediction": "B",\n                "prompt_sha256": "21e46534011b13f3b7b8c51d3262080017e5d9f90773e42ef65da8f5cf278f6d",\n                "lm_tokens": 2063,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              768,\n              774,\n              780,\n              786,\n              792,\n              798,\n              804,\n              810,\n              816,\n              822,\n              828,\n              834,\n              840,\n              888,\n              894,\n              900,\n              906,\n              912,\n              918,\n              924,\n              930,\n              936,\n              942,\n              948,\n              954,\n              960,\n              1008,\n              1014,\n              1020,\n              1026,\n              1032,\n              1038,\n              1044,\n              1050,\n              1056,\n              1062,\n              1068,\n              1074,\n              1080\n            ],\n            "hashes": [\n              "321bace629d850b589580b9963bba87a30a43bfe717934569c1d643c522614ec",\n              "5d215a6f008948c54ff47e62f42a7b7f05afa3f738de82d09cdacc5de161dcda",\n              "49da6833928d8e9f6195447489ca81017ddb3985f910aaafbfd31f9725dfc7de",\n              "11784ad25f5d559f12962de002c05224d6f6beaa20ecc512ad79717481f0db72",\n              "e029bcd4016852960beb9a3f07584f01c49971f1ce830649ffc3561f51a0b47f",\n              "ddc53d027dbec85f97dd58d081988592d3de63461e4eef9fa79f24d35ea5a711",\n              "570f0730b98e6999068776206205f29a9308f700e66253059dcfe0d50b50b755",\n              "f1eb35615f4578436c51578456d8bfffc926b901cf1b58ed957f43b8ac8e63b1",\n              "f213739918ebb2649dd97a375b8f26f99a4911336f38c95d43b7184fb447fea2",\n              "4ada5f83c13c9fb55a37a877f9b9c129190845c46445ca4f8dd054c96c886b52",\n              "ee7de8ad1f318a30632925f0747c1addbc1c0bc32a5d825c92464dbf9c4732e2",\n              "4f3021264c0e11efe8313c2b4cf53b765c212fc162f5536904bad46d594eee2a",\n              "ec925f9fda9a3f168a65eafeffc5a4b55e66f8d10864e9603b57e61d6cb61c3c",\n              "8bba6fb74992d1dc8ed8bd551c033251a61cca041a45f284014e40343eefd2cc",\n              "208503d190ec1d24361047e97c273f66b6e7554e2fa250211f936dbf7c6009f7",\n              "24364281099cf004ca5ec2236128dc5cdc6d9b865519f1253fecc6b1b4f55f37",\n              "9ec61497a42f3891d9546bd9b9add1e082e9472a32ca8eb18c703df1ac1bf1ba",\n              "fd453ab40a34217d3faf9a669e593b5a34e055943dd695092c360f26a8df76f1",\n              "eabf1d2dc027f76c0a144f9698e0e9b0c4faac4c14538690ab0379f8f566f1ee",\n              "06cd9690c41bff58b07e902117d74b19742eecda93f8ee7c9cf5d4c2350e86b3",\n              "b06861538fe4ed41fdf0f135fb0b46542be35b4bd73327f9f40cc4930604e5c7",\n              "96b1e80c00f5b13cc5af6bd32a60280aee93cbf40d4999080891ac44b7f82787",\n              "2a2f324042648087fe0df9d821aeb7bc2e05c9ef27c5df2ac2bda9271a8ec645",\n              "57eb0e0ce8b9b0cc01729f7c5f62a996acf9a5094d5d69342a9f9fd7987ba63f",\n              "333c75a8f59c3f6661b17ccc07a622e74bb062ff04c1d5d2fa01ec156a3fc313",\n              "a6341d3c97df58ddc714cdde3fdfcbb8d1d1ddb9f1dee59e2c4282bf4c86418f",\n              "b9e675e8a3c406baf20a42cc1634f101b6f10f82b0560ea05b9b76c4a0ccde15",\n              "8cff872e43d21cbc8e805aefdb1a010cecb527ac90e5c7e827fc1bc5ce6f00d7",\n              "534606244b6a00fa60a08a34039942a3fa41958901a9193256f034de711f879d",\n              "b178acf2c9dd1347c070754393c6201bae3098cd9bc35720827858770b0e016e",\n              "917d59da2f65eec6e03a1981f6f58ca249a3c0c8bdc482ab806041d3a0d158cc",\n              "67bb904a9d3fea2c4c9a315a3f990982695001c993e7ffb291ff0e5c3f585ed9",\n              "b1473e57b405f35ea24f5abb6f621c216f5d27186ed58c7a1d95455168e58af4",\n              "b81efa126fee51b050b8a503c9e34dce2120c5551d765cde63de38f9f8477d7a",\n              "25fd3a8e28eb437e5171d9fff6cea0e0b5fc19cfba3a3f0d47cf094fc000eea6",\n              "453aaa4446bf9d2c6fbdad89711500d7acfa6e12a8af9e64d9b4ef85cc317624",\n              "c1db45f592f6a80a0fc47a6d26f8d81bcde40a390bfb4315afbde75bf4175b3e",\n              "c07b07b6adfa00f73652d8e18207ff0d20fcb9b4039df733993c3796c0667621",\n              "722e47fdb243bce61192039c30c77d4f886e516872977e06675a384836c163b1"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": 0.2421875,\n              "yes_no": {\n                "canonical": [\n                  21.703125,\n                  21.78125\n                ],\n                "reversed": [\n                  21.90625,\n                  21.34375\n                ]\n              }\n            },\n            {\n              "slots": [\n                5,\n                8\n              ],\n              "relevance": -0.4296875,\n              "yes_no": {\n                "canonical": [\n                  21.46875,\n                  22.4375\n                ],\n                "reversed": [\n                  21.84375,\n                  21.734375\n                ]\n              }\n            },\n            {\n              "slots": [\n                10,\n                13\n              ],\n              "relevance": 0.1328125,\n              "yes_no": {\n                "canonical": [\n                  21.875,\n                  22.046875\n                ],\n                "reversed": [\n                  21.984375,\n                  21.546875\n                ]\n              }\n            },\n            {\n              "slots": [\n                16,\n                19\n              ],\n              "relevance": 0.375,\n              "yes_no": {\n                "canonical": [\n                  21.828125,\n                  21.765625\n                ],\n                "reversed": [\n                  22.046875,\n                  21.359375\n                ]\n              }\n            },\n            {\n              "slots": [\n                21,\n                24\n              ],\n              "relevance": 0.125,\n              "yes_no": {\n                "canonical": [\n                  21.703125,\n                  22.0\n                ],\n                "reversed": [\n                  21.953125,\n                  21.40625\n                ]\n              }\n            },\n            {\n              "slots": [\n                26,\n                29\n              ],\n              "relevance": 0.5078125,\n              "yes_no": {\n                "canonical": [\n                  22.046875,\n                  21.8125\n                ],\n                "reversed": [\n                  22.25,\n                  21.46875\n                ]\n              }\n            },\n            {\n              "slots": [\n                32,\n                35\n              ],\n              "relevance": 0.84375,\n              "yes_no": {\n                "canonical": [\n                  22.1875,\n                  21.546875\n                ],\n                "reversed": [\n                  22.359375,\n                  21.3125\n                ]\n              }\n            },\n            {\n              "slots": [\n                37,\n                40\n              ],\n              "relevance": 0.8828125,\n              "yes_no": {\n                "canonical": [\n                  22.265625,\n                  21.609375\n                ],\n                "reversed": [\n                  22.375,\n                  21.265625\n                ]\n              }\n            },\n            {\n              "slots": [\n                42,\n                45\n              ],\n              "relevance": 0.546875,\n              "yes_no": {\n                "canonical": [\n                  21.984375,\n                  21.703125\n                ],\n                "reversed": [\n                  22.171875,\n                  21.359375\n                ]\n              }\n            },\n            {\n              "slots": [\n                48,\n                51\n              ],\n              "relevance": -0.5390625,\n              "yes_no": {\n                "canonical": [\n                  21.75,\n                  22.6875\n                ],\n                "reversed": [\n                  21.96875,\n                  22.109375\n                ]\n              }\n            },\n            {\n              "slots": [\n                53,\n                56\n              ],\n              "relevance": -0.390625,\n              "yes_no": {\n                "canonical": [\n                  21.859375,\n                  22.546875\n                ],\n                "reversed": [\n                  22.03125,\n                  22.125\n                ]\n              }\n            },\n            {\n              "slots": [\n                59,\n                62\n              ],\n              "relevance": -0.1875,\n              "yes_no": {\n                "canonical": [\n                  22.0,\n                  22.4375\n                ],\n                "reversed": [\n                  22.109375,\n                  22.046875\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                32,\n                35\n              ],\n              [\n                37,\n                40\n              ],\n              [\n                42,\n                45\n              ]\n            ],\n            "constrained_windows": [\n              [\n                32,\n                35\n              ],\n              [\n                37,\n                40\n              ],\n              [\n                42,\n                45\n              ]\n            ],\n            "indices": [\n              786,\n              822,\n              906,\n              942,\n              1026,\n              1062\n            ],\n            "scanned_indices": 156\n          }\n        },\n        "b": {\n          "coarse": {\n            "indices": [\n              0,\n              24,\n              48,\n              72,\n              96,\n              120,\n              144,\n              168,\n              192,\n              216,\n              240,\n              264,\n              288,\n              312,\n              336,\n              360,\n              384,\n              408,\n              432,\n              456,\n              480,\n              504,\n              528,\n              552,\n              576,\n              600,\n              624,\n              648,\n              672,\n              696,\n              720,\n              744,\n              768,\n              792,\n              816,\n              840,\n              864,\n              888,\n              912,\n              936,\n              960,\n              984,\n              1008,\n              1032,\n              1056,\n              1080,\n              1104,\n              1128,\n              1152,\n              1176,\n              1200,\n              1224,\n              1248,\n              1272,\n              1296,\n              1320,\n              1344,\n              1368,\n              1392,\n              1416,\n              1440,\n              1464,\n              1488,\n              1512,\n              1536,\n              1560,\n              1584\n            ],\n            "hashes": [\n              "010ded0367a7efddd8a6a9035597984d1b82d616e38204dc874dd00e040dc01a",\n              "d465e54b4e6b8042f59342b6ec09425580b369b603c8d0bbfb370ce3cd68cac5",\n              "e1c7840c75bcd121687a5e4bd365e7001e017e30946fd3103d53044af025884d",\n              "72f3d812035f5555894fa5d317cf0f5e187a08b24a9ff68df21767b49f6e9d75",\n              "52d4f0c163acf3122f82dd9cc7a2297c9d287cd67a11e43d8daf818bc476137f",\n              "fa7c92a8c7855f7901f38aca034f18d272eba6387aa89126712672d7a4bbb698",\n              "b7c4ef7d083c6033a2f471cd9f90ab3c09ec082a5cdcb8f44c1493dc6b1df7af",\n              "df6bef0fefc11ce61b5e13273f436bad482180a5c6e7eaaa46adf3ca6d1bfdb5",\n              "1dfd3b952e8074736a3509a6fb6c43d2a15f5d6aaa4f0079ce270682c8dfe37a",\n              "83c126182a9b4320e07418bc49de69471f347fb2ddc904e85cf57a03e30c8d97",\n              "96c6b7a8926428e037bd2f1a12325b8949f0e0698d010e4425e1c97b6aa1fd34",\n              "9ddedca03df64c0b7e44acabb394d80c8651a4c30cb784cda8449934b2a93e4e",\n              "93f1effd7a760288340d184a9331b7d3df86345f1fae6cdd856b51e9866d70aa",\n              "fa191517c9e22aa5f582c815649dbdcf3d7c2f1bcd185f9416c2d238f71ddf87",\n              "3c893683685252c2dd9b32580885c42384878ffa084578d9877407beaacee864",\n              "ce0edeca4e80cb917cbe0bf9d0751e32a00d02a40a992aa3cacc174f9e2f921d",\n              "3da08d403032bc994d956c071255a722b8d99a468b8f3c5d928e5dd9b919bc0f",\n              "164057a8b217ede0cca0e7fcc3b7ae27722795d87fc667364d246e08a877aff1",\n              "83f3dbb5ed81b16f6328cc40526ccd8a309c32c910ea808dfc8de4c35b636e7e",\n              "e9de1f77c451d0abdef79a6e7fe2acf202da2da6fb0a7dd45379fda3bb7b078c",\n              "e71f061df5e7b60f219199fc34d96bd2c68960aea8d1d962c533eb04246063d2",\n              "17bcd8aaac01a76a5600d78daab26b3380e8581ed3c897809e17933b6aa3c86f",\n              "a1c6cd9efbbe1c9ae807d4cc309773c2fb1e1855eebb48682dba07ee6d6d5213",\n              "99a238f8d52ca656c21ed2ab67333a0ec9758725955c10c5173147c35075f42a",\n              "877103ffbac59a367740a25a4c8c8907c988cf10dc04a415dd11e48e2e372084",\n              "33e868c301dfc5477b0493056c2a86c499da7012003782326100c5dfdbdb2f3d",\n              "6e33243e16975e65e6b661417b57bc46b0c2e77b25041cf68923aeecb2988a10",\n              "d7856893f900340c415cb9263fba7b4aff541a99766f87894dffd3d7420ac98b",\n              "056fbe01e3a8e352a44972844d41f5241c7df00806733dd5f4c20b7e7acfe2c8",\n              "e9cefc6db5004b6e01154ae61ac93812c3cc0d2813397c5cd98cfee4b83bce7e",\n              "5bf25d77f7bff8e6bb767a4c1e127eb7556c9447328b48201b89c31915b522be",\n              "8786afd84e899868f31d768a0e2dca9bd51e2e5ac03122211f7c19f750dcce96",\n              "321bace629d850b589580b9963bba87a30a43bfe717934569c1d643c522614ec",\n              "e029bcd4016852960beb9a3f07584f01c49971f1ce830649ffc3561f51a0b47f",\n              "f213739918ebb2649dd97a375b8f26f99a4911336f38c95d43b7184fb447fea2",\n              "ec925f9fda9a3f168a65eafeffc5a4b55e66f8d10864e9603b57e61d6cb61c3c",\n              "0e27863181f59832065a487a04f61bda9516d95485929187f750ca03318ee3b8",\n              "8bba6fb74992d1dc8ed8bd551c033251a61cca041a45f284014e40343eefd2cc",\n              "fd453ab40a34217d3faf9a669e593b5a34e055943dd695092c360f26a8df76f1",\n              "96b1e80c00f5b13cc5af6bd32a60280aee93cbf40d4999080891ac44b7f82787",\n              "a6341d3c97df58ddc714cdde3fdfcbb8d1d1ddb9f1dee59e2c4282bf4c86418f",\n              "0fc810e8d8d5680814ced0187a473046eb897d738b8b1f1ed69f320d63361291",\n              "b9e675e8a3c406baf20a42cc1634f101b6f10f82b0560ea05b9b76c4a0ccde15",\n              "917d59da2f65eec6e03a1981f6f58ca249a3c0c8bdc482ab806041d3a0d158cc",\n              "25fd3a8e28eb437e5171d9fff6cea0e0b5fc19cfba3a3f0d47cf094fc000eea6",\n              "722e47fdb243bce61192039c30c77d4f886e516872977e06675a384836c163b1",\n              "99824eb673a84453461cce366e14ac9c284f2f63070843614841801ac69867b1",\n              "38b6d8bedad6ce59741412ab98a396a1b8d2684eacc72a34d3c4762987bf0762",\n              "ff4f0c0bcd01c5617b2d476b36253c1f2fceaca00b743db92f08799a66dc173c",\n              "68a69c8225695dad86e82ca4e064d1b997776404231a22dbd5092a19a585ca1b",\n              "ac4bdaca14a029a651c2373fa156aa557284828164f6d316c1118716266de0a0",\n              "99f0ce76ae123bf4a6d0704766a12b6f594a18d9cae72d869da374fd860db8f9",\n              "56cde5d448da438d6b257f99e5e9b529647a01f5b598e95a74ac2e4112fa0448",\n              "e8794554263e63206e66e358d0f7ba42e584e0668c90a47886cb91b803801a7c",\n              "bcca351806231f25339f62fe4814a31571324ea72a0563c9c0eee5d1d918ed8d",\n              "36f7b23877d32039b40cf88407732d2edf274b3a52b4628a18c150c667512b4b",\n              "8be277ccc53257208d81d8f9629919f9a2db307180f48835fe63c9558d11fa0e",\n              "f20a682d32dccb66d4f59d6fb31148388f6054f99d7192e421a8fc7e58a0d835",\n              "13f98d416623cff7d1de30dfee77fd0a094d3d88385847117614b161914bd7ae",\n              "22fa12976a15061694e24614e3942496a910521adc26d82b27e4c0a1b732e2e2",\n              "8465e4ae653d14cc479dbc9de665f2bb505bf08333357b07352e932b37b61547",\n              "ee298333dee53864b18a01b58a717e7e78a78c237427c682e4e71dbd5c0f2fff",\n              "4a9bc00b08403c06f5123105775c7f0624a7a4451cfb5b1cbd75c0deebe6ca2a",\n              "3de34c51691676b656a329fc130ea17cb635185f9e070e2e3fcb344e47007344",\n              "a060d1d1dedc354281aa7332f8623d1d2bdc32e9186710af21f8e7c78e4b80db",\n              "3fd333ec7946cfac3131f26e6d2ffcb6af45094b1ecae0c1932a0ce1515d05c3",\n              "984fd146f6cb4775df93ee3a4a7995619abc3ad686d6f5616b2f36594145d503"\n            ],\n            "slots": [\n              0,\n              12,\n              24,\n              37,\n              49,\n              62\n            ]\n          },\n          "direct": {\n            "coarse_uniform": {\n              "canonical": {\n                "logits": [\n                  20.3125,\n                  22.46875\n                ],\n                "prediction": "B",\n                "lm_tokens": 2063,\n                "visual_tokens": 1960,\n                "prompt_sha256": "8454c18515049578828ceb0abf747cbd2d3b147467ae4e94c40b05c8df2e953b"\n              },\n              "reversed": {\n                "logits": [\n                  20.8125,\n                  22.46875\n                ],\n                "prediction": "B",\n                "lm_tokens": 2063,\n                "visual_tokens": 1960,\n                "prompt_sha256": "21e46534011b13f3b7b8c51d3262080017e5d9f90773e42ef65da8f5cf278f6d"\n              }\n            },\n            "local_uniform": {\n              "canonical": {\n                "logits": [\n                  20.265625,\n                  22.375\n                ],\n                "prediction": "B",\n                "prompt_sha256": "8454c18515049578828ceb0abf747cbd2d3b147467ae4e94c40b05c8df2e953b",\n                "lm_tokens": 2063,\n                "visual_tokens": 1960\n              },\n              "reversed": {\n                "logits": [\n                  20.765625,\n                  22.390625\n                ],\n                "prediction": "B",\n                "prompt_sha256": "21e46534011b13f3b7b8c51d3262080017e5d9f90773e42ef65da8f5cf278f6d",\n                "lm_tokens": 2063,\n                "visual_tokens": 1960\n              }\n            }\n          },\n          "local": {\n            "indices": [\n              504,\n              510,\n              516,\n              522,\n              528,\n              534,\n              540,\n              546,\n              552,\n              558,\n              564,\n              570,\n              576,\n              768,\n              774,\n              780,\n              786,\n              792,\n              798,\n              804,\n              810,\n              816,\n              822,\n              828,\n              834,\n              840,\n              888,\n              894,\n              900,\n              906,\n              912,\n              918,\n              924,\n              930,\n              936,\n              942,\n              948,\n              954,\n              960\n            ],\n            "hashes": [\n              "17bcd8aaac01a76a5600d78daab26b3380e8581ed3c897809e17933b6aa3c86f",\n              "d16ae202be6d4338aae6aa8f68ce39c513e194808b5b8ace6868381a8a335c7d",\n              "c0aa56d77092207bb28291cc397f829c52e27c105130dd02d6d68b574583dadd",\n              "b7455897840d06a093a8d2cc5b986a5c8e2d7074bee09e11f333dfbc2ec9032b",\n              "a1c6cd9efbbe1c9ae807d4cc309773c2fb1e1855eebb48682dba07ee6d6d5213",\n              "e30d46fa64e71f883984a12ce76ddb83eb6b218f4c443e399107304fb3d90ef8",\n              "a0d0961c89d74aa3910b4ef26566832bb8637a828b7109065ea1722173240267",\n              "53648613b6d0e5f38972c3bc1273c5f37254a012f65e7355f7a0bd702233fba4",\n              "99a238f8d52ca656c21ed2ab67333a0ec9758725955c10c5173147c35075f42a",\n              "61ed1014882d5e696306b9f0c2825afc083d693e77e4a8ae091906bdb5ef39fa",\n              "74c1c74d7d0c211b14784a03b7b18d0d6d2d3f49f8323a601e2ea9878bad3d96",\n              "53fed91aa30d1d8bd30fa2b145287fc4f18ae8e2ffbeadf99fbebe47aebe2c71",\n              "877103ffbac59a367740a25a4c8c8907c988cf10dc04a415dd11e48e2e372084",\n              "321bace629d850b589580b9963bba87a30a43bfe717934569c1d643c522614ec",\n              "5d215a6f008948c54ff47e62f42a7b7f05afa3f738de82d09cdacc5de161dcda",\n              "49da6833928d8e9f6195447489ca81017ddb3985f910aaafbfd31f9725dfc7de",\n              "11784ad25f5d559f12962de002c05224d6f6beaa20ecc512ad79717481f0db72",\n              "e029bcd4016852960beb9a3f07584f01c49971f1ce830649ffc3561f51a0b47f",\n              "ddc53d027dbec85f97dd58d081988592d3de63461e4eef9fa79f24d35ea5a711",\n              "570f0730b98e6999068776206205f29a9308f700e66253059dcfe0d50b50b755",\n              "f1eb35615f4578436c51578456d8bfffc926b901cf1b58ed957f43b8ac8e63b1",\n              "f213739918ebb2649dd97a375b8f26f99a4911336f38c95d43b7184fb447fea2",\n              "4ada5f83c13c9fb55a37a877f9b9c129190845c46445ca4f8dd054c96c886b52",\n              "ee7de8ad1f318a30632925f0747c1addbc1c0bc32a5d825c92464dbf9c4732e2",\n              "4f3021264c0e11efe8313c2b4cf53b765c212fc162f5536904bad46d594eee2a",\n              "ec925f9fda9a3f168a65eafeffc5a4b55e66f8d10864e9603b57e61d6cb61c3c",\n              "8bba6fb74992d1dc8ed8bd551c033251a61cca041a45f284014e40343eefd2cc",\n              "208503d190ec1d24361047e97c273f66b6e7554e2fa250211f936dbf7c6009f7",\n              "24364281099cf004ca5ec2236128dc5cdc6d9b865519f1253fecc6b1b4f55f37",\n              "9ec61497a42f3891d9546bd9b9add1e082e9472a32ca8eb18c703df1ac1bf1ba",\n              "fd453ab40a34217d3faf9a669e593b5a34e055943dd695092c360f26a8df76f1",\n              "eabf1d2dc027f76c0a144f9698e0e9b0c4faac4c14538690ab0379f8f566f1ee",\n              "06cd9690c41bff58b07e902117d74b19742eecda93f8ee7c9cf5d4c2350e86b3",\n              "b06861538fe4ed41fdf0f135fb0b46542be35b4bd73327f9f40cc4930604e5c7",\n              "96b1e80c00f5b13cc5af6bd32a60280aee93cbf40d4999080891ac44b7f82787",\n              "2a2f324042648087fe0df9d821aeb7bc2e05c9ef27c5df2ac2bda9271a8ec645",\n              "57eb0e0ce8b9b0cc01729f7c5f62a996acf9a5094d5d69342a9f9fd7987ba63f",\n              "333c75a8f59c3f6661b17ccc07a622e74bb062ff04c1d5d2fa01ec156a3fc313",\n              "a6341d3c97df58ddc714cdde3fdfcbb8d1d1ddb9f1dee59e2c4282bf4c86418f"\n            ],\n            "slots": [\n              3,\n              9,\n              16,\n              22,\n              29,\n              35\n            ]\n          },\n          "candidates": [\n            {\n              "slots": [\n                0,\n                3\n              ],\n              "relevance": 0.2421875,\n              "yes_no": {\n                "canonical": [\n                  21.703125,\n                  21.78125\n                ],\n                "reversed": [\n                  21.90625,\n                  21.34375\n                ]\n              }\n            },\n            {\n              "slots": [\n                5,\n                8\n              ],\n              "relevance": -0.1640625,\n              "yes_no": {\n                "canonical": [\n                  21.875,\n                  22.5\n                ],\n                "reversed": [\n                  22.140625,\n                  21.84375\n                ]\n              }\n            },\n            {\n              "slots": [\n                10,\n                13\n              ],\n              "relevance": 0.5703125,\n              "yes_no": {\n                "canonical": [\n                  22.109375,\n                  21.78125\n                ],\n                "reversed": [\n                  22.234375,\n                  21.421875\n                ]\n              }\n            },\n            {\n              "slots": [\n                16,\n                19\n              ],\n              "relevance": 0.4765625,\n              "yes_no": {\n                "canonical": [\n                  21.875,\n                  21.65625\n                ],\n                "reversed": [\n                  22.109375,\n                  21.375\n                ]\n              }\n            },\n            {\n              "slots": [\n                21,\n                24\n              ],\n              "relevance": 0.6015625,\n              "yes_no": {\n                "canonical": [\n                  21.953125,\n                  21.640625\n                ],\n                "reversed": [\n                  22.1875,\n                  21.296875\n                ]\n              }\n            },\n            {\n              "slots": [\n                26,\n                29\n              ],\n              "relevance": 0.2734375,\n              "yes_no": {\n                "canonical": [\n                  21.953125,\n                  22.0\n                ],\n                "reversed": [\n                  22.15625,\n                  21.5625\n                ]\n              }\n            },\n            {\n              "slots": [\n                32,\n                35\n              ],\n              "relevance": 0.84375,\n              "yes_no": {\n                "canonical": [\n                  22.1875,\n                  21.546875\n                ],\n                "reversed": [\n                  22.359375,\n                  21.3125\n                ]\n              }\n            },\n            {\n              "slots": [\n                37,\n                40\n              ],\n              "relevance": 0.8828125,\n              "yes_no": {\n                "canonical": [\n                  22.265625,\n                  21.609375\n                ],\n                "reversed": [\n                  22.375,\n                  21.265625\n                ]\n              }\n            },\n            {\n              "slots": [\n                42,\n                45\n              ],\n              "relevance": 0.546875,\n              "yes_no": {\n                "canonical": [\n                  21.984375,\n                  21.703125\n                ],\n                "reversed": [\n                  22.171875,\n                  21.359375\n                ]\n              }\n            },\n            {\n              "slots": [\n                48,\n                51\n              ],\n              "relevance": -0.5390625,\n              "yes_no": {\n                "canonical": [\n                  21.75,\n                  22.6875\n                ],\n                "reversed": [\n                  21.96875,\n                  22.109375\n                ]\n              }\n            },\n            {\n              "slots": [\n                53,\n                56\n              ],\n              "relevance": -0.390625,\n              "yes_no": {\n                "canonical": [\n                  21.859375,\n                  22.546875\n                ],\n                "reversed": [\n                  22.03125,\n                  22.125\n                ]\n              }\n            },\n            {\n              "slots": [\n                59,\n                62\n              ],\n              "relevance": -0.1875,\n              "yes_no": {\n                "canonical": [\n                  22.0,\n                  22.4375\n                ],\n                "reversed": [\n                  22.109375,\n                  22.046875\n                ]\n              }\n            }\n          ],\n          "boundary": {\n            "original_windows": [\n              [\n                21,\n                24\n              ],\n              [\n                32,\n                35\n              ],\n              [\n                37,\n                40\n              ]\n            ],\n            "constrained_windows": [\n              [\n                21,\n                24\n              ],\n              [\n                32,\n                35\n              ],\n              [\n                37,\n                40\n              ]\n            ],\n            "indices": [\n              522,\n              558,\n              786,\n              822,\n              906,\n              942\n            ],\n            "scanned_indices": 156\n          }\n        }\n      },\n      "media": {\n        "a": {\n          "file": "videos/twinpast_0021/A.mp4",\n          "bytes": 24797005,\n          "sha256": "96f47091116bbad5d8bc4c2dda471255372950e5565ff063d57f09e735a812f6"\n        },\n        "b": {\n          "file": "videos/twinpast_0021/B.mp4",\n          "bytes": 24575581,\n          "sha256": "6d2a1d5f22652301859dffbcadba8e970d597538ddc75007fba1ff413b63fbf3"\n        }\n      }\n    }\n  ],\n  "model_revision": "0d50680527681998e456c7b78950205bedd8a068",\n  "video_processor": {\n    "_processor_class": "LlavaOnevisionProcessor",\n    "do_pad": true,\n    "image_grid_pinpoints": [\n      [\n        384,\n        384\n      ],\n      [\n        384,\n        768\n      ],\n      [\n        384,\n        1152\n      ],\n      [\n        384,\n        1536\n      ],\n      [\n        384,\n        1920\n      ],\n      [\n        384,\n        2304\n      ],\n      [\n        768,\n        384\n      ],\n      [\n        768,\n        768\n      ],\n      [\n        768,\n        1152\n      ],\n      [\n        768,\n        1536\n      ],\n      [\n        768,\n        1920\n      ],\n      [\n        768,\n        2304\n      ],\n      [\n        1152,\n        384\n      ],\n      [\n        1152,\n        768\n      ],\n      [\n        1152,\n        1152\n      ],\n      [\n        1152,\n        1536\n      ],\n      [\n        1152,\n        1920\n      ],\n      [\n        1152,\n        2304\n      ],\n      [\n        1536,\n        384\n      ],\n      [\n        1536,\n        768\n      ],\n      [\n        1536,\n        1152\n      ],\n      [\n        1536,\n        1536\n      ],\n      [\n        1536,\n        1920\n      ],\n      [\n        1536,\n        2304\n      ],\n      [\n        1920,\n        384\n      ],\n      [\n        1920,\n        768\n      ],\n      [\n        1920,\n        1152\n      ],\n      [\n        1920,\n        1536\n      ],\n      [\n        1920,\n        1920\n      ],\n      [\n        1920,\n        2304\n      ],\n      [\n        2304,\n        384\n      ],\n      [\n        2304,\n        768\n      ],\n      [\n        2304,\n        1152\n      ],\n      [\n        2304,\n        1536\n      ],\n      [\n        2304,\n        1920\n      ],\n      [\n        2304,\n        2304\n      ]\n    ],\n    "image_processor_type": "LlavaOnevisionVideoProcessor",\n    "do_resize": true,\n    "size": {\n      "height": 384,\n      "width": 384\n    },\n    "resample": 3,\n    "do_rescale": true,\n    "rescale_factor": 0.00392156862745098,\n    "do_normalize": true,\n    "image_mean": [\n      0.5,\n      0.5,\n      0.5\n    ],\n    "image_std": [\n      0.5,\n      0.5,\n      0.5\n    ],\n    "do_convert_rgb": true\n  },\n  "media_source": {\n    "manifest": {\n      "dataset": "younght/TwinPast",\n      "revision": "c1f1cbab68bfda2982565c0491f1a97474b73558"\n    }\n  },\n  "source_archive_sha256": "c3848d6a96dd1fff1fce4c8465fc38c04ee22c7cd18afdea6697cc464f067201",\n  "cut_threshold": 0.2,\n  "cut_fps": 4,\n  "cpu_audit_sha256": "96a88c54cef16feee2acd7ca0d61fa586caee75144d6d893ed0052c8d16a3280"\n}\n', 'provenance/boundary_runner.py': '"""Evaluate the frozen TwinPast scene-boundary policy against matched baselines."""\nimport argparse\nimport json\nfrom pathlib import Path\nfrom statistics import median\nimport time\n\nimport numpy as np\nfrom decord import VideoReader\n\nfrom readout_runner import REPLAY_ATOL, acquire, auto, base, choose\nfrom twinpast_boundary_policy import FPS, THRESHOLD, pick, screen_candidates, select\n\nARTIFACT = "HERMES_twinpast_boundary_evidence.zip"\nMETHODS = ("coarse_uniform", "local_uniform", "boundary")\nREPLAY_METHODS = METHODS[:2]\n\n\ndef timed(operation, device):\n    import torch\n\n    if device.type == "cuda":\n        torch.cuda.synchronize(device)\n    start = time.perf_counter()\n    value = operation()\n    if device.type == "cuda":\n        torch.cuda.synchronize(device)\n    return value, time.perf_counter() - start\n\n\ndef retrieval(model, system, coarse, question, candidates):\n    import torch\n\n    probe = ("Does this earlier excerpt contain visual evidence needed to answer the following "\n             "question about the specified event, state, or entity? " + question)\n    prompts = {order: base.make_prompt(model, probe, ["Yes", "No"], order, True) for order in base.ORDERS}\n    ranked, checks = [], []\n    for candidate in candidates:\n        start, end = candidate["slots"]\n        visual = torch.cat((coarse[start], coarse[end]), dim=1)\n        result = {}\n        for order, (ids, labels, perm, _) in prompts.items():\n            values, _ = base.score(model, system, ids, labels, visual)\n            canonical = [0.0] * len(perm)\n            for i, value in zip(perm, values.tolist(), strict=True):\n                canonical[i] = value\n            result[order] = canonical\n            expected = candidate["yes_no"][order]\n            checks.append(max(abs(a-b) for a, b in zip(canonical, expected, strict=True)))\n        relevance = sum(v[0]-v[1] for v in result.values()) / 2\n        ranked.append(dict(slots=candidate["slots"], relevance=relevance, yes_no=result))\n    return ranked, max(checks)\n\n\ndef boundary_inputs(path, candidates, cutoff):\n    reader = VideoReader(str(path), num_threads=1)\n    stamps = np.asarray(reader.get_frame_timestamp(list(range(len(reader)))), dtype=np.float64)\n    starts = stamps[:, 0] - stamps[0, 0]\n    windows, scanned = screen_candidates(reader, starts, candidates, cutoff)\n    original, constrained = select(windows)\n    indices = pick(constrained, True)\n    frames = reader.get_batch(indices).asnumpy()\n    return dict(original_windows=[list(w["window"]) for w in original],\n                constrained_windows=[list(w["window"]) for w in constrained], indices=indices,\n                hashes=[base.sha(frame.tobytes()) for frame in frames],\n                timestamps_s=starts[indices].tolist(), scanned_indices=scanned, frames=frames)\n\n\ndef compare(case, side, model, system, path):\n    import torch\n\n    archived = case["branches"][side]\n    coarse, local = archived["coarse"], archived["local"]\n    seconds = {}\n    bank, seconds["coarse_decode_encode"] = timed(\n        lambda: auto.encode_frames(model, base.decode(\n            path, dict(source_frames=coarse["indices"], decoded_rgb_sha256=coarse["hashes"])\n        )), model.device,\n    )\n    recent = bank[-4:]\n    (candidates, retrieval_delta), seconds["question_relevance"] = timed(\n        lambda: retrieval(model, system, bank, case["question"], archived["candidates"]), model.device\n    )\n    if retrieval_delta > REPLAY_ATOL:\n        raise ValueError(f"Retrieval replay differs for {case[\'pair_id\']}/{side}: {retrieval_delta}")\n    cut, seconds["boundary_scan_and_select"] = timed(\n        lambda: boundary_inputs(path, candidates, coarse["indices"][-4]), model.device\n    )\n    frozen = archived["boundary"]\n    for field in ("original_windows", "constrained_windows", "indices"):\n        if cut[field] != frozen[field]:\n            raise ValueError(\n                f"Frozen selection differs for {case[\'pair_id\']}/{side}/{field}: "\n                f"expected {frozen[field]}, got {cut[field]}"\n            )\n    if cut["scanned_indices"] != frozen["scanned_indices"]:\n        raise ValueError(f"Boundary probe frame count differs for {case[\'pair_id\']}/{side}")\n\n    def encode_history(frames):\n        return torch.cat(auto.encode_frames(model, frames)+recent, dim=1)\n\n    def local_input():\n        frames = base.decode(path, dict(source_frames=local["indices"], decoded_rgb_sha256=local["hashes"]))\n        return encode_history(frames[local["slots"]])\n\n    visuals = {}\n    for method, operation in (\n        ("coarse_uniform", lambda: torch.cat([bank[i] for i in coarse["slots"]]+recent, dim=1)),\n        ("local_uniform", local_input),\n        ("boundary", lambda: encode_history(cut["frames"])),\n    ):\n        visuals[method], seconds[f"{method}_history"] = timed(operation, model.device)\n    assert len({tuple(item.shape) for item in visuals.values()}) == 1\n    rows, failures = {}, []\n    for order in base.ORDERS:\n        rows[order] = {}\n        for method in METHODS:\n            row, elapsed = timed(lambda: choose(model, system, visuals[method], case["question"],\n                                                case["options"], order), model.device)\n            row["synchronized_answer_s"] = elapsed\n            if method in REPLAY_METHODS:\n                expected = archived["direct"][method][order]\n                delta = max(abs(a-b) for a, b in zip(row["logits"], expected["logits"], strict=True))\n                row["replay_max_abs"] = delta\n                if delta > REPLAY_ATOL or any(row[k] != expected[k] for k in\n                                               ("prediction", "prompt_sha256", "visual_tokens", "lm_tokens")):\n                    failures.append(dict(frame_set=method, order=order, expected=expected, actual=row))\n            rows[order][method] = row\n    cut.pop("frames")\n    return dict(answers=rows, stages_s=seconds, boundary=cut, retrieval_max_abs=retrieval_delta,\n                replay_failures=failures)\n\n\ndef summarize(blocks, targets):\n    methods = {}\n    for method in METHODS:\n        robust, correct, totals = [], 0, []\n        extra = {"coarse_uniform": (), "local_uniform": ("question_relevance",),\n                 "boundary": ("question_relevance", "boundary_scan_and_select")}[method]\n        for block in blocks:\n            pair = block["pair_id"]\n            pair_correct = 0\n            for side, trace in block["branches"].items():\n                scores = [trace["answers"][order][method] for order in base.ORDERS]\n                pair_correct += sum(row["prediction"] == targets[pair][f"answer_{side}"] for row in scores)\n                stages = trace["stages_s"]\n                total = sum(stages[name] for name in ("coarse_decode_encode", f"{method}_history", *extra))\n                totals.append(total + sum(row["synchronized_answer_s"] for row in scores) / len(scores))\n            correct += pair_correct\n            if pair_correct == 4:\n                robust.append(pair)\n        methods[method] = dict(robust_pairs=robust, correct_branch_orders=correct,\n                               median_stage_sum_per_branch_s=median(totals))\n    boundary = set(methods["boundary"]["robust_pairs"])\n    paired = {m: dict(recoveries=sorted(boundary-set(methods[m]["robust_pairs"])),\n                      regressions=sorted(set(methods[m]["robust_pairs"])-boundary)) for m in REPLAY_METHODS}\n    primary = paired["coarse_uniform"]\n    return dict(pairs=len(blocks), scores=len(blocks)*2*len(base.ORDERS)*len(METHODS), methods=methods,\n                boundary_against=paired, development_criterion_met=len(blocks)==8 and bool(primary["recoveries"])\n                and not primary["regressions"])\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work / "provenance"\n    spec = json.loads((provenance / "boundary_cases.json").read_bytes())\n    assert args.model.name == spec["model_revision"]\n    assert len(spec["cases"]) == 8 and spec["cut_threshold"] == THRESHOLD and spec["cut_fps"] == FPS\n    paths = acquire(args, spec)\n    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.dtype == torch.float16 and {str(p.device) for p in model.parameters()} == {"cuda:0"}\n    assert json.loads(json.dumps(processor.video_processor.to_dict())) == spec["video_processor"]\n    exports = work / "exports"\n    config = dict(gpu=torch.cuda.get_device_name(), torch=torch.__version__, cuda=torch.version.cuda,\n                  model_revision=args.model.name, replay_atol=REPLAY_ATOL,\n                  refinement_archive_sha256=spec["source_archive_sha256"], cpu_audit_sha256=spec["cpu_audit_sha256"])\n    base.atomic(exports / "run_config.json", config)\n    blocks = []\n    with torch.inference_mode():\n        system = model.language_model.get_input_embeddings()(torch.as_tensor(model.init_prompt_ids, device=model.device))\n        for i, case in enumerate(spec["cases"], 1):\n            pair = case["pair_id"]\n            block = dict(pair_id=pair, branches={side: compare(case, side, model, system, paths[pair, side])\n                                                  for side in "ab"})\n            base.atomic(work / "checkpoints" / f"{pair}.json", block)\n            blocks.append(block)\n            print(f"BOUNDARY_PAIR {i}/8 {pair}", flush=True)\n    branches = [branch for block in blocks for branch in block["branches"].values()]\n    failures = [dict(pair_id=block["pair_id"], branch=side, **failure) for block in blocks\n                for side, branch in block["branches"].items() for failure in branch["replay_failures"]]\n    archived_scores = [result[m]["replay_max_abs"] for branch in branches for result in branch["answers"].values()\n                       for m in REPLAY_METHODS]\n    base.atomic(exports / "replay_gate.json", dict(scores=len(archived_scores), max_abs=max(archived_scores),\n                retrieval_max_abs=max(b["retrieval_max_abs"] for b in branches), failures=failures))\n    if failures:\n        raise RuntimeError("Archived answer replay failed. See exports/replay_gate.json.")\n    targets = json.loads((provenance / "evaluation_targets.json").read_bytes())\n    assert set(targets) == {block["pair_id"] for block in blocks}\n    report = summarize(blocks, targets)\n    assert report["scores"] == 96\n    base.atomic(exports / "summary.json", report)\n    base.atomic(work / "completion.json", dict(status="complete", pairs=len(blocks)))\n    print("TWINPAST_BOUNDARY_COMPLETE", json.dumps(report), flush=True)\n\n\ndef main():\n    from filelock import FileLock\n\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    work = args.work\n    with FileLock(str(work / "run.lock"), timeout=0):\n        for name in ("exports", "checkpoints"):\n            (work / name).mkdir(exist_ok=True)\n        spec = json.loads((work / "provenance/boundary_cases.json").read_bytes())\n        for case in spec["cases"]:\n            (work / "checkpoints" / f"{case[\'pair_id\']}.json").unlink(missing_ok=True)\n        for name in ("summary.json", "replay_gate.json", "run_config.json"):\n            (work / "exports" / name).unlink(missing_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(work)\n\n\nif __name__ == "__main__":\n    main()\n', 'provenance/twinpast_boundary_policy.py': '"""Shared annotation-blind TwinPast boundary ranking and sampling policy."""\nfrom itertools import combinations\n\nimport numpy as np\nfrom PIL import Image\n\nFPS = 4\nTHRESHOLD = 0.20\nUNIFORM_SLOTS = (3, 9)\n\n\ndef screen_candidates(reader, starts, candidates, first_recent):\n    windows = sorted((tuple(row["slots"]), row["relevance"]) for row in candidates)\n    grids = {key: np.arange(key[0]*FPS, key[1]*FPS+1) / FPS for key, _ in windows}\n    indices = {key: np.searchsorted(starts, grid, side="right") - 1 for key, grid in grids.items()}\n    assert all(len(row) == 13 and min(row) >= 0 and np.all(np.diff(row) > 0) for row in indices.values())\n    unique = sorted({int(i) for row in indices.values() for i in row})\n    assert unique[-1] < first_recent\n    tiny = {}\n    for offset in range(0, len(unique), 12):\n        group = unique[offset:offset+12]\n        frames = reader.get_batch(group).asnumpy()\n        for i, rgb in zip(group, frames, strict=True):\n            tiny[i] = np.asarray(Image.fromarray(rgb).resize((32, 18), Image.Resampling.BILINEAR), dtype=np.float32)\n    checked = []\n    for key, score in windows:\n        images = np.stack([tiny[i] for i in indices[key]])\n        delta = np.abs(np.diff(images, axis=0)).mean(axis=(1, 2, 3)) / 255\n        cut = int(delta.argmax())\n        checked.append(dict(window=key, relevance=score, indices=indices[key].tolist(),\n                            peak_delta=float(delta[cut]), pre_cut_slot=cut if delta[cut] >= THRESHOLD else None))\n    return checked, len(unique)\n\n\ndef select(candidates):\n    groups = [group for group in combinations(candidates, 3)\n              if all(left["window"][1] < right["window"][0] for left, right in zip(group, group[1:]))]\n    relevance = lambda group: sum(w["relevance"] for w in group)\n    original = max(groups, key=relevance)\n    with_cut = [group for group in groups if any(w["pre_cut_slot"] is not None for w in group)]\n    return original, max(with_cut, key=relevance) if with_cut else original\n\n\ndef pick(windows, preserve):\n    qualifying = [w for w in windows if w["pre_cut_slot"] is not None]\n    anchor = max(qualifying, key=lambda w: w["peak_delta"]) if preserve and qualifying else None\n    chosen = []\n    for window in windows:\n        slots = UNIFORM_SLOTS\n        if window is anchor:\n            slot = window["pre_cut_slot"]\n            farther = max(UNIFORM_SLOTS, key=lambda ordinary: abs(ordinary-slot))\n            slots = sorted((slot, farther))\n        chosen.extend(window["indices"][i] for i in slots)\n    assert len(chosen) == 6 and chosen == sorted(set(chosen))\n    return chosen\n'}
EXPECTED = {'pinned_source/inference/llavaov_hermes.py': '45fee483fbc37fe65433c84146eba42fc18dae40f6878c7a2f50fd8e55a387f0', 'pinned_source/inference/abstract_hermes.py': 'cbf62ad9f6dada32db598d7c37c27f995bbb37375e12a282a88cf4426f495dc0', 'pinned_source/inference/reindex_1d.py': '22ca0dee63cd7a4d00161f1fc0485143a57145862b081555598b904bbc95e0b0', 'provenance/twinpast_base.py': '5f4e15c54168502763fb8ad2054e6c701e7440a09295b01240ef05e3865066f0', 'provenance/history_only_runner.py': '7614a0f8f72dd1e3e92e7b0a9a7e8ed232a1ab6778fac9a910fd48171c70f281', 'provenance/backbone_runner.py': '6a6f51a49f1ac1ce2f978f615b6a51a2b26404ca60147cf0b74f2c3b8f187ce4', 'provenance/automatic_runner.py': '6a2f33bb3907aea3a0b2b5aa9bdcd4bd6cb20c000f77247b3d11e1907378ecd3', 'provenance/temporal_runner.py': '5d2532993e68d2b378c395e75fb099ec7afaf32f54ca380a0c52541e84d01ee1', 'provenance/evaluation_targets.json': 'a55b24b6c08cd152da56475e865390ea28d22b15f8441957153a95ae79ea2ec1', 'provenance/readout_runner.py': '7604f96c7ed0051adf5c6d066380c69019b550d8a06f4484365b944bd44b305b', 'provenance/boundary_cases.json': '673badf304045b8e6074f4576ad3edbab651e36188e370ea6c3f3e3774583c96', 'provenance/boundary_runner.py': 'd931f056e30cbe02e413c9de3e98199945a796219dc6553289d17633d768822a', 'provenance/twinpast_boundary_policy.py': 'b6204d798928d22842f2d332b12d1731291fb8c818c47415323cc6770d018ee1'}

for relative, contents in SOURCES.items():
    raw = contents.encode('utf-8')
    assert hashlib.sha256(raw).hexdigest() == EXPECTED[relative]
    if relative.endswith('.py'):
        ast.parse(contents, filename=relative)
    if not VALIDATE_ONLY:
        target = WORK / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.is_file() and target.read_bytes() != raw:
            raise ValueError(f'Existing file differs at {target}. Preserve it and choose a fresh WORK directory.')
        if not target.is_file():
            target.write_bytes(raw)
print('EMBEDDED_SOURCE_AND_MANIFEST_PASS', BASE_REVISION)


In [ ]:
ENV_PROBE = r"""
import sys, torch, transformers, tokenizers, huggingface_hub, pandas, decord, logzero
import json, importlib.metadata as md
assert sys.version_info[:2] == (3, 12)
assert torch.__version__ == '2.10.0+cu128'
assert transformers.__version__ == '4.45.0.dev0'
origin = json.loads(md.distribution('transformers').read_text('direct_url.json') or '{}')
assert '66bc4def9505fa7c7fe4aa7a248c34a026bb552b' in str(origin), 'Transformers source revision differs'
assert tokenizers.__version__ == '0.19.1' and huggingface_hub.__version__ == '0.34.4'
assert torch.cuda.is_available(), 'PyTorch cannot use CUDA; check the container driver'
from transformers.models.qwen2.modeling_qwen2 import Qwen2RotaryEmbedding
import inspect
assert 'seq_len' in inspect.signature(Qwen2RotaryEmbedding.forward).parameters
x = torch.ones((16, 16), device='cuda')
assert float((x @ x)[0, 0]) == 16
print('PINNED_ENV_PASS', torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
"""
if not VALIDATE_ONLY:
    for root in (REUSE, FALLBACK):
        candidate_python = root/'venv/bin/python'
        if candidate_python.is_file() and probe([candidate_python, '-c', ENV_PROBE], ENV).returncode == 0:
            PYTHON = candidate_python
            print('REUSING_VERIFIED_PINNED_ENVIRONMENT', PYTHON, flush=True)
            break
    PIP_ENV = package_environment(ENV)
    if not PYTHON.is_file() or probe([PYTHON, '-c', ENV_PROBE], ENV).returncode:
        if probe([sys.executable, '-m', 'uv', '--version'], ENV).returncode:
            attempt = probe([sys.executable, '-m', 'pip', 'install', '-q', 'uv', '-i',
                             'https://pypi.tuna.tsinghua.edu.cn/simple'], PIP_ENV)
            if attempt.returncode:
                logged([sys.executable, '-m', 'pip', 'install', '-q', 'uv', '-i',
                        'https://mirrors.aliyun.com/pypi/simple'], 'setup.log', PIP_ENV)
        UV = [sys.executable, '-m', 'uv']
        if not PYTHON.is_file():
            logged(UV + ['venv', VENV, '--python', '3.12', '--seed'], 'setup.log', ENV)
        def install(packages, extra=()):
            install_packages(UV, PYTHON, packages, ENV, extra)
        wheel_name = 'torch-2.10.0+cu128-cp312-cp312-manylinux_2_28_x86_64.whl'
        wheel = next((p for root in (REUSE, FALLBACK, WORK)
                      if (p := root/'downloads'/wheel_name).is_file() and zipfile.is_zipfile(p)), None)
        if wheel is None:
            wheel = WORK/'downloads'/wheel_name
            wheel.parent.mkdir(parents=True, exist_ok=True)
            partial = wheel.with_suffix('.whl.part')
            if not zipfile.is_zipfile(partial):
                url = ('https://download-r2.pytorch.org/whl/cu128/'
                       'torch-2.10.0%2Bcu128-cp312-cp312-manylinux_2_28_x86_64.whl')
                logged(['curl', '-fL', '-C', '-', '--connect-timeout', '20', '--speed-limit', '16384',
                        '--speed-time', '90', '--progress-bar', '-o', partial, url], 'torch_download.log', ENV)
            if not zipfile.is_zipfile(partial):
                raise ValueError(f'Incomplete PyTorch wheel: {partial}')
            partial.replace(wheel)
        install([str(wheel)])
        install(['numpy==1.26.4', 'tokenizers==0.19.1', 'huggingface-hub==0.34.4',
                 'accelerate==0.34.2', 'decord==0.6.0', 'logzero', 'pillow',
                 'sentencepiece', 'safetensors', 'packaging', 'filelock', 'regex',
                 'requests', 'pyyaml', 'tqdm', 'fsspec', 'protobuf', 'einops', 'psutil', 'pandas'])
        # Download GitHub source with turbo, then install the local archive using
        # direct package networking for any isolated build dependencies.
        tf_archive = WORK/'downloads'/f'transformers-{TF_SHA}.zip'
        tf_archive.parent.mkdir(parents=True, exist_ok=True)
        if not tf_archive.is_file() or not zipfile.is_zipfile(tf_archive):
            fetch_tf = r"""
import sys, urllib.request, zipfile, os, time
from pathlib import Path
dest = Path(sys.argv[2]); temp = dest.with_suffix('.download')
for attempt in range(4):
    try:
        with urllib.request.urlopen(sys.argv[1], timeout=180) as source, temp.open('wb') as target:
            while True:
                block = source.read(1024 * 1024)
                if not block: break
                target.write(block)
        assert zipfile.is_zipfile(temp), 'Transformers source is not a ZIP'
        os.replace(temp, dest)
        break
    except Exception:
        if attempt == 3: raise
        time.sleep(3 * (attempt + 1))
"""
            logged([sys.executable, '-c', fetch_tf,
                    f'https://codeload.github.com/huggingface/transformers/zip/{TF_SHA}', tf_archive],
                   'source_download.log', ENV)
        install([str(tf_archive)], ['--no-deps'])
    logged([PYTHON, '-c', ENV_PROBE], 'environment.log', ENV)
    ENV['PYTHONPATH'] = str(REPO) + os.pathsep + str(REPO/'scripts')
    logged([PYTHON, '-m', 'pip', 'freeze'], 'environment_freeze.txt', ENV)


In [ ]:
# Standalone notebook cell. Uses WORK, ENV, MODEL_ID, MODEL_SHA and logged from setup.
import hashlib
import json
import re
import shutil
from pathlib import Path

SHARDS = (
    ("model-00001-of-00004.safetensors", 4911200360, "ad4cb27d2a3f6e987147619cf2508a5bbb1b6b4c2d5533501fcc4a287a90e5af"),
    ("model-00002-of-00004.safetensors", 4991497664, "cfaed9e4ba187283a936fd6df5d55148a6de5a8416f1279778b364232c05f39a"),
    ("model-00003-of-00004.safetensors", 4932752752, "86cb0bb7f7d57873b27f16fd66b54be71055e582a48ce26e30884bfb209dffd4"),
    ("model-00004-of-00004.safetensors", 1226266240, "a2d1e40e47e9b524efcbbcab68e97fe4e7aadfe9a51bc14b54cc2331c48ebff6"),
)
METADATA = (
    ("added_tokens.json", 122, "33e0fb93dadacb864bd2f2e8441e147daa2baceb67f94d3ef5283b495572cea0"),
    ("chat_template.json", 826, "2466d1704df30f0067f28d8e30e0190a1bf74e5b430942697af974d162a056bd"),
    ("config.json", 2528, "69277c2c9ba8a4f61d2e72f79fbe02e043b8c3af670858a606818527f971a0c2"),
    ("generation_config.json", 126, "89dc53229f50b59570b6852056dafeac8116c458f1a748bff491b6d4d24d3b51"),
    ("merges.txt", 1671853, "8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5"),
    ("model.safetensors.index.json", 77952, "e389b969d3b9b7120f136fbf6592cbbc1c07326157cae6f863fb64559261dae9"),
    ("preprocessor_config.json", 1732, "3644c108b9f0fa53e62ff422a9be6639642f0e64dab4a71f961c7911d4386384"),
    ("processor_config.json", 178, "04e9899e93f2a412c94e153cab4081457c9a44defb3b2c0b9df673d42c42cdd0"),
    ("special_tokens_map.json", 367, "f4f79e08d97f4d1c87f8d89264f525c8789da3b73b3bb55d1e12f692f41a7b1b"),
    ("tokenizer.json", 7028579, "3c0ce3213b50ff38d8aa1e91136a2d2cb142a3f569246170872e439cb2a29d15"),
    ("tokenizer_config.json", 1801, "b3151cdc153a44fa4e28abc8fc83567fc663aef7cd9dc041f5d66e617fe10b8e"),
    ("video_preprocessor_config.json", 621, "0ea9b672282b78353960b6069a521ed496a9f07c033e1e3362bff669234caa8d"),
    ("video_processor/preprocessor_config.json", 428, "1e71b2e75b90ddf696692529485b4a75fd54ecd4bbc03e2cc7be4af032875765"),
    ("vocab.json", 2776833, "ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910"),
)


def file_bytes(path):
    return path.stat().st_size if path.is_file() else 0


def verified(path, size, expected):
    if file_bytes(path) != size:
        return False
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest() == expected


def range_matches(headers, offset, end, total):
    replies = re.split(r"(?m)(?=^HTTP/)", headers.strip())
    final = replies[-1]
    return (re.match(r"HTTP/\S+\s+206\b", final) is not None
            and re.search(rf"(?im)^content-range:\s*bytes {offset}-{end}/{total}\s*$", final) is not None)


def fetch_pinned(name, size, expected, destination):
    if verified(destination, size, expected):
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + ".incomplete")
    direct_env = {k: v for k, v in ENV.items() if k.lower() not in {
        "http_proxy", "https_proxy", "all_proxy", "no_proxy"}}
    endpoints = (
        (f"https://modelscope.cn/models/{MODEL_ID}/resolve/master/{name}", direct_env),
        (f"https://hf-mirror.com/{MODEL_ID}/resolve/{MODEL_SHA}/{name}", ENV),
        (f"https://huggingface.co/{MODEL_ID}/resolve/{MODEL_SHA}/{name}", ENV),
    )
    offset = file_bytes(partial)
    if offset > size:
        raise RuntimeError(f"Oversized partial file: {partial} ({offset} bytes, expected {size})")
    chunk = partial.with_suffix(".range")
    headers = partial.with_suffix(".headers")
    while offset < size:
        end = min(offset + 64 * 1024**2, size) - 1
        length = end - offset + 1
        for url, env in endpoints:
            host = url.split("/")[2]
            print(f"MODEL_RANGE {name} bytes={offset}-{end}/{size} source={host}", flush=True)
            try:
                logged(["curl", "--fail", "--location", "--range", f"{offset}-{end}", "--connect-timeout", "15",
                        "--speed-limit", "16384", "--speed-time", "90", "--max-time", "600",
                        "--max-filesize", str(length), "--dump-header", headers, "--output", chunk,
                        "--progress-bar", url], "model_download.log", env)
            except RuntimeError:
                print(f"MODEL_ENDPOINT_RETRY {name} source={host}", flush=True)
                continue
            response = headers.read_text(encoding="iso-8859-1")
            # Some mirrors answer a request for an entire small file with a plain HTTP 200.
            whole_file = offset == 0 and length == size and verified(chunk, size, expected)
            if file_bytes(chunk) != length or not (whole_file or range_matches(response, offset, end, size)):
                print(f"MODEL_RANGE_REJECTED {name} source={host}", flush=True)
                continue
            with partial.open("ab") as output, chunk.open("rb") as incoming:
                if output.tell() != offset:
                    raise RuntimeError(f"Another downloader changed {partial}. Stop concurrent downloads.")
                try:
                    shutil.copyfileobj(incoming, output, length=1024**2)
                except BaseException:
                    output.flush()
                    output.truncate(offset)
                    raise
            offset += length
            print(f"MODEL_PROGRESS {name} {offset}/{size} bytes", flush=True)
            break
        else:
            raise RuntimeError(f"All endpoints failed for {name} at byte {offset}. Rerun to resume {partial}.")
    chunk.unlink(missing_ok=True)
    headers.unlink(missing_ok=True)
    if not verified(partial, size, expected):
        raise RuntimeError(f"SHA-256 mismatch in {partial}. The file was kept for inspection.")
    partial.replace(destination)


if not VALIDATE_ONLY:
    assert MODEL_ID == "llava-hf/llava-onevision-qwen2-7b-ov-hf"
    assert MODEL_SHA == "0d50680527681998e456c7b78950205bedd8a068"
    repo = Path(ENV["HF_HOME"]) / "hub" / ("models--" + MODEL_ID.replace("/", "--"))
    blobs = repo / "blobs"
    MODEL_PATH = repo / "snapshots" / MODEL_SHA
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    missing = sum(max(0, size - file_bytes(blobs / (sha + ".incomplete")))
                  for name, size, sha in SHARDS
                  if file_bytes(MODEL_PATH / name) != size and file_bytes(blobs / sha) != size)
    free = shutil.disk_usage(MODEL_PATH).free
    assert free >= missing + 2 * 1024**3, f"Need {(missing + 2 * 1024**3 - free) / 1024**3:.1f} GiB more free"
    print(f"MODEL_CACHE remaining_weight_download={missing / 1024**3:.2f} GiB", flush=True)
    for name, size, sha in SHARDS:
        path = MODEL_PATH / name
        if not verified(path, size, sha):
            blob = blobs / sha
            fetch_pinned(name, size, sha, blob)
            pending_link = path.with_name(path.name + ".verified-link")
            if pending_link.is_symlink():
                pending_link.unlink()
            pending_link.symlink_to(Path("../../blobs") / sha)
            pending_link.replace(path)
        print("MODEL_SHARD_VERIFIED", name, flush=True)
    for name, size, sha in METADATA:
        fetch_pinned(name, size, sha, MODEL_PATH / name)
    index = json.loads((MODEL_PATH / "model.safetensors.index.json").read_bytes())
    assert set(index["weight_map"].values()) == {name for name, _, _ in SHARDS}
    (WORK / "model_path.json").write_text(json.dumps(str(MODEL_PATH)), encoding="utf-8")
    print("MODEL_REVISION_PASS", MODEL_SHA, flush=True)


## 2. Compare the frozen boundary policy with matched baselines


In [ ]:
ARTIFACT = 'HERMES_twinpast_boundary_evidence.zip'

if not VALIDATE_ONLY:
    (WORK/'launch.json').write_text(json.dumps(dict(started_utc=datetime.now(timezone.utc).isoformat(),
        source_revision=BASE_REVISION, model_revision=MODEL_SHA, transformers_revision=TF_SHA,
        embedded_sha256=EXPECTED, gpu=smi.stdout.strip()), indent=2), encoding='utf-8')
    env = {**ENV, 'PYTHONDONTWRITEBYTECODE': '1', 'HF_HUB_OFFLINE': '1'}
    logged([PYTHON, '-u', RUNNER, '--work', WORK, '--model', MODEL_PATH, '--previous', PREVIOUS],
           'twinpast_boundary.log', env, REPO)
else:
    print('VALIDATION_PASS: setup and embedded sources checked without GPU, network or file writes')


## 3. Download the evidence ZIP

After a runner error, run this cell manually to retrieve failure evidence.


In [ ]:

if not VALIDATE_ONLY:
    from IPython.display import FileLink, display
    artifact = WORK/ARTIFACT
    if artifact.is_file():
        link = Path.cwd()/ARTIFACT
        if artifact.resolve() != link.resolve():
            shutil.copyfile(artifact, link)
        print((WORK/'completion.json').read_text(encoding='utf-8'))
        display(FileLink(ARTIFACT))
    else:
        print('No runner ZIP yet. Inspect setup logs under', WORK/'logs')
